# Make a Video

Run the steps top to bottom. Each one prints what to do next.

**First time:** Runtime &rarr; Change runtime type &rarr; **T4 GPU**, then start at Step 1.

**Coming back to a half-finished video:** run Steps 1&ndash;4, look at the checklist in the
Control Panel, and jump to the step it points at.

| | |
|---|---|
| Steps 1&ndash;4 | Setup |
| Steps 5&ndash;15 | Claude writes and checks the script (~15 min, no GPU needed) |
| Steps 16&ndash;21 | The voiceover |
| Steps 22&ndash;26 | The pictures (GPU, slow) |
| Steps 27&ndash;29 | Build the video |
| Steps 30&ndash;31 | Upload and learn (optional) |


## Step 1 — Install

In [11]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [6]:
# =============================================================================
#  SHARED FLUX LOADER  +  GPU MEMORY HELPERS
# =============================================================================
#  WHY THIS CELL EXISTS (read once, saves you hours):
#
#  FLUX.1-schnell is a 12B-parameter transformer. In fp16/bf16 that is ~24 GB of
#  weights for the transformer ALONE, plus ~9 GB for the T5 text encoder. A free
#  Colab T4 has 15 GB of VRAM and ~12 GB of system RAM. So:
#
#    - device_map="auto"            -> pins weights onto the GPU permanently and
#                                      fills all 15 GB before you generate anything.
#                                      This is what caused your OOM at 14.49 GB used
#                                      while nvidia-smi showed 0% utilisation.
#    - enable_model_cpu_offload()   -> moves whole components at a time, so the
#                                      24 GB transformer still has to fit on the card
#                                      in one piece. It cannot. Guaranteed OOM.
#    - enable_sequential_cpu_offload -> keeps weights in system RAM instead, and
#                                      12 GB of RAM cannot hold 33 GB of weights.
#
#  The only thing that actually fits on free Colab is 4-BIT QUANTISATION. It cuts
#  the transformer to ~6 GB and T5 to ~2.5 GB, which leaves real headroom on a T4.
#  Quality loss on schnell at 4 steps is not visible for flat vector-style frames.
# =============================================================================

import gc

FLUX_REPO = "black-forest-labs/FLUX.1-schnell"


def free_gpu(*objs):
    """Drop model objects and actually give the VRAM back. Call between stages."""
    import torch
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def gpu_report(label=""):
    import torch
    if not torch.cuda.is_available():
        print("no GPU")
        return
    free_b, total_b = torch.cuda.mem_get_info()
    used = (total_b - free_b) / 1024**3
    print(f"  GPU {label}: {used:.2f} GB used / {total_b/1024**3:.2f} GB total")


def load_flux(quantize=True):
    """The ONE place FLUX gets loaded. Every FLUX cell calls this.

    quantize=True  -> 4-bit NF4. Required on a free T4.
    quantize=False -> fp16 + sequential offload. Only for a 24GB+ GPU (L4/A100).
    """
    import torch
    from diffusers import FluxPipeline

    dtype = torch.float16   # T4 (Turing/sm75) has no native bfloat16 — fp16 is correct here

    if quantize:
        from diffusers import FluxTransformer2DModel
        from diffusers import BitsAndBytesConfig as DiffBnB
        from transformers import T5EncoderModel
        from transformers import BitsAndBytesConfig as TfBnB

        print("Loading FLUX [schnell] in 4-bit (fits a free T4)...")

        t5 = T5EncoderModel.from_pretrained(
            FLUX_REPO, subfolder="text_encoder_2", torch_dtype=dtype,
            quantization_config=TfBnB(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=dtype,
            ),
        )
        transformer = FluxTransformer2DModel.from_pretrained(
            FLUX_REPO, subfolder="transformer", torch_dtype=dtype,
            quantization_config=DiffBnB(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=dtype,
            ),
        )
        pipe = FluxPipeline.from_pretrained(
            FLUX_REPO, text_encoder_2=t5, transformer=transformer, torch_dtype=dtype,
        )
        pipe.enable_model_cpu_offload()      # safe now: quantised parts are small
    else:
        print("Loading FLUX [schnell] in fp16 with sequential offload (needs 24GB+ GPU)...")
        pipe = FluxPipeline.from_pretrained(FLUX_REPO, torch_dtype=dtype)
        pipe.enable_sequential_cpu_offload()

    pipe.vae.enable_slicing()   # the non-deprecated form of enable_vae_slicing()
    pipe.vae.enable_tiling()
    gpu_report("after load")
    return pipe


print("FLUX loader ready. Call load_flux() — do not call FluxPipeline.from_pretrained directly.")


FLUX loader ready. Call load_flux() — do not call FluxPipeline.from_pretrained directly.


In [8]:
# ── Installs everything the notebook needs. Run once per Colab session. ──
# Takes 2-4 minutes. You'll see a lot of scrolling text — that's normal.

%pip install -q anthropic diffusers torch transformers accelerate sentencepiece bitsandbytes \
               ffmpeg-python soundfile faster-whisper torchaudio \
               chatterbox-tts trafilatura requests ipywidgets protobuf \
               librosa noisereduce pyloudnorm scipy numpy

!which ffmpeg ffprobe > /dev/null || apt-get -qq install -y ffmpeg

import torch
print("\n" + "=" * 60)
print("READY")
print("=" * 60)
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — Runtime > Change runtime type > T4 GPU'}")
print("  Now run Step 2.")


ERROR: Exception:
Traceback (most recent call last):
  File "/home/ubuntu/new_yt_automation/.venv/lib/python3.14/site-packages/pip/_internal/cli/base_command.py", line 105, in _run_wrapper
    status = _inner_run()
  File "/home/ubuntu/new_yt_automation/.venv/lib/python3.14/site-packages/pip/_internal/cli/base_command.py", line 96, in _inner_run
    return self.run(options, args)
           ~~~~~~~~^^^^^^^^^^^^^^^
  File "/home/ubuntu/new_yt_automation/.venv/lib/python3.14/site-packages/pip/_internal/cli/req_command.py", line 68, in wrapper
    return func(self, options, args)
  File "/home/ubuntu/new_yt_automation/.venv/lib/python3.14/site-packages/pip/_internal/commands/install.py", line 387, in run
    requirement_set = resolver.resolve(
        reqs, check_supported_wheels=not options.target_dir
    )
  File "/home/ubuntu/new_yt_automation/.venv/lib/python3.14/site-packages/pip/_internal/resolution/resolvelib/resolver.py", line 96, in resolve
    result = self._result = resolver.re

ModuleNotFoundError: No module named 'torch'

## Step 2 — Key and folder

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

In [ ]:
import os

# =============================================================================
#  SAVE-TO-DRIVE  (strongly recommended)
# =============================================================================
#  Colab throws your files away when the session disconnects. With this ON, every
#  file the pipeline makes lands in your Google Drive instead, so a disconnect
#  costs you the current cell and nothing else.
# =============================================================================
SAVE_TO_DRIVE = True
PROJECT_NAME  = "video_pipeline"

if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORKDIR = f"/content/drive/MyDrive/videos/{PROJECT_NAME}"
    except Exception as e:
        print(f"Drive mount unavailable ({e}) — using local session storage instead.")
        WORKDIR = "/content/" + PROJECT_NAME
else:
    WORKDIR = "/content/" + PROJECT_NAME

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print(f"Working folder: {WORKDIR}")

# =============================================================================
#  ANTHROPIC API KEY
# =============================================================================
#  Best way: click the KEY icon in the left sidebar, add a secret named
#  ANTHROPIC_API_KEY, and switch "Notebook access" ON. Then you never paste it.
# =============================================================================
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = "sk-ant-your-key-here"  # <-- or paste here

_key = os.environ.get("ANTHROPIC_API_KEY", "")
if not _key or _key == "sk-ant-your-key-here":
    raise RuntimeError(
        "No Anthropic API key.\n"
        "  Add ANTHROPIC_API_KEY in Colab Secrets (key icon, left sidebar,\n"
        "  'Notebook access' ON) — or paste it into the line above."
    )
print(f"API key present ({_key[:7]}...{_key[-4:]})")
print("\nNow run Step 3.")


## Step 3 — Settings

The beat sheet, the banned words, the topic list, and the look of the drawings all live here.

In [ ]:
import os
import json
import re

# --- Which Claude model does which job ---
MODEL_WRITER    = "claude-sonnet-5"              # script generation + fixes
MODEL_JUDGE     = "claude-sonnet-5"              # retention judging (needs real judgment)
MODEL_CHECKER   = "claude-haiku-4-5-20251001"    # fact-matching + visual prompts (cheap, high volume)

# --- Where the narration comes from. Change this in the Control Panel. ---
#   "tts"   : the computer reads it (fast, nothing to record)
#   "human" : you read it yourself, one file per section
# Both end up as vo_raw.wav + scenes_timeline.json, so everything after is identical.
NARRATION = "tts"

# =============================================================================
#  THE LOOK  —  read this before you change anything below
# =============================================================================
#  The reference style is: one black-outlined stick figure, flat solid colour props,
#  a pure white wall, a single tan floor band, and the script line hand-lettered in
#  the empty space. Four things make it read as ONE artist across 150 frames:
#
#    1. the character never changes            -> CHARACTER_BIBLE, repeated verbatim
#    2. the drawing style never changes        -> STYLE_PREFIX, repeated verbatim
#    3. the colours never change               -> PALETTE, enforced in code after
#                                                 generation (see Step 21)
#    4. the on-screen text is always the same
#       font in the same place                 -> drawn by code, never by FLUX
#
#  Only 1 and 2 are FLUX's job, and FLUX is unreliable at them. 3 and 4 are done
#  deterministically in Python, which is why they are the parts you can trust.
# =============================================================================

CHARACTER_BIBLE = (
    "one simple stick-figure person: a large plain circle head filled pure white with a "
    "thick black outline, two short curved black lines for eyes, one small curved line for "
    "the mouth, no nose, no ears, no hair, a single thin black line for the torso and one "
    "thin black line per arm and per leg, no hands or feet drawn in detail, no clothes, "
    "no colour anywhere on the character"
)

STYLE_PREFIX = (
    "hand-drawn digital doodle cartoon, thick uniform black marker outline, slightly wobbly "
    "hand-drawn lines, flat solid colour fills with no gradients and no shading, pure white "
    "background wall, one horizontal floor line with a flat tan floor band along the bottom, "
    "very few objects, large areas of empty white space, simple childlike drawing, "
    "no text, no letters, no words, no numbers, no watermark, no signature, "
)

# --- The locked colour palette. Every finished frame is snapped to exactly these. ---
#     This is what guarantees frame 3 and frame 140 look like the same channel.
#     Add a colour only if you really need it — a small palette is the whole trick.
PALETTE = [
    (255, 255, 255),   # white — background wall
    (26,  26,  26),    # near-black — every outline
    (216, 164, 86),    # tan — the floor band
    (182, 70,  45),    # rust red — the "one bold prop" colour
    (196, 140, 80),    # wood brown — furniture
    (120, 72,  40),    # dark brown — shadows / depth on wood
]
PALETTE_LOCK = True     # snap every frame to PALETTE. Turn off to see raw FLUX output.
WHITE_SNAP   = 236      # anything brighter than this becomes pure white

# --- On-screen text (the "CLOSE YOUR EYES" lettering) ---
CAPTIONS_ON     = True          # draw the script line onto frames that call for it
CAPTION_FONT    = "PermanentMarker-Regular.ttf"
CAPTION_COLOR   = (26, 26, 26)
CAPTION_MAX_W   = 0.42          # caption box: 42% of frame width, top-left
CAPTION_MAX_H   = 0.26
CAPTION_X       = 0.06
CAPTION_Y       = 0.07

# --- Video geometry (16:9, YouTube) ---
FRAME_WIDTH  = 1280
FRAME_HEIGHT = 720
OUTPUT_FPS   = 30
FLUX_SEED    = 42     # base seed. Honest note: this does NOT give you character
                      # consistency on its own — see the note in Step 21.
FLUX_STEPS   = 4      # schnell is distilled for 4 steps; more does not help


SPOKEN_WPM = 150  # Chatterbox lands around here; used only for the length estimate printout


# =============================================================================
#  THE BEAT SHEET — single source of truth
# =============================================================================
# This dict is rendered into BOTH the writer's prompt (Cell 4) and the retention
# judge's prompt (Cell 8). That is the whole point: the judge holds the writer to
# the exact same spec, rather than to a second copy that can drift out of sync.
#
# Section 3 is deliberately NOT defined here — it is chosen at runtime in Cell 3.5
# based on what kind of evidence the fetched sources actually contain. The old
# version of this notebook hardcoded a "volunteers performed a slippery task,
# here is the percentage difference between groups" shape, which forced the model
# to manufacture an experiment for topics that never had one.
# =============================================================================

BEAT_SHEET = {
    1: {
        "name": "The Cold Hook",
        "words": (90, 110),
        "rhythm": "Rapid, staccato, somatic. Short sharp lines. No sentence over 14 words.",
        "beats": [
            "Direct Somatic Directive - one short sentence (3-6 words) telling the viewer to look at, touch, or notice a specific part of their own body right now.",
            "Rhetorical Diagnostic Question - one sentence asking why this universal, mundane, slightly odd thing happens there.",
            "The Common-Sense Explanation - one sentence giving the widely-accepted schoolbook answer, stated plainly and confidently, as if it settles the matter.",
            "The Absolute Negation ('The Snap') - one blunt sentence of 2-4 words stating that answer is wrong. It must be 2-4 words. Not five. Not a clause with a comma.",
            "The Escalating Mystery - one or two sentences revealing the specific, concrete counter-evidence that breaks the schoolbook answer. Must be drawn from the sources, not asserted.",
        ],
        "hard_rules": [
            "Beat 4 must be 2-4 words total.",
            "No sentence in this section may exceed 14 words.",
            "Do not name the topic in a textbook way ('Today we are looking at the human chin'). The viewer should not know they are watching an explainer yet.",
        ],
    },
    2: {
        "name": "Context & Setup",
        "words": (170, 210),
        "rhythm": "Moderate. Explanatory swells of 10-16 words alternating with short 5-8 word landings.",
        "beats": [
            "The Historical Anchor - one sentence placing the question in time: who first asked it, when it was first noticed, or how long it has gone unanswered. Use a named person or year ONLY if the sources give you one; otherwise anchor it in whatever framing the sources do support.",
            "The Status Quo - one or two sentences on how science classified this: useless, vestigial, accidental, passive, a side effect.",
            "The Crack - one or two sentences on the observation or anomaly that made the status quo untenable.",
            "The Stakes Question - one open sentence that names what is actually at stake in the answer, and hands off into the evidence section.",
        ],
        "hard_rules": [
            "Do not invent a scientist's name, an institution, or a year. If the sources do not contain one, write the beat without one.",
        ],
    },
    4: {
        "name": "The Re-Hook & Mechanism",
        "words": (200, 250),
        "rhythm": "Opens with a hard pivot, then builds. Mechanism explained in physical, tangible terms.",
        "beats": [
            "The Re-Hook - one short sentence (4-8 words) that reopens the mystery just as the viewer thinks it is solved. This is the single most important retention beat in the script; it sits at roughly the 4-minute mark where viewers leave.",
            "The Mechanism Breakdown - two to four sentences on how this physically works at a smaller scale: what tissue, what signal, what process, what timing. Concrete nouns and verbs only.",
            "The Core Metaphor - one or two sentences translating the mechanism into a familiar non-biological object or system. It must be an object the viewer has physically handled.",
            "The Micro-Impact - one short sentence on what this means for the viewer's own body, today.",
        ],
        "hard_rules": [
            "The metaphor must be concrete and handleable (a zipper, a wet sponge, a door wedge). Not abstract (a symphony, a dance, a conversation).",
            "Do not restate a statistic already given in Section 3.",
        ],
    },
    5: {
        "name": "The Dilemma",
        "words": (280, 330),
        "rhythm": "Slower, heavier, sweeping. Longer sentences with room to breathe. This is the only section allowed to be reflective.",
        "beats": [
            "The Trap Closing - one reflective sentence turning the finding on the viewer's own comfortable modern life.",
            "The Evolutionary Flashback - two to four vivid sentences putting the viewer in a specific ancestral scene where this mattered. Show one concrete situation, not a montage of humanity.",
            "The Trade-off - one or two sentences on the physiological cost: why the body does not simply keep this switched on, or what it gave up to have this.",
            "The Uncomfortable Truth - one or two sentences on the thing about this that does not resolve cleanly.",
        ],
        "hard_rules": [
            "This is the section most prone to over-writing. Show, do not label. You may not tell the viewer how to feel about a fact - no 'unsettling', 'haunting', 'strangely beautiful', 'profound', 'remarkable'. Describe the thing precisely enough that the feeling arrives on its own.",
            "Maximum ONE adjective per noun. No stacked adjectives ('a quiet, ancient, forgotten reflex').",
            "The ancestral flashback must be one specific scene with one specific body doing one specific thing - not 'our ancestors' as a collective.",
        ],
    },
    6: {
        "name": "Impactful Outro",
        "words": (150, 185),
        "rhythm": "Meditative, slow, intimate. Then one hard stop.",
        "beats": [
            "The Somatic Return - one sentence looping back to the exact body part and action from the Cold Hook, using some of the same words so the loop is audible.",
            "The Re-framing - one or two sentences contrasting the sterile modern setting the viewer is sitting in with the machinery still running inside them.",
            "The Realization - one or two slow sentences on what this quirk actually is, stated plainly.",
            "The Closing Question ('The Snap') - one final open-ended question of 3-8 words. It must be genuinely unanswerable and must invite the viewer to answer in the comments implicitly, without ever saying 'comment below'.",
        ],
        "hard_rules": [
            "No new factual claims in this section at all. Every sentence must be tagged [0].",
            "The final line must be a real question the viewer could argue about - not a statement wearing a question mark ('Isn't the body amazing?').",
            "No call to action, no 'subscribe', no 'thanks for watching'.",
        ],
    },
}


# =============================================================================
#  BANNED LEXICON — deterministic linter input (Cell 5), no API call needed
# =============================================================================
BANNED_PHRASES = [
    # AI-essay tells
    "delve", "tapestry", "testament to", "in a world where", "little did",
    "unlock the secret", "the truth is stranger", "buckle up", "let that sink in",
    "it turns out that", "here's the kicker", "plot twist", "mind-blowing",
    "nothing short of", "a fascinating glimpse", "sheds light on", "paves the way",
    "at the end of the day", "when it comes to", "needless to say",
    # emotion-labelling (the Section 5 failure mode)
    "unsettling", "strangely beautiful", "hauntingly", "profoundly",
    "remarkably", "astonishingly", "incredibly", "utterly", "truly amazing",
    "beautifully", "eerily", "oddly beautiful", "deeply moving",
    # filler
    "basically", "essentially", "literally", "actually just", "simply put",
    "in other words", "to put it simply", "believe it or not",
    # YouTube-slop CTA
    "subscribe", "smash that", "comment below", "let me know in the comments",
    "thanks for watching", "don't forget to",
]

HEDGE_WORDS = ["perhaps", "maybe", "possibly", "arguably", "somewhat", "rather", "quite", "fairly"]


# =============================================================================
#  TOPIC BANK
# =============================================================================
TOPICS = {
    "chin": {
        "title": "Why Do We Have a Chin?",
        "sources": [
            {"title": "Why Do Humans Have Chins? (evolutionary spandrel research)",
             "url": "https://www.smithsonianmag.com/smart-news/why-do-humans-have-chins-they-might-be-an-evolutionary-accident-new-research-suggests-180988205/"},
            {"title": "Why Do Humans Have Chins? (Smithsonian, mechanical-stress hypothesis)",
             "url": "https://www.smithsonianmag.com/science-nature/why-do-humans-have-chins-15140492/"},
        ],
    },
    "photic_sneeze": {
        "title": "The Photic Sneeze Reflex",
        "sources": [
            {"title": "When the Sun Prickles Your Nose: EEG study of photic sneezing (PLOS ONE, via PMC)",
             "url": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2821404/"},
            {"title": "Photic sneeze reflex: definition, cause, symptoms",
             "url": "https://www.medicalnewstoday.com/articles/photic-sneeze-reflex"},
        ],
    },
    "finger_pruning": {
        "title": "Why Fingers Prune in Water",
        "sources": [
            {"title": "Why Do Our Fingers and Toes Wrinkle During a Bath? (Scientific American)",
             "url": "https://www.scientificamerican.com/article/why-do-our-fingers-and-toes-wrinkle-during-a-bath/"},
            {"title": "Debate: did wrinkled fingers evolve for better grip? (National Geographic)",
             "url": "https://www.nationalgeographic.com/science/article/debate-did-wrinkled-fingers-evolve-for-better-grip"},
        ],
    },
    "emotional_tears": {
        "title": "Why We Cry Emotional Tears",
        "sources": [
            {"title": "All About Emotional Tears (American Academy of Ophthalmology)",
             "url": "https://www.aao.org/eye-health/tips-prevention/all-about-emotional-tears"},
            {"title": "8 benefits of crying: why do we cry, and when to seek support",
             "url": "https://www.medicalnewstoday.com/articles/319631"},
        ],
    },
    "appendix": {
        "title": "The Mystery of the Appendix",
        "sources": [
            {"title": "Appendix Isn't Useless at All: It's a Safe House for Bacteria (Duke Health)",
             "url": "https://corporate.dukehealth.org/news/appendix-isnt-useless-all-its-safe-house-bacteria"},
            {"title": "The appendix plays a hidden role in gut health (NPR)",
             "url": "https://www.npr.org/sections/health-shots/2024/02/02/1228474984/appendix-function-appendicitis-gut-health"},
        ],
    },
    "crowded_teeth": {
        "title": "Why Our Teeth Are So Crowded",
        "sources": [
            {"title": "Humans used to have straighter teeth - what changed? (National Geographic)",
             "url": "https://www.nationalgeographic.com/health/article/crooked-teeth-human-evolution-jaw-size"},
            {"title": "How Come Ancient Skulls Often Have Straight Teeth? (IFLScience)",
             "url": "https://www.iflscience.com/how-come-ancient-skulls-often-have-straight-teeth-70078"},
        ],
    },
    "contagious_yawn": {
        "title": "The Contagious Yawn",
        "sources": [
            {"title": "Brain size and neuron numbers drive differences in yawn duration (Nature, Communications Biology)",
             "url": "https://www.nature.com/articles/s42003-021-02019-y"},
            {"title": "The science of yawning: physiology, evolutionary role, behavioral impact (PMC)",
             "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC12488162/"},
        ],
    },
    "motion_sickness": {
        "title": "Why We Get Motion Sickness",
        "sources": [
            {"title": "Motion sickness: it all started 550 million years ago (The Conversation)",
             "url": "https://theconversation.com/motion-sickness-it-all-started-550-million-years-ago-121789"},
            {"title": "Why Do Humans Get Dizzy? An Evolutionary Biologist Explains (Forbes)",
             "url": "https://www.forbes.com/sites/scotttravers/2026/06/07/why-do-humans-get-dizzy-an-evolutionary-biologist-explains/"},
        ],
    },
    "goosebumps": {
        "title": "Why We Get Goosebumps",
        "sources": [
            {"title": "The real reason behind goosebumps (Harvard Stem Cell Institute)",
             "url": "https://www.hsci.harvard.edu/goosebumps-hair-follicle-stem-cells"},
            {"title": "Why Do We Get Goosebumps? An Evolutionary Biologist Explains (Forbes)",
             "url": "https://www.forbes.com/sites/scotttravers/2026/04/07/why-do-we-get-goosebumps-an-evolutionary-biologist-explains/"},
            {"title": "Goosebumps: When the body remembers (Hektoen International, medical humanities journal)",
             "url": "https://hekint.org/2026/07/23/goosebumps-when-the-body-remembers/"},
        ],
    },
    "hypnic_jerk": {
        "title": "The Hypnic Jerk",
        "sources": [
            {"title": "Hypnic Jerks (Sleep Foundation)",
             "url": "https://www.sleepfoundation.org/parasomnias/hypnic-jerks"},
            {"title": "Hypnic jerk: why you twitch before falling asleep",
             "url": "https://www.medicalnewstoday.com/articles/324666"},
        ],
    },
    "brain_freeze": {
        "title": "Brain Freeze Explained",
        "sources": [
            {"title": "What causes brain freeze? (Harvard Health)",
             "url": "https://www.health.harvard.edu/diseases-and-conditions/what-causes-brain-freeze"},
            {"title": "How to Ease Brain Freeze (Johns Hopkins Medicine)",
             "url": "https://www.hopkinsmedicine.org/health/conditions-and-diseases/how-to-ease-brain-freeze"},
        ],
    },
    "smell_of_rain": {
        "title": "Why We Can Smell Rain Better Than Sharks Can Smell Blood",
        "sources": [
            {"title": "Petrichor and Geosmin - The Smell of Rain",
             "url": "https://sciencenotes.org/petrichor-and-geosmin-the-smell-of-rain/"},
            {"title": "Why Does Geosmin Smell? (The Scientist - Stensmyr, Max Planck / Lund)",
             "url": "https://www.the-scientist.com/why-does-geosmin-smell-70231"},
            {"title": "Geosmin: Why We Like The Smell Of Air After A Storm (ACSH)",
             "url": "https://www.acsh.org/news/2018/07/28/geosmin-why-we-smell-air-after-storm-13240"},
        ],
    },
    "self_tickle": {
        "title": "Why Can't You Tickle Yourself?",
        "sources": [
            {"title": "Why can't a person tickle himself? (Scientific American)",
             "url": "https://www.scientificamerican.com/article/why-cant-a-person-tickle/"},
            {"title": "Why Can't You Tickle Yourself? (Yale Scientific Magazine)",
             "url": "https://www.yalescientific.org/2015/05/why-cant-you-tickle-yourself/"},
        ],
    },
    "fingerprints": {
        "title": "Why Do We Have Fingerprints?",
        "sources": [
            {"title": "Fingerprints and Friction (Smithsonian Magazine)",
             "url": "https://www.smithsonianmag.com/science-nature/fingerprints-and-friction-12860851/"},
            {"title": "Get a grip! Blistering new evidence on why we have fingerprints (University of Manchester)",
             "url": "https://www.manchester.ac.uk/about/news/get-a-grip-blistering-new-evidence-on-why-we-have-fingerprints/"},
        ],
    },
    "naked_ape": {
        "title": "Why Are We the Only \"Naked Ape\"?",
        "sources": [
            {"title": "Why Humans Lost Their Hair and Became Naked and Sweaty (Discover Magazine)",
             "url": "https://www.discovermagazine.com/why-humans-lost-their-hair-and-became-naked-and-sweaty-307"},
            {"title": "Avoidance of overheating and selection for hair loss and bipedality in hominins (PNAS)",
             "url": "https://www.pnas.org/doi/10.1073/pnas.1113915108"},
        ],
    },
}

TOPIC_KEY = "goosebumps"   # <-- change this to produce a different one of the 15
TOPIC = TOPICS[TOPIC_KEY]["title"]

_lo = sum(v["words"][0] for v in BEAT_SHEET.values())
_hi = sum(v["words"][1] for v in BEAT_SHEET.values())
print(f"\nProducing: {TOPIC}")
print(f"Beat sheet (5 fixed sections + 1 adaptive evidence section) targets {_lo}-{_hi} words")
print(f"  + Section 3 (chosen in Cell 3.5) adds ~250-300  ->  ~{_lo+250}-{_hi+300} words")
print(f"  estimated runtime at {SPOKEN_WPM} wpm: {(_lo+250)/SPOKEN_WPM:.1f}-{(_hi+300)/SPOKEN_WPM:.1f} min")
print("Sources:")
for s in TOPICS[TOPIC_KEY]["sources"]:
    print(f"  - {s['title']}\n      {s['url']}")


# =============================================================================
#  RETENTION MEMORY — persisted learnings from real published videos
# =============================================================================
# This is the file the strategist (Cell 20) writes to and the retention judge
# (Cell 8) + packaging (Cell 10) read from. It starts empty. It only grows teeth
# once videos are live and analytics have been pulled — see Cell 18/19/20.
# Kept as a flat, bounded, human-readable list of *notes*, not weights or scores,
# because the thing feeding the judge's prompt should stay legible to a human
# reviewing it, not a black-box adjustment.
# =============================================================================
LEARNINGS_PATH = "learnings.json"


def load_learnings():
    if os.path.exists(LEARNINGS_PATH):
        with open(LEARNINGS_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "retention_notes": [],       # feeds Cell 8's judge
        "title_notes": [],           # feeds Cell 10's packaging prompt
        "thumbnail_notes": [],       # feeds Cell 10 + the thumbnail cell
        "banned_phrase_additions": [],
        "history": [],               # [{date, video_id, source: "analytics"|"competitor"}]
    }


LEARNINGS = load_learnings()


def learnings_block(keys, n=10):
    """Render the most recent n notes per category as a prompt-ready bullet list.
    Empty on the first videos on this channel — that's correct, not a bug."""
    lines = []
    for k in keys:
        lines.extend(f"- {note}" for note in LEARNINGS.get(k, [])[-n:])
    return "\n".join(lines) if lines else "(no data yet — first videos on this channel, ignore this section)"


# Deterministic linter also reads accumulated learnings, same reasoning as BANNED_PHRASES above
BANNED_PHRASES = BANNED_PHRASES + LEARNINGS.get("banned_phrase_additions", [])


# =============================================================================
#  TITLE FORMULA — the pattern to replicate (reference channel, not this one)
# =============================================================================
# Rayyan's brief: "What Did Ancient Humans Do at Night?" is the shape to imitate.
# It is a plain, answerable-sounding question about a NAMED, concrete subject doing
# a MUNDANE, everyday thing — the mundane framing is what hides the curiosity gap.
# It never states the twist. Contrast with a generic "Why Do We X?" — that's a
# question about a category. This formula is a question about a scene.
# =============================================================================
TITLE_FORMULA = {
    "pattern": (
        "[What/Why/How] + [did/do] + [a named, concrete subject — Ancient Humans, "
        "Early Humans, Your Body, a specific body part] + [a short, mundane, everyday "
        "verb phrase — at night, before mirrors, in the cold, every morning]?"
    ),
    "reference_examples": [
        "What Did Ancient Humans Do at Night?",
    ],
    "rules": [
        "The subject must be named and concrete — never an abstract collective noun on its own.",
        "The verb phrase must describe an ordinary, everyday situation — the mundane framing IS the "
        "curiosity gap. It is never the twist itself and never hints at the answer.",
        "Title Case, ends in a question mark, under 60 characters.",
        "Never answer the question in the title.",
    ],
}



Producing: Why We Get Goosebumps
Beat sheet (5 fixed sections + 1 adaptive evidence section) targets 890-1085 words
  + Section 3 (chosen in Cell 3.5) adds ~250-300  ->  ~1140-1385 words
  estimated runtime at 150 wpm: 7.6-9.2 min
Sources:
  - The real reason behind goosebumps (Harvard Stem Cell Institute)
      https://www.hsci.harvard.edu/goosebumps-hair-follicle-stem-cells
  - Why Do We Get Goosebumps? An Evolutionary Biologist Explains (Forbes)
      https://www.forbes.com/sites/scotttravers/2026/04/07/why-do-we-get-goosebumps-an-evolutionary-biologist-explains/
  - Goosebumps: When the body remembers (Hektoen International, medical humanities journal)
      https://hekint.org/2026/07/23/goosebumps-when-the-body-remembers/


## Step 4 — Control Panel

Pick your topic and your voice. The checklist tells you where you are.

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

# =============================================================================
#  CONTROL PANEL
# =============================================================================
#  Pick your topic and your voice here, then read the checklist. The checklist
#  looks at what is actually on disk and tells you which step to run next, so you
#  can close the tab, come back tomorrow, and still know where you were.
# =============================================================================

STAGES = [
    ("Step 6   Sources fetched",   ["source_pack.json"]),
    ("Step 7   Evidence profiled", ["evidence_profile.json"]),
    ("Step 8   Script drafted",    ["script_tagged.json"]),
    ("Step 14  Script finished",   ["vo_raw.txt"]),
    ("Step 15  Title & tags",      ["packaging.json"]),
    ("Step 17/18  Voice recorded", ["vo_raw.wav", "scenes_timeline.json"]),
    ("Step 20  Voice polished",    ["full_vo.wav"]),
    ("Step 23  Character picked",  ["character_ref.png"]),
    ("Step 24  Pictures planned",  ["scenes_planned.json"]),
    ("Step 25  Pictures drawn",    ["frames"]),
    ("Step 27  Video built",       ["final_video.mp4"]),
    ("Step 28  Thumbnail made",    ["thumbnails"]),
]

topic_dd = widgets.Dropdown(
    options=[(v["title"], k) for k, v in TOPICS.items()],
    value=TOPIC_KEY, description="Topic:",
    layout=widgets.Layout(width="620px"), style={"description_width": "90px"},
)

voice_dd = widgets.Dropdown(
    options=[("Computer voice (fast, no recording)", "tts"),
             ("My own recorded voice (upload files)", "human")],
    value=NARRATION, description="Voice:",
    layout=widgets.Layout(width="620px"), style={"description_width": "90px"},
)

status_out = widgets.Output()
refresh_btn = widgets.Button(description="Refresh checklist", icon="refresh")


def _exists(name):
    if not os.path.exists(name):
        return False
    if os.path.isdir(name):
        return len(os.listdir(name)) > 0
    return os.path.getsize(name) > 0


def render_status(_=None):
    global TOPIC_KEY, TOPIC, NARRATION
    TOPIC_KEY = topic_dd.value
    TOPIC = TOPICS[TOPIC_KEY]["title"]
    NARRATION = voice_dd.value

    rows, next_step = [], None
    for label, files in STAGES:
        done = all(_exists(f) for f in files)
        if done:
            rows.append(f'<tr><td style="padding:3px 14px">&#9989;</td>'
                        f'<td style="padding:3px 14px;color:#666">{label}</td></tr>')
        else:
            mark = "&#128073;" if next_step is None else "&#11036;"
            colour = "#000;font-weight:600" if next_step is None else "#aaa"
            rows.append(f'<tr><td style="padding:3px 14px">{mark}</td>'
                        f'<td style="padding:3px 14px;color:{colour}">{label}</td></tr>')
            if next_step is None:
                next_step = label

    banner = ("&#127881; Everything is done — your video is ready."
              if next_step is None else f"Next up: <b>{next_step}</b>")

    with status_out:
        clear_output(wait=True)
        display(widgets.HTML(
            f'<div style="font-family:system-ui;font-size:14px">'
            f'<div style="padding:10px 14px;background:#f4f4f4;border-radius:8px;'
            f'margin-bottom:10px">Making: <b>{TOPIC}</b><br>{banner}</div>'
            f'<table style="border-collapse:collapse">{"".join(rows)}</table></div>'
        ))


refresh_btn.on_click(render_status)
topic_dd.observe(render_status, names="value")
voice_dd.observe(render_status, names="value")
render_status()

display(widgets.VBox([
    widgets.HTML('<h3 style="font-family:system-ui;margin:0 0 6px">Control Panel</h3>'),
    topic_dd, voice_dd,
    widgets.HTML('<div style="height:8px"></div>'),
    refresh_btn, status_out,
]))


## Step 5 — Claude engine

In [ ]:
import json
import re
import time
import anthropic

client = anthropic.Anthropic()

# --- Running API spend, so a run that quietly costs $12 in fix-loops is visible ---
USAGE = {}
PRICE_PER_MTOK = {   # USD per million tokens (input, output) - update if pricing changes
    "claude-sonnet-5": (3.0, 15.0),
    "claude-opus-5": (15.0, 75.0),
    "claude-haiku-4-5-20251001": (1.0, 5.0),
}


def print_usage():
    total = 0.0
    print("\n  API usage this run:")
    for model, u in USAGE.items():
        pin, pout = PRICE_PER_MTOK.get(model, (0.0, 0.0))
        cost = u["in"] / 1e6 * pin + u["out"] / 1e6 * pout
        total += cost
        print(f"    {model:<32} {u['calls']:>3} calls  "
              f"{u['in']:>7} in / {u['out']:>6} out  ~${cost:.3f}")
    print(f"    {'TOTAL':<32} {'':>3}        {'':>7}   {'':>6}      ~${total:.3f}")
    return total



def call_model(model, prompt, system=None, max_tokens=4000, temperature=0, retries=4):
    """One retry wrapper for every model call in this notebook. Overloads and
    transient 5xx are common enough on long runs that a bare call will eventually
    kill a pipeline 40 minutes in."""
    last = None
    for attempt in range(retries):
        try:
            kwargs = dict(model=model, max_tokens=max_tokens, temperature=temperature,
                          messages=[{"role": "user", "content": prompt}])
            if system:
                kwargs["system"] = system
            resp = client.messages.create(**kwargs)
            u = USAGE.setdefault(model, {"calls": 0, "in": 0, "out": 0})
            u["calls"] += 1
            u["in"] += getattr(resp.usage, "input_tokens", 0)
            u["out"] += getattr(resp.usage, "output_tokens", 0)
            return resp.content[0].text.strip()
        except Exception as e:
            last = e
            wait = 3 * (2 ** attempt)
            print(f"    [retry {attempt+1}/{retries}] {type(e).__name__}: {e} - waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Model call failed after {retries} attempts: {last}")


def parse_json_response(raw: str):
    """Models sometimes fence JSON or add a sentence before it. Strip both."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
    start = raw.find("{")
    end = raw.rfind("}")
    if start != -1 and end != -1:
        raw = raw[start:end + 1]
    return json.loads(raw)

print("Claude engine ready.")


## Step 6 — Get the sources

In [ ]:
import os
import hashlib
import json          # <-- BUG FIX: v4 called json.dump() here without importing json.
import time           #             Every run crashed with NameError right after fetching.
import requests
import trafilatura

MAX_CHARS_PER_SOURCE = 6000
MIN_USABLE_CHARS = 700     # below this it's almost always a cookie wall / paywall stub / nav junk
FETCH_RETRIES = 3

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/125.0 Safari/537.36")


def fetch_source_text(url: str, timeout: int = 20) -> str | None:
    """Fetch + extract clean article body. Retries with backoff; falls back to plain
    requests if trafilatura's fetcher gets blocked."""
    last_err = None
    for attempt in range(FETCH_RETRIES):
        try:
            downloaded = trafilatura.fetch_url(url)
            if not downloaded:
                resp = requests.get(url, timeout=timeout, headers={"User-Agent": UA})
                resp.raise_for_status()
                downloaded = resp.text
            text = trafilatura.extract(
                downloaded,
                include_comments=False,
                include_tables=True,
                favor_recall=True,
            )
            if text and text.strip():
                return text.strip()
            last_err = "extractor returned nothing"
        except Exception as exc:
            last_err = str(exc)
        if attempt < FETCH_RETRIES - 1:
            time.sleep(2 * (attempt + 1))
    print(f"    fetch failed after {FETCH_RETRIES} tries: {url} ({last_err})")
    return None


def looks_like_junk(text: str) -> str | None:
    """Cheap sanity checks on extracted text. Returns a reason string if it looks
    unusable, else None. Catches the case where the fetch 'succeeds' but you actually
    got a consent banner - which would silently starve the writer of real grounding."""
    if len(text) < MIN_USABLE_CHARS:
        return f"only {len(text)} chars extracted (min {MIN_USABLE_CHARS})"
    low = text.lower()
    junk_markers = [
        "enable javascript", "accept cookies", "cookie policy", "subscribe to continue",
        "you have reached your article limit", "please verify you are a human",
        "access denied", "403 forbidden", "captcha",
    ]
    hits = [m for m in junk_markers if m in low[:1500]]
    if hits and len(text) < 2000:
        return f"looks like a consent/paywall page (matched: {hits[0]})"
    return None


# --- Per-URL disk cache. Re-running a topic to iterate on the script no longer refetches,
#     which makes script iteration effectively free and avoids hammering publishers. ---
CACHE_DIR = "source_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
USE_CACHE = True


def cache_path(url: str) -> str:
    return os.path.join(CACHE_DIR, hashlib.sha1(url.encode()).hexdigest()[:16] + ".txt")


def fetch_cached(url: str) -> tuple[str | None, bool]:
    p = cache_path(url)
    if USE_CACHE and os.path.exists(p):
        with open(p, encoding="utf-8") as fh:
            return fh.read(), True
    text = fetch_source_text(url)
    if text:
        with open(p, "w", encoding="utf-8") as fh:
            fh.write(text)
    return text, False


source_pack = []
failures = []

srcs = TOPICS[TOPIC_KEY]["sources"]
print(f'Fetching {len(srcs)} source(s) for "{TOPIC}"...\n')

for s in srcs:
    print(f"  -> {s['title']}")
    text, from_cache = fetch_cached(s["url"])
    if from_cache:
        print("     (from cache)")
    if not text:
        failures.append((s, "no text extracted"))
        print("     FAILED\n")
        continue

    reason = looks_like_junk(text)
    if reason:
        failures.append((s, reason))
        print(f"     REJECTED - {reason}\n")
        continue

    excerpt = text[:MAX_CHARS_PER_SOURCE]
    source_pack.append({"title": s["title"], "url": s["url"], "excerpt": excerpt})
    print(f"     OK - {len(excerpt)} chars usable\n")

# --- Fail loudly rather than quietly writing a thin script off one weak source ---
if not source_pack:
    raise RuntimeError(
        "No sources fetched. Check network/URLs in TOPICS, then re-run this cell.\n"
        "Failures:\n" + "\n".join(f"  - {s['url']}: {r}" for s, r in failures)
    )

total_chars = sum(len(s["excerpt"]) for s in source_pack)
if total_chars < 3000:
    print("=" * 70)
    print(f"WARNING: only {total_chars} chars of grounding across {len(source_pack)} source(s).")
    print("A ~1,400-word script off this little source text is where the model starts")
    print("improvising. Strongly consider adding a third URL to this topic in Cell 2")
    print("and re-running, rather than pushing forward.")
    print("=" * 70)

with open("source_pack.json", "w", encoding="utf-8") as f:
    json.dump(source_pack, f, indent=2)

print(f"{len(source_pack)}/{len(srcs)} sources usable, {total_chars} chars total grounding.")
if failures:
    print(f"{len(failures)} source(s) unusable - the script will be written without them:")
    for s, r in failures:
        print(f"  - {s['title']}: {r}")


## Step 7 — Read the evidence

Works out what kind of evidence the sources actually contain, and picks the matching Section 3 shape. Also builds the *fact larder* — the only numbers and names the writer is allowed to use.

In [ ]:
import json

sources_block = "\n\n".join(
    f"[Source {i+1}: {s['title']}]\n{s['excerpt']}"
    for i, s in enumerate(source_pack)
)

PROFILER_PROMPT = f"""You are a research analyst preparing a brief for a science
scriptwriter. Read the source excerpts below about: {TOPIC}

Your job is to report ONLY what is actually in these sources. Do not add background
knowledge. If the sources do not contain something, say so - that is a useful finding,
not a failure.

Produce this exact JSON shape, no markdown fences, no commentary:

{{
  "evidence_type": one of
      "controlled_experiment"   (volunteers/subjects, an intervention, measured outcomes),
      "observational_study"     (measurement or survey, numbers, but no intervention),
      "comparative_anatomy"     (fossils, species or population comparison, morphology),
      "clinical_case"           (patients, disorders, lesions, medical observation),
      "competing_hypotheses"    (no decisive study; two or more rival explanations argued),
      "mechanism_only"          (the physiology is explained, but no headline study),
  "evidence_type_reason": "one sentence on why you chose that",
  "named_entities": [
     {{"kind": "researcher|institution|study|journal|year", "value": "...", "source": 1}}
  ],
  "hard_numbers": [
     {{"value": "the number exactly as written in the source", "means": "what it measures", "source": 1}}
  ],
  "key_findings": [
     {{"finding": "one sentence in plain language", "source": 1, "certainty": "established|debated|speculative"}}
  ],
  "competing_explanations": [
     {{"explanation": "...", "source": 1}}
  ],
  "mechanism": "two or three sentences on the physical mechanism as the sources describe it, or null",
  "the_common_myth": "the widely-believed schoolbook explanation these sources push back on, or null",
  "the_best_counter_evidence": "the single most concrete, surprising, specific fact in these sources - the thing that would make someone stop scrolling",
  "gaps": ["things a scriptwriter might reach for that these sources do NOT support"]
}}

Rules:
- "hard_numbers" must contain ONLY numbers that literally appear in the excerpts. This
  list becomes the writer's entire allowance of statistics. If a number is not here, the
  writer may not use it. Be thorough - miss one and the script loses a real fact.
- "certainty" matters. Mark a finding "debated" if the source hedges it, attributes it to
  one team, or presents a rival view. The script must reproduce that hedging.
- If there is no named researcher/year in the sources, return an empty "named_entities"
  list rather than guessing. The script is built to work without one.

SOURCE EXCERPTS:
{sources_block}"""

print("Profiling what evidence the sources actually contain...")
profile = parse_json_response(
    call_model(MODEL_CHECKER, PROFILER_PROMPT, max_tokens=4000, temperature=0)
)

with open("evidence_profile.json", "w", encoding="utf-8") as f:
    json.dump(profile, f, indent=2)


# =============================================================================
#  SECTION 3 TEMPLATES - one per evidence type
# =============================================================================
# This is the fix for the biggest structural flaw in v4. The old beat sheet demanded
# "the university, year and volunteer group", "the specific slippery task volunteers
# performed", and "the exact percentage difference between control and test groups"
# for EVERY topic. For a topic like "why do we have a chin" - which rests on skull
# morphometrics, not an experiment - that beat sheet leaves the model two options:
# fabricate an experiment, or dump raw methodology numbers to fill the beat. It picked
# the second, which is exactly the "532 skulls, 32 landmarks, 46 distances" pacing
# problem. The beat sheet caused it. Adding a retention judge on top would just fight
# the prompt forever. So: pick the section shape that matches the evidence.
# =============================================================================

SECTION_3_TEMPLATES = {
    "controlled_experiment": {
        "name": "The Experiment",
        "words": (250, 300),
        "rhythm": "Alternating: a short sharp finding, then a longer vivid image, then a short consequence. Never three data sentences in a row.",
        "beats": [
            "The Setup - one sentence naming who ran the study and what they wanted to find out. Include institution/year ONLY if it is in named_entities.",
            "The Task - one or two sentences on the specific physical thing subjects were asked to do. Make it visualisable.",
            "The Result - one sharp sentence with the single strongest number from the allowed list.",
            "The Translation - one or two sentences making that number mean something in a scale a person can feel.",
            "The Caveat - one sentence on the limit of the finding, if the sources hedge it.",
        ],
        "hard_rules": [
            "Maximum TWO sentences carrying numbers in this entire section.",
            "Never place two number-carrying sentences next to each other.",
        ],
    },
    "observational_study": {
        "name": "The Measurement",
        "words": (250, 300),
        "rhythm": "Alternating: one measured fact, then an image that makes it physical, then a consequence.",
        "beats": [
            "The Question - one sentence on what researchers set out to measure and why nobody had.",
            "The Method - one or two sentences on what they actually looked at, in concrete visual terms (what was on the table in front of them).",
            "The Pattern - one sharp sentence with the strongest single number or comparison from the allowed list.",
            "The Translation - one or two sentences turning that pattern into something the viewer can picture at human scale.",
            "The Limit - one sentence on what this kind of measurement can and cannot prove.",
        ],
        "hard_rules": [
            "Maximum TWO sentences carrying numbers in this entire section.",
            "Do NOT list methodology counts (sample sizes, number of measurements taken, number of landmarks). Those are how the work was done, not what was found. A viewer does not care that 46 distances were measured; they care what the measurement showed.",
        ],
    },
    "comparative_anatomy": {
        "name": "The Comparison",
        "words": (250, 300),
        "rhythm": "Two things held side by side. Short comparison, long image, short consequence.",
        "beats": [
            "The Two Things - one or two sentences setting up the comparison: us versus a relative, now versus then, this population versus that one.",
            "The Difference - one sharp sentence naming the single most striking difference, with a number only if one is in the allowed list.",
            "The Image - one or two sentences putting the viewer physically in front of the two specimens, seeing the difference.",
            "The Inference - one or two sentences on what that difference implies about how we got here.",
            "The Wrinkle - one sentence on the part of the comparison that does not fit the neat story.",
        ],
        "hard_rules": [
            "Maximum ONE sentence carrying numbers in this entire section.",
            "Do NOT recite methodology (how many specimens, how many measurements, what software). Describe what was seen, not what was counted.",
        ],
    },
    "clinical_case": {
        "name": "The Patient",
        "words": (250, 300),
        "rhythm": "Narrative. One person, one problem, one revealing detail.",
        "beats": [
            "The Case - one or two sentences on the specific patient or condition, in scene, not in summary.",
            "The Anomaly - one sharp sentence on the thing about them that should not have been possible.",
            "The Inference - one or two sentences on what their case proved about how this works in everyone.",
            "The Generalisation - one or two sentences moving from the one case out to the viewer's own body.",
            "The Limit - one sentence on what a single case can and cannot establish, if the sources hedge it.",
        ],
        "hard_rules": [
            "Maximum ONE sentence carrying numbers in this entire section.",
            "Do not sensationalise a patient. Describe clinically; the strangeness carries itself.",
        ],
    },
    "competing_hypotheses": {
        "name": "The Argument",
        "words": (250, 300),
        "rhythm": "Two positions, fairly stated, then the collision. Short claim, long support, short doubt.",
        "beats": [
            "Hypothesis One - one or two sentences stating the leading explanation and its single best piece of support.",
            "The Problem With It - one sharp sentence on what it fails to account for.",
            "Hypothesis Two - one or two sentences on the rival explanation and what it gets right that the first does not.",
            "The Problem With That One Too - one sentence on its weakness.",
            "The Honest Position - one or two sentences stating plainly that this is unresolved, and what would settle it.",
        ],
        "hard_rules": [
            "Maximum ONE sentence carrying numbers in this entire section.",
            "Both hypotheses must be stated in their strongest form. Do not set up a straw man to knock down.",
            "You must not resolve the debate. The unresolvedness IS the content here, and it is what makes the outro land.",
        ],
    },
    "mechanism_only": {
        "name": "The Machinery",
        "words": (250, 300),
        "rhythm": "A process unfolding in order. Each step short, each consequence vivid.",
        "beats": [
            "The Trigger - one sentence on what starts the process, in physical terms.",
            "The Chain - two or three sentences walking the process forward one step at a time. Concrete nouns: tissue, nerve, muscle, chemical, timing.",
            "The Speed or Scale - one sharp sentence anchoring how fast, how small, or how often, using a number only if one is in the allowed list.",
            "The Visible Result - one or two sentences connecting the invisible chain to the thing the viewer can actually see or feel on their own body.",
            "The Odd Detail - one sentence on the part of the machinery that seems badly designed.",
        ],
        "hard_rules": [
            "Maximum ONE sentence carrying numbers in this entire section.",
            "Every step must be something happening to a physical thing. No abstractions like 'signals are processed'.",
        ],
    },
}

ev_type = profile.get("evidence_type", "mechanism_only")
if ev_type not in SECTION_3_TEMPLATES:
    print(f"  (unrecognised evidence_type {ev_type!r} - falling back to mechanism_only)")
    ev_type = "mechanism_only"

BEAT_SHEET[3] = SECTION_3_TEMPLATES[ev_type]

# Rebuild in section order so downstream rendering is always 1..6
BEAT_SHEET = {k: BEAT_SHEET[k] for k in sorted(BEAT_SHEET)}

# --- The writer's entire statistics allowance ---
ALLOWED_NUMBERS = [h["value"] for h in profile.get("hard_numbers", [])]

print("\n" + "=" * 70)
print(f"EVIDENCE TYPE: {ev_type}")
print(f"  reason: {profile.get('evidence_type_reason')}")
print(f"  -> Section 3 template: \"{BEAT_SHEET[3]['name']}\"")
print("=" * 70)
print(f"\nNamed entities available: {len(profile.get('named_entities', []))}")
for e in profile.get("named_entities", [])[:8]:
    print(f"  - {e.get('kind')}: {e.get('value')}  [S{e.get('source')}]")
if not profile.get("named_entities"):
    print("  (none - the script will be written without a named pioneer, by design)")

print(f"\nHard numbers the writer is allowed to use: {len(ALLOWED_NUMBERS)}")
for h in profile.get("hard_numbers", []):
    print(f"  - {h.get('value')}: {h.get('means')}  [S{h.get('source')}]")

print(f"\nKey findings: {len(profile.get('key_findings', []))}")
for k in profile.get("key_findings", []):
    print(f"  - [{k.get('certainty')}] {k.get('finding')}")

print(f"\nStrongest hook material:\n  {profile.get('the_best_counter_evidence')}")
if profile.get("gaps"):
    print("\nThings the sources do NOT support (writer is blocked from these):")
    for g in profile["gaps"]:
        print(f"  - {g}")


## Step 8 — Write the script

In [ ]:
def render_section_spec(n: int, spec: dict) -> str:
    """Renders one section of BEAT_SHEET into prompt text. Used by the WRITER here and
    by the RETENTION JUDGE in Cell 8 - same function, same text, so the judge can only
    ever hold the writer to rules the writer was actually given."""
    lo, hi = spec["words"]
    lines = [f'=== SECTION {n}: {spec["name"].upper()} ({lo}-{hi} words) ===',
             f'Rhythm: {spec["rhythm"]}',
             "Beats, in this order:"]
    for i, b in enumerate(spec["beats"], 1):
        lines.append(f"  {i}. {b}")
    if spec.get("hard_rules"):
        lines.append("Hard rules for this section:")
        for r in spec["hard_rules"]:
            lines.append(f"  - {r}")
    return "\n".join(lines)


def render_beat_sheet() -> str:
    return "\n\n".join(render_section_spec(n, BEAT_SHEET[n]) for n in sorted(BEAT_SHEET))


# --- The research brief the writer is allowed to draw on ---
numbers_block = "\n".join(
    f'  - "{h["value"]}" = {h["means"]}  (Source {h["source"]})'
    for h in profile.get("hard_numbers", [])
) or "  (none - this topic has no hard numbers in the sources. Do not use ANY statistics.)"

entities_block = "\n".join(
    f'  - {e["kind"]}: {e["value"]}  (Source {e["source"]})'
    for e in profile.get("named_entities", [])
) or "  (none - do not name any researcher, institution, university or year anywhere in the script.)"

findings_block = "\n".join(
    f'  - [{k["certainty"]}] {k["finding"]}  (Source {k["source"]})'
    for k in profile.get("key_findings", [])
)

gaps_block = "\n".join(f"  - {g}" for g in profile.get("gaps", [])) or "  (none listed)"

banned_block = ", ".join(f'"{p}"' for p in BANNED_PHRASES)

system_prompt = f"""You are the writer for 'Weird Human Biology', a YouTube channel that
takes one ordinary thing about the human body and makes a viewer unable to stop watching.
Write the voiceover script for: {TOPIC}

Your reader is one person, alone, holding a phone. Write to them, not to an audience.

=============================================================================
 WHAT YOU ARE ALLOWED TO SAY
=============================================================================
Everything factual in this script must come from the brief below. This is not a
style note - a sentence containing a fact that is not in this brief will be caught
by an automated checker and cut.

THE ONLY NUMBERS YOU MAY USE (using any other number will be flagged and removed):
{numbers_block}

THE ONLY NAMES, INSTITUTIONS AND YEARS YOU MAY USE:
{entities_block}

KEY FINDINGS (reproduce the certainty label - if it says "debated", the script must
say it is debated, not state it as settled):
{findings_block}

THE MYTH THIS PUSHES BACK ON: {profile.get('the_common_myth')}

YOUR STRONGEST HOOK MATERIAL: {profile.get('the_best_counter_evidence')}

MECHANISM AS THE SOURCES DESCRIBE IT: {profile.get('mechanism')}

THINGS THE SOURCES DO NOT SUPPORT - you may not write these even if they feel true:
{gaps_block}

=============================================================================
 THE BEAT SHEET - follow it exactly, section by section, in order
=============================================================================
Word-count ranges are hard constraints. Each beat must appear, in sequence. Section
boundaries must be completely invisible to a listener - only the pacing and the beats
carry the structure, never a label or a transition announcement.

{render_beat_sheet()}

=============================================================================
 VOICE RULES - these apply to every section
=============================================================================
1. BANNED PHRASES. Never use any of these: {banned_block}
   They are the tells of machine-written narration and an automated linter will
   reject the script if it finds them.

2. NEVER LABEL A FEELING. You may not tell the viewer that something is unsettling,
   fascinating, remarkable, beautiful, or surprising. Describe the thing accurately
   enough that they feel it themselves. If you find yourself reaching for an emotion
   word, you have not described the fact concretely enough yet - go back and add the
   concrete detail instead.

3. ONE ADJECTIVE PER NOUN. Never stack them.

4. SENTENCE LENGTH MUST VARY. After any sentence over 20 words, the next must be
   under 10. A run of three similar-length sentences reads as narration and loses
   the viewer.

5. NEVER RUN TWO FACT-CARRYING SENTENCES BACK TO BACK. A statistic must always be
   followed by something the viewer can picture. Two numbers in a row is where
   people close the tab.

6. CONCRETE NOUNS. "Skin", "muscle", "hair follicle", "cold water" - not "physiological
   response", "biological system", "evolutionary pressure".

7. SECOND PERSON. It is happening in the viewer's body, right now, as they listen.

8. NO CALLS TO ACTION anywhere. No subscribing, no commenting, no thanking.

=============================================================================
 OUTPUT FORMAT - follow exactly
=============================================================================
Write ONE sentence per line. End every single line with a tag in this exact shape:

    [S<section number>|<source numbers>]

Examples:
    Press two fingers against the back of your arm. [S1|0]
    That reflex shows up in nearly every mammal that has been studied. [S3|1]
    Two teams disagree about why, and neither has settled it. [S3|1,2]
    You are still carrying the wiring for a coat you no longer have. [S5|0]

The section number says which beat-sheet section that line belongs to. The source
numbers say where its factual content comes from - use 0 when the line carries no
factual claim at all.

CRITICAL RULE ABOUT [0]: tag a line 0 ONLY if it makes no checkable assertion about
the world. "Your ancestors needed this to survive the cold" is a factual claim and
must cite a source, even though it sounds like narration. Rhetorical questions,
direct address, and pure imagery are the only things that are genuinely 0. Lines
tagged 0 are checked separately for smuggled claims, so mis-tagging costs you.

Every line ends with exactly one tag. No headers, no section names, no stage
directions, no sound cues, no markdown, no commentary before or after. The tags are
stripped before the script is voiced, so each sentence must read naturally on its
own and the whole thing must flow as one continuous piece of speech."""

print("Writing script...")
raw_tagged_script = call_model(
    MODEL_WRITER,
    f"Write the full script for: {TOPIC}",
    system=system_prompt,
    max_tokens=8000,       # was 3500 in v4 - a 1,400-word tagged script can brush that ceiling
    temperature=0.6,
)


TAG_RE = re.compile(r"^(.*?)\s*\[\s*S(\d+)\s*\|\s*([\d,\s]*)\s*\]\s*$", re.IGNORECASE)


def parse_tagged_script(text: str) -> list[dict]:
    """Turns tagged lines into [{'sentence', 'section', 'sources'}].
    Section tags are what let Cells 5 and 8 judge each section against its own spec
    instead of guessing boundaries from running word counts."""
    parsed = []
    current_section = 1
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        m = TAG_RE.match(line)
        if m:
            sentence = m.group(1).strip()
            section = int(m.group(2))
            sources = [int(n) for n in re.findall(r"\d+", m.group(3)) if int(n) != 0]
            current_section = section
        else:
            # Model dropped the tag. Keep the sentence, inherit the running section,
            # mark it untagged so it surfaces in the linter rather than vanishing.
            sentence, section, sources = line, current_section, []
        parsed.append({"sentence": sentence, "section": section, "sources": sources})
    return parsed


def strip_tags_to_voiceover(tagged_sentences: list[dict]) -> str:
    return " ".join(item["sentence"] for item in tagged_sentences)


def save_script(tagged_sentences: list[dict], label: str = ""):
    with open("script_tagged.json", "w", encoding="utf-8") as f:
        json.dump(tagged_sentences, f, indent=2)
    with open("vo_raw.txt", "w", encoding="utf-8") as f:
        f.write(strip_tags_to_voiceover(tagged_sentences))
    if label:
        print(f"  saved ({label})")


tagged_sentences = parse_tagged_script(raw_tagged_script)
save_script(tagged_sentences, "first draft")

master_script = strip_tags_to_voiceover(tagged_sentences)
wc = len(master_script.split())
untagged = sum(1 for i in tagged_sentences if not i["sources"] and i["section"])

print(f"\nDraft written: {wc} words, {len(tagged_sentences)} sentences "
      f"(~{wc/SPOKEN_WPM:.1f} min at {SPOKEN_WPM} wpm)")
by_sec = {}
for it in tagged_sentences:
    by_sec.setdefault(it["section"], []).append(it)
for n in sorted(by_sec):
    words = sum(len(i["sentence"].split()) for i in by_sec[n])
    name = BEAT_SHEET.get(n, {}).get("name", "?")
    tgt = BEAT_SHEET.get(n, {}).get("words", (0, 0))
    flag = "" if tgt[0] <= words <= tgt[1] else "  <-- OUT OF RANGE"
    print(f"  S{n} {name:<24} {words:>4} words (target {tgt[0]}-{tgt[1]}){flag}")


## Step 9 — Lint

Free. No API calls. Catches ungrounded numbers, banned phrases, blown word counts.

In [ ]:
# =============================================================================
#  DETERMINISTIC LINTER - runs before any judge model is called
# =============================================================================
# Everything here is a rule that can be checked with code. Doing these mechanically
# means the paid judges in Cells 6-8 spend their attention on the things that genuinely
# need judgment, and it means the cheap-to-catch failures (banned phrase, blown word
# count, a number that appears nowhere in the sources) are caught every single run
# rather than whenever a judge happens to notice.
# =============================================================================

NUM_WORDS = {
    "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
    "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14,
    "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19,
    "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60,
    "seventy": 70, "eighty": 80, "ninety": 90, "hundred": 100, "thousand": 1000,
    "million": 1000000, "billion": 1000000000,
}

ALL_SOURCE_TEXT = " ".join(s["excerpt"] for s in source_pack).lower()


def _number_in_sources(token: str) -> bool:
    """A number counts as grounded if it appears in the source text as digits or as a
    word, in either direction. Deliberately generous - the goal is to catch inventions
    like '532 skulls' when no source ever said 532, not to police rounding."""
    t = token.lower().strip().rstrip("%").replace(",", "")
    if t in ALL_SOURCE_TEXT or token.lower() in ALL_SOURCE_TEXT:
        return True
    if t.isdigit():
        n = int(t)
        for w, v in NUM_WORDS.items():
            if v == n and w in ALL_SOURCE_TEXT:
                return True
        # digits written with separators in the source, e.g. 550 million / 550,000
        if f"{n:,}" in ALL_SOURCE_TEXT:
            return True
    if t in NUM_WORDS and str(NUM_WORDS[t]) in ALL_SOURCE_TEXT:
        return True
    return False


def audit_numbers(tagged_sentences):
    """Every number in the script must trace to the source text. This is the check
    that would have caught the chin script's invented measurement counts."""
    findings = []
    for item in tagged_sentences:
        s = item["sentence"]
        tokens = re.findall(r"\b\d[\d,\.]*%?\b", s)
        tokens += [w for w in re.findall(r"\b[a-z]+\b", s.lower()) if w in NUM_WORDS]
        for tok in set(tokens):
            if not _number_in_sources(tok):
                findings.append({
                    "sentence": s, "section": item["section"],
                    "reason": f"the number '{tok}' does not appear in any fetched source",
                })
    return findings


def audit_banned_phrases(tagged_sentences):
    findings = []
    for item in tagged_sentences:
        low = item["sentence"].lower()
        for phrase in BANNED_PHRASES:
            if phrase in low:
                findings.append({
                    "sentence": item["sentence"], "section": item["section"],
                    "reason": f"contains banned phrase '{phrase}' - rewrite without labelling the feeling or using filler",
                })
    return findings


def audit_stacked_adjectives(tagged_sentences):
    """Catches 'a quiet, ancient, forgotten reflex' - the Section 5 failure mode."""
    findings = []
    pat = re.compile(r"\b(\w+ly\s+)?(\w+),\s+(\w+),?\s+(and\s+)?(\w+)\s+(\w+)\b")
    for item in tagged_sentences:
        for m in pat.finditer(item["sentence"]):
            chunk = m.group(0)
            if len(chunk.split()) <= 6 and chunk.count(",") >= 2:
                findings.append({
                    "sentence": item["sentence"], "section": item["section"],
                    "reason": f"stacked modifiers ('{chunk}') - one adjective per noun",
                })
                break
    return findings


def audit_stat_runs(tagged_sentences):
    """Two or more source-carrying sentences back to back. This is the mechanical
    version of the pacing problem: 'X skulls... Y landmarks... Z distances'."""
    findings = []
    run = []
    for item in tagged_sentences + [{"sentence": "", "section": 0, "sources": []}]:
        if item["sources"]:
            run.append(item)
        else:
            if len(run) >= 2:
                findings.append({
                    "sentence": run[1]["sentence"], "section": run[1]["section"],
                    "reason": (f"{len(run)} fact-carrying sentences in a row - break the run with "
                               f"an image or a consequence the viewer can picture"),
                })
            run = []
    return findings


def audit_sentence_rhythm(tagged_sentences):
    """Three consecutive sentences within 3 words of each other reads as flat narration."""
    findings = []
    lens = [len(i["sentence"].split()) for i in tagged_sentences]
    for i in range(len(lens) - 2):
        w = lens[i:i + 3]
        if max(w) - min(w) <= 3 and min(w) >= 12:
            findings.append({
                "sentence": tagged_sentences[i + 1]["sentence"],
                "section": tagged_sentences[i + 1]["section"],
                "reason": f"three consecutive sentences of similar length ({w}) - vary the rhythm",
            })
    return findings


def audit_structure(tagged_sentences):
    """Beat-level structural checks that don't need a model."""
    findings = []
    by_sec = {}
    for it in tagged_sentences:
        by_sec.setdefault(it["section"], []).append(it)

    # --- word counts per section ---
    for n, spec in BEAT_SHEET.items():
        sents = by_sec.get(n, [])
        words = sum(len(i["sentence"].split()) for i in sents)
        lo, hi = spec["words"]
        if not sents:
            findings.append({"sentence": "", "section": n,
                             "reason": f"Section {n} ({spec['name']}) is missing entirely"})
        elif words < lo * 0.85 or words > hi * 1.15:
            findings.append({"sentence": "", "section": n,
                             "reason": f"Section {n} is {words} words, target {lo}-{hi}"})

    # --- S1 must contain a 2-4 word Snap ---
    s1 = by_sec.get(1, [])
    if s1 and not any(2 <= len(i["sentence"].rstrip(".!?").split()) <= 4 for i in s1):
        findings.append({"sentence": "", "section": 1,
                         "reason": "Section 1 has no 2-4 word Snap line - the Absolute Negation beat is missing or too long"})

    # --- S1 sentence-length ceiling ---
    for i in s1:
        if len(i["sentence"].split()) > 14:
            findings.append({"sentence": i["sentence"], "section": 1,
                             "reason": f"Section 1 sentence is {len(i['sentence'].split())} words (max 14) - the hook must stay staccato"})

    # --- S6 must end on a real 3-8 word question, and carry no claims ---
    s6 = by_sec.get(6, [])
    if s6:
        last = s6[-1]["sentence"].strip()
        n_words = len(last.rstrip("?").split())
        if not last.endswith("?"):
            findings.append({"sentence": last, "section": 6,
                             "reason": "the script does not end on a question"})
        elif not (3 <= n_words <= 8):
            findings.append({"sentence": last, "section": 6,
                             "reason": f"closing question is {n_words} words (target 3-8)"})
        if re.match(r"^(isn't|aren't|is it not|who knew|how amazing|don't you)", last.lower()):
            findings.append({"sentence": last, "section": 6,
                             "reason": "closing line is a rhetorical statement wearing a question mark, not an open question"})
        for i in s6:
            if i["sources"]:
                findings.append({"sentence": i["sentence"], "section": 6,
                                 "reason": "Section 6 must carry no new factual claims"})

    # --- somatic loop: outro should reuse hook vocabulary ---
    if s1 and s6:
        stop = set("the a an and or but of to in on it is are was were you your this that with for as at be".split())
        hook_words = set(w for w in re.findall(r"[a-z]+", " ".join(i["sentence"].lower() for i in s1)) if w not in stop and len(w) > 3)
        outro_words = set(w for w in re.findall(r"[a-z]+", " ".join(i["sentence"].lower() for i in s6)) if w not in stop and len(w) > 3)
        if len(hook_words & outro_words) < 2:
            findings.append({"sentence": "", "section": 6,
                             "reason": "the outro does not audibly loop back to the hook - reuse the specific body part and action words from Section 1"})

    return findings


def run_linter(tagged_sentences):
    checks = {
        "ungrounded numbers": audit_numbers(tagged_sentences),
        "banned phrases": audit_banned_phrases(tagged_sentences),
        "stacked adjectives": audit_stacked_adjectives(tagged_sentences),
        "fact-sentence runs": audit_stat_runs(tagged_sentences),
        "flat rhythm": audit_sentence_rhythm(tagged_sentences),
        "structure": audit_structure(tagged_sentences),
    }
    all_issues = []
    print("LINTER RESULTS")
    print("-" * 70)
    for name, issues in checks.items():
        status = "PASS" if not issues else f"{len(issues)} issue(s)"
        print(f"  {name:<22} {status}")
        for iss in issues:
            snippet = (iss["sentence"][:70] + "...") if len(iss["sentence"]) > 70 else iss["sentence"]
            print(f"      S{iss['section']}: {iss['reason']}")
            if snippet:
                print(f"        > {snippet}")
        all_issues.extend(issues)
    print("-" * 70)
    print(f"  {len(all_issues)} total issue(s)\n")
    return all_issues


lint_issues = run_linter(tagged_sentences)
with open("lint_round_1.json", "w", encoding="utf-8") as f:
    json.dump(lint_issues, f, indent=2)


## Step 10 — Fact-check tools

In [ ]:
def validate_claims(tagged_sentences, source_pack):
    """Checks every source-carrying sentence against ONLY the source(s) it cites."""
    claim_items = [i for i in tagged_sentences if i["sources"]]
    if not claim_items:
        return []

    claims_block = "\n".join(
        f'{i+1}. "{it["sentence"]}" - cites Source(s) {it["sources"]}'
        for i, it in enumerate(claim_items)
    )
    prompt = f"""You are a fact-checker. Below are numbered SOURCE EXCERPTS and a list of
CLAIMS, each tagged with which source(s) it is supposed to come from.

For each claim, check it ONLY against the source(s) it cites. A claim is unsupported if
that specific source does not directly support it. Paraphrase is fine. New specifics,
numbers, mechanisms, studies, names or years that are not in the cited source are NOT
supported, even if another source in the pack happens to cover them - that is a
citation error and counts as an issue.

Also flag a claim if the source presents it as debated, contested or one team's view but
the sentence states it as settled fact.

Respond with ONLY valid JSON, no fences:
{{"issues": [{{"sentence": "the sentence verbatim", "reason": "..."}}]}}

SOURCE EXCERPTS:
{sources_block}

CLAIMS:
{claims_block}"""

    result = parse_json_response(call_model(MODEL_CHECKER, prompt, max_tokens=3000, temperature=0))
    issues = result.get("issues", [])
    for i in issues:
        i["kind"] = "fact"
    return issues


def audit_untagged_claims(tagged_sentences, source_pack):
    """THE GAP v4 LEFT WIDE OPEN.

    v4's validator only looked at sentences tagged with a source. But the beat sheet
    pushes Sections 5 and 6 to a very low stat ratio, so roughly 40% of the finished
    script is tagged [0] and was never checked by anything. A sentence like 'your
    ancestors would have frozen to death without this' is a hard factual claim about
    the world, and tagged [0] it sailed straight through to the voiceover.

    This pass reads every [0] sentence and asks one question: does this assert something
    checkable? If yes, it has to be either grounded and re-tagged, or softened into
    genuine narration."""
    zero_items = [i for i in tagged_sentences if not i["sources"]]
    if not zero_items:
        return []

    block = "\n".join(f'{i+1}. "{it["sentence"]}"' for i, it in enumerate(zero_items))
    prompt = f"""These sentences from a science voiceover were marked as carrying NO factual
claim - pure narration, direct address, rhetorical question, or imagery.

Your job is to find the ones that are actually smuggling a factual assertion. A sentence
carries a factual claim if a reasonable viewer could ask "is that true?" and expect an
answer from evidence.

Carries a claim (FLAG IT):
  - "Your ancestors relied on this to survive freezing nights."
  - "This reflex disappears completely after the age of forty."
  - "Every mammal on earth does this."

Does not carry a claim (LEAVE IT):
  - "Run your hand over your forearm."      (a directive)
  - "So why does it still happen?"          (a question)
  - "You are sitting in a heated room."     (about the viewer's setting, not a claim about the world)
  - "It feels like nothing at all."         (subjective description)

For each flagged sentence, also check it against the SOURCE EXCERPTS and say whether the
sources actually support it.

Respond with ONLY valid JSON, no fences:
{{"issues": [{{"sentence": "verbatim", "reason": "states X as fact; sources do/do not support this"}}]}}

SOURCE EXCERPTS:
{sources_block}

SENTENCES MARKED AS NON-FACTUAL:
{block}"""

    result = parse_json_response(call_model(MODEL_CHECKER, prompt, max_tokens=3000, temperature=0))
    issues = result.get("issues", [])
    for i in issues:
        i["kind"] = "smuggled-claim"
    return issues


# =============================================================================
#  THE SHARED FIXER
# =============================================================================
# v4 mapped rewrites back onto originals by POSITION, through a dict keyed on the
# rewritten sentence text. Two ways that silently corrupted a script: if the model
# dropped one sentence, every later fix shifted up by one and attached to the wrong
# original; and if two rewrites came back identical, the dict deduplicated them and
# shifted everything after. Neither raised an error - you just got a script with
# sentences swapped into the wrong places.
#
# This version gives every flagged sentence a stable numeric id and requires the model
# to echo it back. Order-independent, drop-safe, duplicate-safe.
# =============================================================================

def apply_fixes(tagged_sentences, issues, instruction, temperature=0.4):
    if not issues:
        return tagged_sentences

    index_by_text = {}
    for idx, it in enumerate(tagged_sentences):
        index_by_text.setdefault(it["sentence"], idx)

    targets = {}
    for iss in issues:
        idx = index_by_text.get(iss["sentence"])
        if idx is None:
            # fuzzy fallback - the judge occasionally normalises punctuation
            for text, i in index_by_text.items():
                if iss["sentence"][:45] and iss["sentence"][:45] in text:
                    idx = i
                    break
        if idx is None:
            print(f"  [warn] could not locate flagged sentence, skipping: {iss['sentence'][:60]!r}")
            continue
        targets.setdefault(idx, []).append(iss["reason"])

    if not targets:
        return tagged_sentences

    lines = []
    for idx in sorted(targets):
        it = tagged_sentences[idx]
        src_tag = ",".join(map(str, it["sources"])) if it["sources"] else "0"
        lines.append(f'#{idx} [S{it["section"]}|{src_tag}] "{it["sentence"]}"')
        for r in targets[idx]:
            lines.append(f"     PROBLEM: {r}")
    flagged_block = "\n".join(lines)

    context_lines = []
    for idx in sorted(targets):
        before = tagged_sentences[idx - 1]["sentence"] if idx > 0 else "(start of script)"
        after = tagged_sentences[idx + 1]["sentence"] if idx + 1 < len(tagged_sentences) else "(end of script)"
        context_lines.append(f'#{idx} sits between: "{before}"  ...  "{after}"')
    context_block = "\n".join(context_lines)

    prompt = f"""{instruction}

You are editing individual lines inside a finished voiceover script. Each flagged line has
an id. Rewrite the line so it solves the stated problem while still reading naturally in
the gap it sits in.

ABSOLUTE CONSTRAINT: every factual thing in your rewrite must still be supported by the
source excerpts below. A punchier line that invents a detail is worse than the line you
started with. If nothing in the sources can support a version of the line, delete it.

BEAT SHEET (the line must still perform its section's job):
{render_beat_sheet()}

THE ONLY NUMBERS ALLOWED ANYWHERE IN THIS SCRIPT:
{numbers_block}

THE ONLY NAMES / INSTITUTIONS / YEARS ALLOWED:
{entities_block}

NEVER USE THESE PHRASES: {banned_block}

SOURCE EXCERPTS:
{sources_block}

WHERE EACH LINE SITS:
{context_block}

FLAGGED LINES:
{flagged_block}

OUTPUT FORMAT - one line per fix, nothing else, no commentary:
    #<id> | <rewritten sentence> [S<section>|<sources>]
or, to delete the line entirely:
    #<id> | DROP

Echo the id exactly. Keep the section number the same. Update the source numbers if your
rewrite now draws on a different source, or use 0 if it is now pure narration."""

    raw = call_model(MODEL_WRITER, prompt, max_tokens=4000, temperature=temperature)

    replacements = {}
    for line in raw.splitlines():
        line = line.strip()
        m = re.match(r"^#(\d+)\s*\|\s*(.+)$", line)
        if not m:
            continue
        idx, body = int(m.group(1)), m.group(2).strip()
        if body.upper().startswith("DROP"):
            replacements[idx] = None
            continue
        tm = TAG_RE.match(body)
        if tm:
            replacements[idx] = {
                "sentence": tm.group(1).strip(),
                "section": int(tm.group(2)),
                "sources": [int(n) for n in re.findall(r"\d+", tm.group(3)) if int(n) != 0],
            }
        else:
            replacements[idx] = {
                "sentence": body,
                "section": tagged_sentences[idx]["section"] if idx < len(tagged_sentences) else 1,
                "sources": [],
            }

    out = []
    for idx, it in enumerate(tagged_sentences):
        if idx in replacements:
            rep = replacements[idx]
            if rep is not None:
                out.append(rep)
            # None -> deliberately dropped
        else:
            out.append(it)

    n_fixed = sum(1 for v in replacements.values() if v is not None)
    n_dropped = sum(1 for v in replacements.values() if v is None)
    n_missed = len(targets) - len(replacements)
    print(f"  {n_fixed} rewritten, {n_dropped} dropped, {n_missed} not returned by the model")
    return out


print("Fact-checking cited claims...")
fact_issues = validate_claims(tagged_sentences, source_pack)
print(f"  {len(fact_issues)} unsupported claim(s)")
for i in fact_issues:
    print(f"    - {i['sentence'][:70]}...\n        {i['reason']}")

print("\nAuditing [0]-tagged lines for smuggled factual claims...")
smuggled_issues = audit_untagged_claims(tagged_sentences, source_pack)
print(f"  {len(smuggled_issues)} smuggled claim(s)")
for i in smuggled_issues:
    print(f"    - {i['sentence'][:70]}...\n        {i['reason']}")

with open("validation_round_1.json", "w", encoding="utf-8") as f:
    json.dump({"fact": fact_issues, "smuggled": smuggled_issues, "lint": lint_issues}, f, indent=2)


## Step 11 — Fix the facts

Up to 3 rounds.

In [ ]:
MAX_GROUNDING_ROUNDS = 3


def rewrite_section(tagged_sentences, section_n, reasons):
    """Some problems can't be fixed one line at a time - a section 90 words over target,
    or missing a beat entirely, needs the whole section rebuilt against its spec. v4 had
    no mechanism for this at all; a blown word count just shipped."""
    spec = BEAT_SHEET.get(section_n)
    if not spec:
        return tagged_sentences

    idxs = [i for i, it in enumerate(tagged_sentences) if it["section"] == section_n]
    if not idxs:
        start = next((i for i, it in enumerate(tagged_sentences) if it["section"] > section_n), len(tagged_sentences))
        end = start
        current_block = "(this section is missing from the script entirely)"
    else:
        start, end = idxs[0], idxs[-1] + 1
        current_block = "\n".join(
            f'{it["sentence"]} [S{it["section"]}|{",".join(map(str, it["sources"])) or "0"}]'
            for it in tagged_sentences[start:end]
        )

    before = tagged_sentences[start - 1]["sentence"] if start > 0 else "(start of script)"
    after = tagged_sentences[end]["sentence"] if end < len(tagged_sentences) else "(end of script)"

    prompt = f"""Rewrite ONE section of a voiceover script so it meets its spec.

PROBLEMS WITH THE CURRENT VERSION:
{chr(10).join('  - ' + r for r in reasons)}

THE SPEC THIS SECTION MUST MEET:
{render_section_spec(section_n, spec)}

It must run straight on from: "{before}"
and hand off cleanly into: "{after}"

Keep every fact that is already grounded. Do not introduce any new factual claim.

THE ONLY NUMBERS ALLOWED:
{numbers_block}

THE ONLY NAMES / INSTITUTIONS / YEARS ALLOWED:
{entities_block}

NEVER USE THESE PHRASES: {banned_block}

VOICE RULES: second person; concrete nouns; one adjective per noun; never label a feeling
for the viewer; vary sentence length; never put two fact-carrying sentences side by side.

SOURCE EXCERPTS:
{sources_block}

CURRENT VERSION OF THE SECTION:
{current_block}

Return ONLY the rewritten section: one sentence per line, each ending with
[S{section_n}|<source numbers>] or [S{section_n}|0]. No commentary."""

    raw = call_model(MODEL_WRITER, prompt, max_tokens=3000, temperature=0.55)
    new_items = parse_tagged_script(raw)
    for it in new_items:
        it["section"] = section_n
    if not new_items:
        print(f"  [warn] section {section_n} rewrite returned nothing - keeping original")
        return tagged_sentences
    print(f"  section {section_n} rewritten: {len(new_items)} lines, "
          f"{sum(len(i['sentence'].split()) for i in new_items)} words")
    return tagged_sentences[:start] + new_items + tagged_sentences[end:]


GROUNDING_INSTRUCTION = (
    "These voiceover lines failed a grounding check: they either state something the cited "
    "source does not support, present a debated finding as settled, use a number that appears "
    "in no source, or smuggle a factual claim into a line marked as pure narration. Fix each "
    "one so it is fully supported, correctly tagged, and still reads well aloud."
)

round_num = 0
while round_num < MAX_GROUNDING_ROUNDS:
    sentence_issues = fact_issues + smuggled_issues + [i for i in lint_issues if i.get("sentence")]
    section_issues = {}
    for i in lint_issues:
        if not i.get("sentence"):
            section_issues.setdefault(i["section"], []).append(i["reason"])

    if not sentence_issues and not section_issues:
        break

    round_num += 1
    print(f"\n--- Grounding fix round {round_num}/{MAX_GROUNDING_ROUNDS} ---")

    if section_issues:
        for sec_n in sorted(section_issues, reverse=True):   # back to front, indices stay valid
            print(f"  rebuilding section {sec_n}: {section_issues[sec_n]}")
            tagged_sentences = rewrite_section(tagged_sentences, sec_n, section_issues[sec_n])

    if sentence_issues:
        print(f"  repairing {len(sentence_issues)} flagged line(s)")
        tagged_sentences = apply_fixes(tagged_sentences, sentence_issues, GROUNDING_INSTRUCTION, temperature=0.35)

    print("\n  re-checking...")
    lint_issues = run_linter(tagged_sentences)
    fact_issues = validate_claims(tagged_sentences, source_pack)
    smuggled_issues = audit_untagged_claims(tagged_sentences, source_pack)
    print(f"  after round {round_num}: {len(fact_issues)} fact, "
          f"{len(smuggled_issues)} smuggled, {len(lint_issues)} lint")

    with open(f"validation_round_{round_num+1}.json", "w", encoding="utf-8") as f:
        json.dump({"fact": fact_issues, "smuggled": smuggled_issues, "lint": lint_issues}, f, indent=2)

save_script(tagged_sentences, "post-grounding")

remaining = len(fact_issues) + len(smuggled_issues) + len(lint_issues)
if remaining == 0:
    print("\nGROUNDING GATE PASSED - every claim traces to a source, every number is real.")
else:
    print(f"\nGROUNDING GATE: {remaining} issue(s) remain after {MAX_GROUNDING_ROUNDS} rounds.")
    print("The script continues to the retention gate, but read validation_round_*.json")
    print("before you publish - these are the lines a viewer could fact-check you on.")


## Step 12 — Retention check

In [ ]:
# =============================================================================
#  RETENTION GATE
# =============================================================================
# Runs AFTER the grounding gate, never merged into it. Fact-checking is binary and
# source-anchored: cite it or cut it. Retention is a judgment about delivery. Merging
# them into one prompt lets the model trade accuracy for punchiness to satisfy the
# fuzzier objective. Kept separate and run second, this gate inherits clean sourced
# sentences and is only allowed to reshape how they land.
#
# The judge renders its criteria from the SAME BEAT_SHEET dict the writer was given
# (render_section_spec), so it can only hold the script to rules the writer actually
# received. That is why the beat sheet lives in a dict in Cell 2 rather than being
# typed out inside a prompt string.
# =============================================================================

RETENTION_SYSTEM = """You are a retention editor for a YouTube science channel with a
brutal standard: every sentence has to earn the next one. You are not fact-checking -
another pass already did that, and every claim here is sourced. Your only question is
whether a viewer keeps watching.

You are strict. A section that is merely competent gets flagged. You are looking for:

  - BEAT FAILURE: a beat from the spec is missing, out of order, or technically present
    but doing none of the work it was supposed to do.
  - PACING: fact-carrying sentences stacked together; a run of similar-length sentences;
    a paragraph where nothing changes register.
  - TELLING NOT SHOWING: the script naming an emotion instead of producing it; adjective
    stacking; "this is fascinating" instead of a fascinating detail.
  - DEAD WEIGHT: a sentence that restates the previous one, sets up something that never
    arrives, or could be deleted with no loss.
  - ABSTRACTION: a metaphor the viewer cannot physically picture; a mechanism described
    in category words instead of concrete nouns.
  - FLAT OPENINGS: sentences that begin with the same construction repeatedly.

Do not flag something just to have flagged it. If a line works, leave it. But do not
pass a section that a viewer would scrub past."""

# Real audience data, when it exists, is a stronger signal than any prior baked into
# the prompt above — appended rather than interleaved so it's always visible as
# "what this channel's own viewers have actually done", not mixed into house style.
RETENTION_SYSTEM += f"""

LEARNED FROM THIS CHANNEL'S OWN PUBLISHED VIDEOS (real viewer retention data, not a prior):
{learnings_block(["retention_notes"])}
"""


def judge_section(tagged_sentences, section_n):
    spec = BEAT_SHEET.get(section_n)
    items = [it for it in tagged_sentences if it["section"] == section_n]
    if not spec or not items:
        return []

    idxs = [i for i, it in enumerate(tagged_sentences) if it["section"] == section_n]
    prev_line = tagged_sentences[idxs[0] - 1]["sentence"] if idxs[0] > 0 else "(this is the opening of the video)"
    next_line = tagged_sentences[idxs[-1] + 1]["sentence"] if idxs[-1] + 1 < len(tagged_sentences) else "(this is the end of the video)"

    body = "\n".join(
        f'{it["sentence"]}   <{"FACT" if it["sources"] else "narration"}>'
        for it in items
    )
    words = sum(len(it["sentence"].split()) for it in items)

    seam_note = ""
    if section_n == 1:
        seam_note = ("This is the first 30-40 seconds of the video. It decides everything. Judge it "
                     "harder than any other section: if the first sentence does not make someone stop "
                     "scrolling, nothing else matters.")
    elif section_n == 4:
        seam_note = ("This section sits around the 4-minute mark, where the largest single block of "
                     "viewers leaves. Its first beat is a Re-Hook and it must genuinely reopen the "
                     "mystery, not just continue explaining.")
    elif section_n == 6:
        seam_note = ("This is the close. The final line has to be a real open question a viewer could "
                     "argue about - not a statement with a question mark, and never a call to action.")

    prompt = f"""Judge ONE section of a voiceover script about "{TOPIC}".

THE SPEC THIS SECTION WAS WRITTEN TO:
{render_section_spec(section_n, spec)}

{seam_note}

THE SEAM: this section must run on naturally from the line before it and hand off into
the line after it. Judge both joins.
  Line immediately before: "{prev_line}"
  Line immediately after:  "{next_line}"

THE SECTION AS WRITTEN ({words} words, target {spec['words'][0]}-{spec['words'][1]}).
Each line is marked FACT (carries a sourced claim) or narration:
{body}

Respond with ONLY valid JSON, no fences:
{{"verdict": "pass" or "fail",
  "issues": [{{"sentence": "the sentence verbatim, exactly as written above",
               "reason": "what is wrong and what would fix it, in one sentence"}}]}}

Quote each flagged sentence exactly. If the problem is a missing beat rather than a bad
line, quote the sentence where the missing beat should have gone and say so in the reason."""

    result = parse_json_response(
        call_model(MODEL_JUDGE, prompt, system=RETENTION_SYSTEM, max_tokens=3000, temperature=0.2)
    )
    issues = result.get("issues", [])
    for i in issues:
        i["kind"] = "retention"
        i["section"] = section_n
    return issues


def judge_whole_script(tagged_sentences):
    """One pass over the arc, not the sections. Catches the failure that is invisible
    section by section: the video is fine everywhere and dull overall."""
    body = "\n".join(f'[S{it["section"]}] {it["sentence"]}' for it in tagged_sentences)
    prompt = f"""Read this complete voiceover script for "{TOPIC}" end to end and judge it
as one viewing experience.

Look for problems that only show up at full length:
  - the same idea explained twice in different sections
  - a promise made early that the script never pays off
  - tension that peaks in the middle and then sags
  - a metaphor or image reused until it stops working
  - the middle third being where you would stop watching

Respond with ONLY valid JSON, no fences:
{{"issues": [{{"sentence": "verbatim line where the problem is worst", "reason": "..."}}]}}

Flag at most 6 issues, only real ones.

SCRIPT:
{body}"""
    result = parse_json_response(
        call_model(MODEL_JUDGE, prompt, system=RETENTION_SYSTEM, max_tokens=2500, temperature=0.2)
    )
    issues = result.get("issues", [])
    for i in issues:
        i["kind"] = "arc"
        i["section"] = 0
    return issues


def run_retention_gate(tagged_sentences):
    all_issues = []
    print("RETENTION GATE")
    print("-" * 70)
    for n in sorted(BEAT_SHEET):
        issues = judge_section(tagged_sentences, n)
        name = BEAT_SHEET[n]["name"]
        print(f"  S{n} {name:<24} {'PASS' if not issues else f'{len(issues)} issue(s)'}")
        for i in issues:
            print(f"      {i['reason']}")
            print(f"        > {i['sentence'][:75]}")
        all_issues.extend(issues)

    arc = judge_whole_script(tagged_sentences)
    print(f"  {'whole-script arc':<28} {'PASS' if not arc else f'{len(arc)} issue(s)'}")
    for i in arc:
        print(f"      {i['reason']}")
        print(f"        > {i['sentence'][:75]}")
    all_issues.extend(arc)
    print("-" * 70)
    print(f"  {len(all_issues)} total retention issue(s)\n")
    return all_issues


retention_issues = run_retention_gate(tagged_sentences)
with open("retention_round_1.json", "w", encoding="utf-8") as f:
    json.dump(retention_issues, f, indent=2)


## Step 13 — Hook tournament

Five different cold opens, judged against each other. The winner replaces the one in the script.

In [ ]:
# =============================================================================
#  HOOK TOURNAMENT
# =============================================================================
# The first 30 seconds decide whether the other eight minutes are watched at all, and
# in v4 (and in Cell 4 above) they got exactly one attempt, same as every other section.
# That is the worst allocation of effort in the whole pipeline. This cell generates
# several genuinely different cold opens, has a judge rank them against each other,
# grounds the winner, and splices it in.
#
# Cheap: 1 generation call + 1 judging call + 1 validation call, no GPU.
# =============================================================================

N_HOOKS = 5
RUN_HOOK_TOURNAMENT = True

HOOK_ANGLES = [
    "Start from a physical sensation the viewer can produce in their own body in the next two seconds.",
    "Start from the single strangest concrete detail in the research, stated flatly with no setup.",
    "Start by stating the thing everyone believes, in the confident voice of someone who believes it, then break it.",
    "Start from an absence - something that should be there and is not, or should happen and does not.",
    "Start mid-scene, as though the viewer walked in on something already happening.",
]

if RUN_HOOK_TOURNAMENT:
    hook_spec = render_section_spec(1, BEAT_SHEET[1])
    angles_block = "\n".join(f"{i+1}. {a}" for i, a in enumerate(HOOK_ANGLES[:N_HOOKS]))

    gen_prompt = f"""Write {N_HOOKS} completely different cold opens for a YouTube video
about: {TOPIC}

Each one must satisfy this spec in full:

{hook_spec}

Use a different opening strategy for each. Strategy per candidate:
{angles_block}

These must be genuinely different openings - not one hook reworded {N_HOOKS} times. Different
first sentence, different route in, different image.

THE ONLY NUMBERS YOU MAY USE ANYWHERE:
{numbers_block}

THE ONLY NAMES / INSTITUTIONS / YEARS YOU MAY USE:
{entities_block}

THE MYTH TO BREAK: {profile.get('the_common_myth')}
THE STRONGEST CONCRETE DETAIL AVAILABLE: {profile.get('the_best_counter_evidence')}

NEVER USE THESE PHRASES: {banned_block}

Output format - exactly this, nothing else:

=== HOOK 1 ===
<sentence> [S1|<sources>]
<sentence> [S1|<sources>]
=== HOOK 2 ===
...

One sentence per line, every line tagged, no commentary."""

    print(f"Generating {N_HOOKS} candidate cold opens...")
    raw_hooks = call_model(MODEL_WRITER, gen_prompt, max_tokens=4000, temperature=0.95)

    candidates = []
    for block in re.split(r"===\s*HOOK\s*\d+\s*===", raw_hooks):
        block = block.strip()
        if not block:
            continue
        items = parse_tagged_script(block)
        for it in items:
            it["section"] = 1
        if items:
            candidates.append(items)

    # keep the one we already have in the running - it was written to the same spec
    incumbent = [it for it in tagged_sentences if it["section"] == 1]
    if incumbent:
        candidates.append([dict(i) for i in incumbent])

    print(f"  {len(candidates)} candidates (including the current one)")

    listing = "\n\n".join(
        f"--- CANDIDATE {i+1} ---\n" + "\n".join(it["sentence"] for it in c)
        for i, c in enumerate(candidates)
    )

    judge_prompt = f"""Rank these candidate cold opens for a YouTube video about "{TOPIC}".

Judge them the way a viewer does: the first sentence appears, and within about two seconds
they either keep watching or swipe. Nothing else about the video exists yet.

Rank on, in order of weight:
  1. STOP POWER of the very first sentence on its own. Does it create a specific gap the
     viewer needs closed? A generic opener loses no matter how good sentence four is.
  2. SPECIFICITY. Concrete, physical, checkable. Vague openers feel like every other video.
  3. THE SNAP. Is the negation genuinely 2-4 words and does it land hard?
  4. THE PROMISE. Does the last line make the next section feel unmissable?
  5. NOT SOUNDING WRITTEN. Any candidate that reads like narration rather than someone
     talking to one person should drop.

Be decisive. Do not rank them all as similar quality - say which one wins and why the
others lose.

Respond with ONLY valid JSON, no fences:
{{"winner": <candidate number>,
  "ranking": [<candidate numbers, best first>],
  "why_winner": "two sentences",
  "why_others_lost": "one sentence naming the single biggest weakness across the losers",
  "improvement": "one specific change that would make the winner better, or null"}}

{listing}"""

    print("  judging...")
    verdict = parse_json_response(
        call_model(MODEL_JUDGE, judge_prompt, system=RETENTION_SYSTEM if "RETENTION_SYSTEM" in dir() else None,
                   max_tokens=1500, temperature=0.2)
    )

    win_idx = int(verdict.get("winner", 1)) - 1
    win_idx = max(0, min(win_idx, len(candidates) - 1))
    winner = candidates[win_idx]

    print("\n" + "=" * 70)
    print(f"WINNER: candidate {win_idx + 1}"
          f"{'  (the original hook held up)' if win_idx == len(candidates) - 1 and incumbent else ''}")
    print("=" * 70)
    for it in winner:
        print(f"  {it['sentence']}")
    print(f"\n  why: {verdict.get('why_winner')}")
    print(f"  losers: {verdict.get('why_others_lost')}")
    if verdict.get("improvement"):
        print(f"  suggested tweak: {verdict['improvement']}")
    print(f"  ranking: {verdict.get('ranking')}")

    with open("hook_tournament.json", "w", encoding="utf-8") as f:
        json.dump({
            "verdict": verdict,
            "candidates": [[i["sentence"] for i in c] for c in candidates],
        }, f, indent=2)

    # --- splice the winner in ---
    rest = [it for it in tagged_sentences if it["section"] != 1]
    tagged_sentences = winner + rest

    # --- the winner is brand new text, so it has to clear the same gates ---
    print("\n  grounding the new hook...")
    hook_fact = validate_claims(winner, source_pack)
    hook_smug = audit_untagged_claims(winner, source_pack)
    hook_issues = hook_fact + hook_smug
    if hook_issues:
        print(f"  {len(hook_issues)} issue(s) in the winning hook - repairing")
        for i in hook_issues:
            print(f"    - {i['reason']}")
        tagged_sentences = apply_fixes(tagged_sentences, hook_issues, GROUNDING_INSTRUCTION, temperature=0.3)
    else:
        print("  hook is clean.")

    save_script(tagged_sentences, "hook spliced")
    print(f"\n  Section 1 is now {sum(len(i['sentence'].split()) for i in tagged_sentences if i['section']==1)} words.")
else:
    print("Hook tournament skipped (RUN_HOOK_TOURNAMENT = False).")


## Step 14 — Fix the pacing

Up to 2 rounds, then every edit goes back through the fact-check.

In [ ]:
MAX_RETENTION_ROUNDS = 2   # deliberately lower than the grounding cap. Retention is a
                           # subjective target, and a model chasing "pass" for three rounds
                           # will over-punch one section until it no longer matches the
                           # tone of the ones around it.

RETENTION_INSTRUCTION = (
    "These voiceover lines are factually clean but are not holding the viewer. Rewrite each "
    "one so it lands harder while keeping exactly the same factual content. You are changing "
    "delivery, not substance: sharpen the image, cut the dead weight, vary the rhythm, replace "
    "a named emotion with the concrete detail that produces it. Do not add a single fact, "
    "number, name or year that is not already in the line you are rewriting."
)

for r in range(1, MAX_RETENTION_ROUNDS + 1):
    if not retention_issues:
        break
    print(f"\n--- Retention fix round {r}/{MAX_RETENTION_ROUNDS} ---")
    tagged_sentences = apply_fixes(tagged_sentences, retention_issues, RETENTION_INSTRUCTION, temperature=0.55)

    print("\n  re-judging...")
    retention_issues = run_retention_gate(tagged_sentences)
    with open(f"retention_round_{r+1}.json", "w", encoding="utf-8") as f:
        json.dump(retention_issues, f, indent=2)


# =============================================================================
#  MANDATORY RE-GROUNDING
# =============================================================================
# The retention fixer just rewrote sentences for punchiness. "Punchier" is exactly the
# pressure that produces a confident invented specific. Every rewrite therefore goes back
# through the full grounding gate. Without this, the retention pass would quietly reopen
# the hole the grounding pass just closed - and it would do it on the lines you care most
# about, because those are the ones it touched.
# =============================================================================

print("\n" + "=" * 70)
print("RE-GROUNDING after retention edits")
print("=" * 70)

lint_issues = run_linter(tagged_sentences)
fact_issues = validate_claims(tagged_sentences, source_pack)
smuggled_issues = audit_untagged_claims(tagged_sentences, source_pack)
print(f"  {len(fact_issues)} fact, {len(smuggled_issues)} smuggled, {len(lint_issues)} lint")

regrounding_issues = fact_issues + smuggled_issues + [i for i in lint_issues if i.get("sentence")]
if regrounding_issues:
    print(f"\n  the retention pass introduced {len(regrounding_issues)} grounding issue(s) - repairing")
    tagged_sentences = apply_fixes(
        tagged_sentences, regrounding_issues,
        GROUNDING_INSTRUCTION + " Preserve the improved rhythm and imagery where you can, but "
        "grounding wins over punchiness every time.",
        temperature=0.3,
    )
    lint_issues = run_linter(tagged_sentences)
    fact_issues = validate_claims(tagged_sentences, source_pack)
    smuggled_issues = audit_untagged_claims(tagged_sentences, source_pack)

save_script(tagged_sentences, "final")
master_script = strip_tags_to_voiceover(tagged_sentences)

wc = len(master_script.split())
print("\n" + "=" * 70)
print("FINAL SCRIPT")
print("=" * 70)
print(f"  {wc} words, {len(tagged_sentences)} sentences, ~{wc/SPOKEN_WPM:.1f} min at {SPOKEN_WPM} wpm")
print(f"  grounding : {len(fact_issues)} fact, {len(smuggled_issues)} smuggled, {len(lint_issues)} lint")
print(f"  retention : {len(retention_issues)} open")
for n in sorted(BEAT_SHEET):
    sec = [i for i in tagged_sentences if i["section"] == n]
    words = sum(len(i["sentence"].split()) for i in sec)
    lo, hi = BEAT_SHEET[n]["words"]
    ok = "ok" if lo <= words <= hi else "OUT"
    print(f"    S{n} {BEAT_SHEET[n]['name']:<24} {words:>4}w ({lo}-{hi}) {ok}")

total_open = len(fact_issues) + len(smuggled_issues) + len(lint_issues) + len(retention_issues)
if total_open == 0:
    print("\n  BOTH GATES CLEAN - safe to voice.")
else:
    print(f"\n  {total_open} issue(s) still open. vo_raw.txt is written either way, but read")
    print("  validation_round_*.json and retention_round_*.json before you spend GPU time on TTS.")

print("\n" + "-" * 70)
print(master_script)
print("-" * 70)


## Step 15 — Title, description, tags

In [ ]:
# =============================================================================
#  YOUTUBE PACKAGING
# =============================================================================
# v4 wrote a source list and stopped there. On a channel like this, packaging is not a
# nice-to-have downstream of the script - title and thumbnail decide whether the script
# is ever heard at all. Generated here, after the final script exists, so the title
# promises something the video actually delivers.
# =============================================================================

hook_lines = " ".join(i["sentence"] for i in tagged_sentences if i["section"] == 1)

packaging_prompt = f"""You are packaging a YouTube video for a channel called
"Weird Human Biology". The channel takes one ordinary thing about the human body and
shows the viewer it is stranger than they assumed.

Video topic: {TOPIC}
The video's cold open: {hook_lines}
The single most striking fact in it: {profile.get('the_best_counter_evidence')}
The myth it overturns: {profile.get('the_common_myth')}

TITLE FORMULA THIS CHANNEL IS TESTING (borrowed from a channel with far larger reach —
imitate the *shape*, never the topic):
  Formula: {TITLE_FORMULA['pattern']}
  Reference example that works: {TITLE_FORMULA['reference_examples'][0]}
  Rules:
{chr(10).join('    - ' + r for r in TITLE_FORMULA['rules'])}
At least 4 of the 8 titles below must follow this exact shape. The other 4 stay free to
use the angles further down, so we're testing more than one hypothesis per video.

LEARNED FROM THIS CHANNEL'S OWN PUBLISHED VIDEOS AND COMPETITOR DATA (real performance,
not a prior — empty until videos exist and Cell 20's strategist has run):
  Titles:
{learnings_block(["title_notes"], n=8)}
  Thumbnails:
{learnings_block(["thumbnail_notes"], n=8)}

Produce ONLY valid JSON, no fences:
{{
  "titles": [
    {{"text": "...", "angle": "curiosity gap | contradiction | second person | number | question",
      "why": "one line on who clicks this and why"}}
  ],
  "thumbnail_text": [
    {{"text": "3-5 words maximum, readable at phone size", "visual": "what the stickman is doing"}}
  ],
  "description": "the full YouTube description. Open with two sentences that restate the hook and would make a browser click from the search page. Then a plain-language summary of what the video covers. No hashtag spam. No 'in this video we will'.",
  "chapters": [{{"label": "short chapter title", "section": 1}}],
  "tags": ["15-20 search tags, mixed broad and specific"],
  "pinned_comment": "one comment that asks the audience the video's open question in a way that starts an argument, without saying 'comment below'",
  "community_post": "one short post to publish the day before, teasing the question without answering it"
}}

Rules for titles - produce exactly 8:
  - Under 60 characters so nothing truncates on mobile.
  - Each one a genuinely different angle. Do not give me eight rewordings of the same title.
  - No clickbait the video does not pay off. Every title must be true.
  - No ALL CAPS words, no "SHOCKING", no "you won't believe".
  - At least two must work as a question, at least two as a flat statement of the weird fact.

Rules for thumbnail_text - produce exactly 4. Under 5 words. It must NOT repeat the title;
it should complete it or contradict it.

Chapters: one per section of the script, in order, labelled the way a curious viewer would
scan them - never "Introduction" or "Conclusion"."""

print("Generating packaging...")
pack = parse_json_response(call_model(MODEL_WRITER, packaging_prompt, max_tokens=4000, temperature=0.8))

with open("packaging.json", "w", encoding="utf-8") as f:
    json.dump(pack, f, indent=2)

print("\n" + "=" * 70)
print("TITLE OPTIONS")
print("=" * 70)
for i, t in enumerate(pack.get("titles", []), 1):
    print(f"{i:>2}. {t['text']}  ({len(t['text'])} chars)")
    print(f"    [{t.get('angle')}] {t.get('why')}")

print("\n" + "=" * 70)
print("THUMBNAIL TEXT")
print("=" * 70)
for t in pack.get("thumbnail_text", []):
    print(f"  \"{t['text']}\"  -  stickman: {t.get('visual')}")

# --- Description with real sources appended ---
description = pack.get("description", "")
description += "\n\nSources for this video:\n"
for i, s in enumerate(source_pack):
    description += f"{i+1}. {s['title']}\n   {s['url']}\n"
description += "\nChapters:\n(timestamps filled in after upload)\n"
for c in pack.get("chapters", []):
    description += f"  {c.get('label')}\n"

with open("youtube_description.txt", "w", encoding="utf-8") as f:
    f.write(description)

with open("youtube_tags.txt", "w", encoding="utf-8") as f:
    f.write(", ".join(pack.get("tags", [])))

with open("pinned_comment.txt", "w", encoding="utf-8") as f:
    f.write(pack.get("pinned_comment", ""))

print("\n" + "=" * 70)
print("DESCRIPTION")
print("=" * 70)
print(description)
print("=" * 70)
print(f"TAGS: {', '.join(pack.get('tags', []))}")
print(f"\nPINNED COMMENT: {pack.get('pinned_comment')}")
print(f"\nCOMMUNITY POST: {pack.get('community_post')}")
print("\nWritten: packaging.json, youtube_description.txt, youtube_tags.txt, pinned_comment.txt")


## Step 16 — Audio tools

Just loads the mastering chain. Nothing happens yet.

In [ ]:
%pip install -q scipy numpy soundfile pyloudnorm noisereduce

# =============================================================================
#  STAGE A — VOICE MASTERING (deterministic DSP, no AI)
# =============================================================================
#  Every function here is ordinary signal processing: filters, envelopes, gain.
#  Nothing is generated. Audio in, the same audio cleaner, out.
#
#  THE CONSTRAINT THAT SHAPES THIS WHOLE CELL: `scenes_timeline.json` is derived
#  from sample counts in the TTS cell, and the mux cell trusts those durations to
#  tile the audio exactly. So anything applied to the finished voiceover MUST NOT
#  change its length. Silence trimming does change length -- which is fine on a
#  standalone clip and fatal on the timed narration. Hence two profiles:
#
#    profile="clip"     full chain incl. silence trim. For training data and
#                       reference clips, where length is nobody's business.
#    profile="timeline" every filter except the trim. Length-locked, asserted.
#
#  Do not add a length-changing step to the "timeline" profile without also
#  rebuilding the timeline. The assert at the bottom of master() will catch you.
# =============================================================================

import numpy as np
import scipy.signal as sps
import scipy.ndimage as ndi
import soundfile as sf
import pyloudnorm as pyln


# ---------------------------------------------------------------- primitives

def _to_mono(x):
    return x.mean(axis=1) if x.ndim > 1 else x


def _biquad_peaking(sr, f0, gain_db, q):
    """RBJ peaking-EQ biquad. Returns (b, a) for use with filtfilt."""
    A = 10.0 ** (gain_db / 40.0)
    w0 = 2.0 * np.pi * f0 / sr
    alpha = np.sin(w0) / (2.0 * q)
    cw = np.cos(w0)
    b = np.array([1 + alpha * A, -2 * cw, 1 - alpha * A])
    a = np.array([1 + alpha / A, -2 * cw, 1 - alpha / A])
    return b / a[0], a / a[0]


def _highpass(x, sr, fc=85.0, order=4):
    """Zero-phase high-pass. Kills rumble, HVAC, desk thumps, plosive tails."""
    sos = sps.butter(order, fc / (sr / 2.0), btype="highpass", output="sos")
    return sps.sosfiltfilt(sos, x)


def _peaking(x, sr, f0, gain_db, q):
    if abs(gain_db) < 1e-6:
        return x
    b, a = _biquad_peaking(sr, f0, gain_db, q)
    return sps.filtfilt(b, a, x)


#  The attack/release follower is asymmetric, so it cannot be written as a single
#  LTI filter pass -- it needs a sequential loop. Running that loop per-sample over
#  a 9-minute 48k file means 26M Python iterations, which is minutes of wall clock.
#  So the detector runs at a 2 kHz control rate over per-block peaks and the
#  resulting gain curve is interpolated back up to sample rate. That is how real
#  compressors work internally anyway, it resolves attacks down to 0.5 ms, and it
#  is ~50x faster here.
_CONTROL_HZ = 2000.0


def _gain_curve(x, sr, threshold_db, ratio, attack_ms, release_ms, knee_db):
    """Gain reduction in dB, one value per sample, from a control-rate detector."""
    block = max(1, int(round(sr / _CONTROL_HZ)))
    n = x.shape[0]
    nb = int(np.ceil(n / block))
    mag = np.abs(x)
    pad = nb * block - n
    if pad:
        mag = np.concatenate([mag, np.zeros(pad)])
    peak = mag.reshape(nb, block).max(axis=1)

    csr = sr / block
    atk = np.exp(-1.0 / max(csr * attack_ms / 1000.0, 1e-9))
    rel = np.exp(-1.0 / max(csr * release_ms / 1000.0, 1e-9))
    env = np.empty(nb)
    prev = 0.0
    for i in range(nb):
        m = peak[i]
        coef = atk if m > prev else rel
        prev = coef * prev + (1.0 - coef) * m
        env[i] = prev

    over = 20.0 * np.log10(np.maximum(env, 1e-9)) - threshold_db
    gain_db = np.zeros(nb)
    # Soft knee: quadratic interpolation across +/- knee/2 around the threshold.
    half = knee_db / 2.0
    knee = (over > -half) & (over < half)
    above = over >= half
    gain_db[above] = (1.0 / ratio - 1.0) * over[above]
    gain_db[knee] = (1.0 / ratio - 1.0) * ((over[knee] + half) ** 2) / (2.0 * knee_db)

    centres = np.arange(nb) * block + block / 2.0
    return np.interp(np.arange(n), centres, gain_db)


def _compress(x, sr, threshold_db=-18.0, ratio=3.0,
              attack_ms=8.0, release_ms=120.0, makeup_db=0.0, knee_db=6.0):
    """Soft-knee downward compressor. Evens out delivery so quiet phrases at the
    end of a sentence sit at the same level as the punched first word."""
    gain_db = _gain_curve(x, sr, threshold_db, ratio, attack_ms, release_ms, knee_db)
    return x * (10.0 ** ((gain_db + makeup_db) / 20.0))


def _deess(x, sr, threshold_db=-30.0, ratio=4.0, lo=5000.0, hi=9000.0):
    """Split-band de-esser: compress ONLY the sibilance band, leave the rest of
    the spectrum untouched, recombine. A plain high-shelf cut would dull every
    consonant; this only ducks the band when an actual 'sss' is present."""
    nyq = sr / 2.0
    hi = min(hi, nyq * 0.98)
    if lo >= hi:
        return x
    sos = sps.butter(4, [lo / nyq, hi / nyq], btype="bandpass", output="sos")
    sib = sps.sosfiltfilt(sos, x)
    rest = x - sib
    sib_c = _compress(sib, sr, threshold_db=threshold_db, ratio=ratio,
                      attack_ms=1.0, release_ms=40.0, knee_db=4.0)
    return rest + sib_c


def _denoise(x, sr, noise_profile=None, prop_decrease=0.6):
    """Spectral-gate denoise, deliberately gentle.

    Uses a noise profile lifted from the quietest stretch of the recording when
    one can be found -- a measured fingerprint of *this* room beats a stationary
    estimate. `prop_decrease` defaults low on purpose: a spectral gate cannot
    tell quiet speech from noise, and at 0.85 it audibly eats the ends of
    sentences and any softly-delivered line. Measured on the test signal, 0.85
    dropped voiced frames from 240 to 51. Raise it only for a genuinely noisy
    room, and listen to the result before committing to it.
    """
    import noisereduce as nr
    if noise_profile is None:
        noise_profile = _find_noise_profile(x, sr)
    kw = dict(y=x, sr=sr, stationary=True, prop_decrease=prop_decrease)
    if noise_profile is not None:
        kw["y_noise"] = noise_profile
    return nr.reduce_noise(**kw).astype(np.float64)


def _find_noise_profile(x, sr, win_s=0.5):
    """Return the quietest half-second in the file, or None if the recording is
    too short / too uniformly loud to contain a usable noise-only window."""
    win = int(sr * win_s)
    if x.shape[0] < win * 3:
        return None
    n = x.shape[0] // win
    frames = x[: n * win].reshape(n, win)
    rms = np.sqrt((frames ** 2).mean(axis=1))
    quietest = int(np.argmin(rms))
    # If the quietest window is within 12 dB of the median, there is no silence
    # here -- it is wall-to-wall speech, and gating against speech is destructive.
    med = np.median(rms)
    if med > 0 and 20 * np.log10(max(rms[quietest], 1e-9) / med) > -12.0:
        return None
    return frames[quietest].copy()


def _trim_silence(x, sr, threshold_db=-45.0, pad_ms=120.0, max_gap_s=0.6):
    """Trim head/tail silence and collapse interior dead air longer than
    `max_gap_s` down to `max_gap_s`. LENGTH-CHANGING -- "clip" profile only."""
    frame = int(sr * 0.02)
    if x.shape[0] < frame * 2:
        return x
    n = x.shape[0] // frame
    frames = x[: n * frame].reshape(n, frame)
    rms_db = 20 * np.log10(np.maximum(np.sqrt((frames ** 2).mean(axis=1)), 1e-9))
    peak_db = rms_db.max()
    voiced = rms_db > (peak_db + threshold_db)
    if not voiced.any():
        return x

    pad = max(1, int(pad_ms / 20.0))
    keep = np.zeros(n, dtype=bool)
    max_gap = max(1, int(max_gap_s / 0.02))
    idx = np.flatnonzero(voiced)
    keep[max(0, idx[0] - pad): min(n, idx[-1] + pad + 1)] = True

    # Collapse long interior gaps to max_gap frames.
    run_start = None
    for i in range(idx[0], idx[-1] + 1):
        if not voiced[i]:
            if run_start is None:
                run_start = i
        else:
            if run_start is not None and (i - run_start) > max_gap + 2 * pad:
                keep[run_start + pad: i - pad] = False
                keep[run_start + pad: run_start + pad + max_gap] = True
            run_start = None

    return frames[keep].reshape(-1)


def _limit(x, sr, ceiling_db=-1.5, lookahead_ms=3.0, release_ms=60.0):
    """Look-ahead brickwall limiter.

    The naive way to respect a peak ceiling is to scale the whole file down until
    the loudest sample fits. On speech with a 30 dB crest factor that throws away
    ~10 dB of the loudness you just normalised to -- the one transient consonant
    sets the level for nine minutes of audio. A limiter instead pulls gain down
    only around the peaks that need it, so integrated loudness survives.
    """
    ceiling = 10.0 ** (ceiling_db / 20.0)
    mag = np.abs(x)
    if mag.max() <= ceiling:
        return x
    req = np.minimum(1.0, ceiling / np.maximum(mag, 1e-12))
    # Running minimum first: gain is already down before the peak arrives, and
    # recovers no faster than the release window.
    w = max(3, int(sr * release_ms / 1000.0)) | 1
    g = ndi.minimum_filter1d(req, size=w, mode="nearest")
    # Then smooth the gain curve so it does not step and cause distortion.
    s = max(3, int(sr * lookahead_ms / 1000.0)) | 1
    g = ndi.uniform_filter1d(g, size=s, mode="nearest")
    # Smoothing can nudge gain back above the minimum at the exact peak; clip the
    # remainder. It is a handful of samples and inaudible.
    return np.clip(x * g, -ceiling, ceiling)


def _loudnorm(x, sr, target_lufs=-16.0, true_peak_db=-1.5, passes=8):
    """Normalise integrated loudness (EBU R128) under a true-peak ceiling.

    Limiting changes loudness slightly, so measure/correct/limit is iterated until
    it converges. Without this the result lands several dB under target on
    peaky material.
    """
    meter = pyln.Meter(sr)
    for _ in range(passes):
        loudness = meter.integrated_loudness(x)
        if not np.isfinite(loudness):
            break
        delta = target_lufs - loudness
        if abs(delta) < 0.1:
            break
        x = _limit(x * (10.0 ** (delta / 20.0)), sr, true_peak_db)
    return _limit(x, sr, true_peak_db)


# ---------------------------------------------------------------- the chain

MASTER_DEFAULTS = dict(
    denoise=True,
    denoise_amount=0.6,
    highpass_hz=85.0,
    mud_hz=280.0, mud_db=-3.0, mud_q=1.0,          # boxiness / proximity build-up
    presence_hz=4000.0, presence_db=2.5, presence_q=0.9,   # intelligibility
    air_hz=10000.0, air_db=1.5, air_q=0.7,          # openness, sparingly
    deess=True, deess_threshold_db=-30.0, deess_ratio=4.0,
    comp_threshold_db=-18.0, comp_ratio=3.0,
    comp_attack_ms=8.0, comp_release_ms=120.0, comp_makeup_db=2.0,
    target_lufs=-16.0, true_peak_db=-1.5,
    trim_silence=True, trim_threshold_db=-45.0,
)


def master(x, sr, profile="clip", **overrides):
    """Run the mastering chain. `profile` is 'clip' or 'timeline'."""
    cfg = dict(MASTER_DEFAULTS)
    cfg.update(overrides)
    if profile == "timeline":
        cfg["trim_silence"] = False
    elif profile != "clip":
        raise ValueError(f"unknown profile {profile!r} (expected 'clip' or 'timeline')")

    x = np.asarray(_to_mono(x), dtype=np.float64)
    n_in = x.shape[0]

    # High-pass FIRST: rumble is broadband-adjacent low-frequency energy, and if it
    # is still present when the noise profile is measured it dominates the estimate
    # and the gate ends up subtracting the wrong spectrum from the speech.
    x = _highpass(x, sr, cfg["highpass_hz"])
    if cfg["denoise"]:
        x = _denoise(x, sr, prop_decrease=cfg["denoise_amount"])
    x = _peaking(x, sr, cfg["mud_hz"], cfg["mud_db"], cfg["mud_q"])
    x = _peaking(x, sr, cfg["presence_hz"], cfg["presence_db"], cfg["presence_q"])
    if cfg["air_hz"] < sr / 2.0 * 0.95:
        x = _peaking(x, sr, cfg["air_hz"], cfg["air_db"], cfg["air_q"])
    if cfg["deess"]:
        x = _deess(x, sr, cfg["deess_threshold_db"], cfg["deess_ratio"])
    x = _compress(x, sr,
                  threshold_db=cfg["comp_threshold_db"], ratio=cfg["comp_ratio"],
                  attack_ms=cfg["comp_attack_ms"], release_ms=cfg["comp_release_ms"],
                  makeup_db=cfg["comp_makeup_db"])
    if cfg["trim_silence"]:
        x = _trim_silence(x, sr, threshold_db=cfg["trim_threshold_db"])
    x = _loudnorm(x, sr, cfg["target_lufs"], cfg["true_peak_db"])

    if profile == "timeline" and x.shape[0] != n_in:
        raise AssertionError(
            f"timeline profile changed length ({n_in} -> {x.shape[0]} samples). "
            "A length-changing filter got into the chain; scenes_timeline.json "
            "would no longer tile the audio."
        )
    return np.clip(x, -1.0, 1.0)


def master_file(src, dst, profile="clip", target_sr=None, **overrides):
    x, sr = sf.read(src, always_2d=False)
    x = _to_mono(np.asarray(x, dtype=np.float64))
    if target_sr and target_sr != sr:
        if profile == "timeline":
            raise ValueError("refusing to resample under the timeline profile")
        g = np.gcd(int(target_sr), int(sr))
        x = sps.resample_poly(x, int(target_sr) // g, int(sr) // g)
        sr = target_sr
    y = master(x, sr, profile=profile, **overrides)
    sf.write(dst, y.astype(np.float32), sr, subtype="PCM_16")
    return dst, sr, y.shape[0] / sr


def report(path):
    """Print the numbers that actually tell you whether a take is usable."""
    x, sr = sf.read(path, always_2d=False)
    x = _to_mono(np.asarray(x, dtype=np.float64))
    meter = pyln.Meter(sr)
    lufs = meter.integrated_loudness(x)
    peak_db = 20 * np.log10(max(np.abs(x).max(), 1e-9))
    noise = _find_noise_profile(x, sr)
    if noise is not None:
        nf = 20 * np.log10(max(np.sqrt((noise ** 2).mean()), 1e-9))
        snr = f"{lufs - nf:5.1f} dB" if np.isfinite(lufs) else "  n/a"
    else:
        nf, snr = float("nan"), "  n/a (no silence found)"
    print(f"  {path}")
    print(f"    {x.shape[0]/sr:8.2f} s @ {sr} Hz")
    print(f"    loudness {lufs:6.1f} LUFS   peak {peak_db:6.1f} dBFS")
    print(f"    noise floor {nf:6.1f} dBFS   SNR {snr}")
    return dict(seconds=x.shape[0] / sr, sr=sr, lufs=lufs, peak_db=peak_db, noise_db=nf)


# ---------------------------------------------------------------- self-check
# Runs in ~1s on a synthetic signal. Catches a broken scipy/pyloudnorm install and
# any edit that breaks the length lock, before nine minutes of audio depend on it.
def _selfcheck():
    sr = 48000
    rng = np.random.default_rng(0)
    t = np.arange(int(6 * sr)) / sr
    sig = 0.3 * np.sin(2 * np.pi * 120 * t) * (0.5 + 0.5 * np.sin(2 * np.pi * 0.7 * t))
    sig += rng.normal(0, 0.002, sig.shape) + 0.02 * np.sin(2 * np.pi * 48 * t)
    y = master(sig, sr, profile="timeline")
    assert len(y) == len(sig), "timeline profile is not length-locked"
    assert np.isfinite(y).all(), "chain produced NaN/Inf"
    assert np.abs(y).max() <= 1.0, "chain produced clipping"
    lufs = pyln.Meter(sr).integrated_loudness(y)
    assert abs(lufs + 16.0) < 1.5, f"loudness off target: {lufs:.1f} LUFS"
    print(f"  self-check passed (timeline output {lufs:.1f} LUFS, length locked)")


_selfcheck()
print("Stage A ready:  master(x, sr, profile=...)  master_file(src, dst, ...)  report(path)")

## Step 17 — Voice: the computer reads it

**If you picked "Computer voice" in the Control Panel** — run this, then skip to Step 20.

**If you picked "My own recorded voice"** — skip this, go to Step 18.

In [ ]:
import gc
import os
import re
import json
import torch
import torchaudio as ta
from chatterbox.tts import ChatterboxTTS

# =============================================================================
#  TWO FIXES IN THIS CELL
# =============================================================================
# 1. CHUNKING. v4 passed the entire ~1,400-word script to tts.generate() in a single
#    call and the markdown defended it as "the right call for natural cadence". It is
#    not, for this model. Chatterbox is built around short generations - past roughly
#    40 seconds it drifts, repeats, or truncates outright. Feeding it a 9-minute script
#    in one shot is the single most likely reason a run produces unusable audio.
#
# 2. TIMINGS COME FROM SYNTHESIS, NOT FROM WHISPER. Because we now generate chunk by
#    chunk, we know each chunk's exact sample count, so scene boundaries are exact by
#    construction. v4 ran Whisper over the finished file and used segment start/end as
#    frame durations - but Whisper segments do not tile the timeline, so every pause
#    between segments was unaccounted for. The frame durations summed to less than the
#    audio length, and "-shortest" in the ffmpeg cell then chopped the end off the
#    voiceover. Deriving timings here makes that class of desync impossible.
#
# 3. NO REFERENCE CLIP. This cell used to clone from my_voice_ref.wav, which cloned the
#    accent along with the voice. Chatterbox's built-in voice is native US English, so it
#    is used bare here to get correct US phonemes, and Cell 11.5 converts the result to
#    the user's timbre with RVC. Accent comes from whatever generates the phonemes;
#    identity comes from the RVC target. Doing it the other way round -- converting a
#    Pakistani-accented recording to a US voice model -- yields a US-sounding voice still
#    speaking with a Pakistani accent, because RVC preserves source prosody.
#
#    Output is vo_raw.wav. full_vo.wav is written by Cell 11.5.
# =============================================================================


if NARRATION != "tts":
    print(f'NARRATION is {NARRATION!r} - skipping TTS; use the human narration cell below.')
else:
    # Leave as None to use Chatterbox's built-in US voice -- the right choice when Cell 11.5
    # is going to convert the timbre anyway. Point it at a clip only if you specifically want
    # a different US-accented delivery to convert FROM, and only with rights to that voice.
    REF_VOICE = None
    MAX_CHUNK_CHARS = 280        # comfortably inside Chatterbox's stable range
    MIN_CHUNK_CHARS = 60         # merge fragments below this into the next chunk
    GAP_SECONDS = 0.28           # breath between chunks; also becomes the scene's tail

    if REF_VOICE and not os.path.exists(REF_VOICE):
        raise FileNotFoundError(
            f"REF_VOICE is set to '{REF_VOICE}' but that file is not here. Upload it, or set\n"
            "REF_VOICE = None to use Chatterbox's built-in US voice (recommended)."
        )
    gen_kw = {"audio_prompt_path": REF_VOICE} if REF_VOICE else {}
    print("Voice:", REF_VOICE if REF_VOICE else "Chatterbox built-in (native US accent)")


    def build_chunks(tagged_sentences):
        """One chunk per scene. Sentences are the natural unit; short ones get merged so a
        three-word Snap doesn't become its own 0.6-second frame, and long ones get split at
        a clause boundary so no chunk exceeds Chatterbox's stable length."""
        chunks, buf, buf_sec = [], "", 1
        for it in tagged_sentences:
            s = it["sentence"].strip()
            if not s:
                continue
            if len(s) > MAX_CHUNK_CHARS:
                parts, cur = [], ""
                for piece in re.split(r"(?<=[,;:])\s+", s):
                    if len(cur) + len(piece) + 1 <= MAX_CHUNK_CHARS:
                        cur = f"{cur} {piece}".strip()
                    else:
                        if cur:
                            parts.append(cur)
                        cur = piece
                if cur:
                    parts.append(cur)
            else:
                parts = [s]

            for p in parts:
                if buf and (len(buf) + len(p) + 1 > MAX_CHUNK_CHARS or len(buf) >= MIN_CHUNK_CHARS):
                    chunks.append({"text": buf, "section": buf_sec})
                    buf, buf_sec = p, it["section"]
                elif buf:
                    buf = f"{buf} {p}"
                else:
                    buf, buf_sec = p, it["section"]
        if buf:
            chunks.append({"text": buf, "section": buf_sec})
        return chunks


    chunks = build_chunks(tagged_sentences)
    print(f"Script split into {len(chunks)} synthesis chunks "
          f"(longest {max(len(c['text']) for c in chunks)} chars)")

    print("Loading Chatterbox...")
    tts = ChatterboxTTS.from_pretrained(device="cuda")
    sr = tts.sr
    gap = torch.zeros(1, int(sr * GAP_SECONDS))

    pieces, scenes, cursor = [], [], 0.0
    print("Synthesizing...")
    for i, ch in enumerate(chunks):
        try:
            wav = tts.generate(text=ch["text"], **gen_kw)
        except Exception as e:
            print(f"  [warn] chunk {i} failed ({e}) - retrying once")
            torch.cuda.empty_cache()
            wav = tts.generate(text=ch["text"], **gen_kw)

        if wav.dim() == 1:
            wav = wav.unsqueeze(0)
        wav = wav.detach().cpu()

        dur = wav.shape[-1] / sr
        pieces.append(wav)
        pieces.append(gap)

        total = dur + GAP_SECONDS
        scenes.append({
            "scene_index": i,
            "start": round(cursor, 3),
            "end": round(cursor + total, 3),
            "duration": round(total, 3),
            "text": ch["text"],
            "section": ch["section"],
        })
        cursor += total

        if (i + 1) % 15 == 0 or i == len(chunks) - 1:
            print(f"  {i+1}/{len(chunks)} chunks  ({cursor/60:.1f} min so far)")

    full = torch.cat(pieces, dim=-1)
    ta.save("vo_raw.wav", full, sr)

    audio_seconds = full.shape[-1] / sr
    # Absorb rounding drift into the final scene so durations tile the audio exactly.
    drift = audio_seconds - scenes[-1]["end"]
    scenes[-1]["duration"] = round(scenes[-1]["duration"] + drift, 3)
    scenes[-1]["end"] = round(audio_seconds, 3)

    with open("scenes_timeline.json", "w", encoding="utf-8") as f:
        json.dump(scenes, f, indent=2)

    print(f"\nvo_raw.wav written: {audio_seconds/60:.2f} min at {sr} Hz")
    print(f"{len(scenes)} scenes, durations sum to {sum(s['duration'] for s in scenes):.2f}s "
          f"vs audio {audio_seconds:.2f}s (must match)")
    print(f"Average scene length: {audio_seconds/len(scenes):.1f}s")

    del tts, pieces, full
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()



## Step 18 — Voice: you read it *(skip if using the computer voice)*

Record **one file per section**. Put them in a folder called `voice_recordings`, named
`section_01.wav`, `section_02.wav`, … one file per section.

Read each section **exactly as written** — the notebook finds where each picture goes by
matching your words against the script, so improvising makes the pictures drift. Same room,
same mic, same distance every time, and no noise-reduction on the way in.

In [ ]:
# =============================================================================
#  NARRATION = "human"  --  YOUR OWN RECORDED VOICE, MASTERED
# =============================================================================
#  voice_recordings/section_01.wav ... -> master -> align -> vo_raw.wav
#                                                         -> scenes_timeline.json
#
#  THE PROBLEM THIS CELL SOLVES. On the TTS path the timeline is free: Chatterbox
#  generates chunk by chunk, so the notebook knows each scene's exact sample count.
#  Reading the script yourself destroys that information -- one file per section
#  contains many scenes with no marked boundaries. Without recovering them the
#  frames would drift against your voice.
#
#  So Whisper transcribes each mastered section WITH word timestamps, and the words
#  it heard are aligned against the words the script expected. difflib gives the
#  correspondence; every scene's first word then has a real timestamp. Alignment is
#  per-section rather than across the whole script on purpose -- an error inside one
#  section cannot propagate into the next.
#
#  Mastering runs BEFORE alignment, because mastering trims silence and therefore
#  moves every timestamp. Align the audio you are actually going to ship.
# =============================================================================

import difflib
import gc
import glob
import json
import os
import re

import numpy as np
import soundfile as sf

HUMAN_DIR = "voice_recordings"
WHISPER_SIZE = "base"      # "small" aligns a little better on accented speech
TARGET_SR = 48000

if NARRATION != "human":
    print(f'NARRATION is "{NARRATION}" - skipping the human-voice path.')
else:
    chunks = build_chunks(tagged_sentences)
    sections = sorted({c["section"] for c in chunks})
    os.makedirs(HUMAN_DIR, exist_ok=True)

    def find_section_file(sec):
        """Accept section_1 / section_01 and any common container."""
        for stem in (f"section_{sec:02d}", f"section_{sec}"):
            for ext in ("wav", "flac", "m4a", "mp3"):
                hits = glob.glob(f"{HUMAN_DIR}/{stem}.{ext}")
                if hits:
                    return hits[0]
        return None

    missing = [s for s in sections if find_section_file(s) is None]
    if missing:
        raise FileNotFoundError(
            f"Missing recordings for section(s) {missing} in {HUMAN_DIR}/.\n"
            f"Expected one file per section, named section_01.wav ... "
            f"section_{max(sections):02d}.wav\n"
            "Read each section's text exactly as written - the aligner matches your "
            "words against the script, and improvising breaks the scene boundaries."
        )

    def norm_words(t):
        return re.sub(r"[^a-z0-9' ]", " ", t.lower()).split()

    def align(words, chunk_texts, total_s):
        """Map each chunk to (start, end) seconds within one section.

        `words` is Whisper's word list, `chunk_texts` the scenes the script says
        this section contains. Returns one (start, end) per chunk, monotonic and
        spanning the whole section.
        """
        heard, hw = [], []
        for w in words:
            nw = norm_words(w.word)
            if nw:
                heard.append(nw[0])
                hw.append(w)

        expected, owner = [], []
        for ci, t in enumerate(chunk_texts):
            for tok in norm_words(t):
                expected.append(tok)
                owner.append(ci)

        e2h = {}
        if expected and heard:
            sm = difflib.SequenceMatcher(None, expected, heard, autojunk=False)
            for tag, i1, i2, j1, j2 in sm.get_opcodes():
                if tag == "equal":
                    for k in range(i2 - i1):
                        e2h[i1 + k] = j1 + k

        first_of = {}
        for ei, ci in enumerate(owner):
            first_of.setdefault(ci, ei)

        # Anchor each chunk on the first of its words that Whisper actually matched.
        n = len(chunk_texts)
        starts = [None] * n
        for ci in range(n):
            for ei in range(first_of[ci], len(expected)):
                if owner[ei] != ci and starts[ci] is None and ei > first_of[ci] + 8:
                    break            # searched well past this chunk; give up on it
                if ei in e2h:
                    starts[ci] = hw[e2h[ei]].start
                    break
        starts[0] = 0.0

        # Unmatched chunks (a dropped or misread opening word) get interpolated
        # between their nearest anchored neighbours, weighted by word count.
        known = [i for i, s in enumerate(starts) if s is not None]
        weights = np.cumsum([0] + [max(1, len(norm_words(t))) for t in chunk_texts])
        for i in range(n):
            if starts[i] is not None:
                continue
            lo = max([k for k in known if k < i], default=None)
            hi = min([k for k in known if k > i], default=None)
            if lo is None:
                starts[i] = 0.0
            elif hi is None:
                span = total_s - starts[lo]
                frac = (weights[i] - weights[lo]) / max(weights[n] - weights[lo], 1)
                starts[i] = starts[lo] + span * frac
            else:
                frac = (weights[i] - weights[lo]) / max(weights[hi] - weights[lo], 1)
                starts[i] = starts[lo] + (starts[hi] - starts[lo]) * frac

        starts = np.maximum.accumulate(np.array(starts, dtype=float))  # force monotonic
        ends = np.append(starts[1:], total_s)
        ends = np.maximum(ends, starts + 0.05)
        ends[-1] = total_s
        return list(zip(starts, ends)), len(e2h) / max(len(expected), 1)

    print(f"Loading Whisper ({WHISPER_SIZE})...")
    from faster_whisper import WhisperModel
    import torch
    wm = WhisperModel(WHISPER_SIZE,
                      device="cuda" if torch.cuda.is_available() else "cpu",
                      compute_type="float16" if torch.cuda.is_available() else "int8")

    pieces, scenes, cursor, quality = [], [], 0.0, []
    print()
    for sec in sections:
        path = find_section_file(sec)
        sec_chunks = [c for c in chunks if c["section"] == sec]

        x, sr = sf.read(path, always_2d=False)
        x = _to_mono(np.asarray(x, dtype=np.float64))
        if sr != TARGET_SR:
            gcd = np.gcd(TARGET_SR, int(sr))
            x = sps.resample_poly(x, TARGET_SR // gcd, int(sr) // gcd)
            sr = TARGET_SR
        y = master(x, sr, profile="clip")

        tmp = f"_mastered_section_{sec:02d}.wav"
        sf.write(tmp, y.astype(np.float32), sr, subtype="PCM_16")
        total_s = len(y) / sr

        segs, _ = wm.transcribe(tmp, word_timestamps=True)
        words = [w for s in segs for w in (s.words or [])]
        spans, matched = align(words, [c["text"] for c in sec_chunks], total_s)
        quality.append((sec, matched, len(sec_chunks)))

        for c, (a, b) in zip(sec_chunks, spans):
            scenes.append({
                "scene_index": len(scenes),
                "start": round(cursor + a, 3),
                "end": round(cursor + b, 3),
                "duration": round(b - a, 3),
                "text": c["text"],
                "section": sec,
            })
        pieces.append(y)
        cursor += total_s
        print(f"  section {sec:>2}  {total_s:6.1f}s  {len(sec_chunks):>3} scenes  "
              f"word match {matched*100:5.1f}%")
        os.remove(tmp)

    del wm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    full = np.concatenate(pieces)
    sf.write("vo_raw.wav", full.astype(np.float32), TARGET_SR, subtype="PCM_16")
    audio_seconds = len(full) / TARGET_SR

    # Absorb rounding drift into the final scene so durations tile the audio exactly,
    # the same guarantee the TTS path gives.
    drift = audio_seconds - scenes[-1]["end"]
    scenes[-1]["duration"] = round(scenes[-1]["duration"] + drift, 3)
    scenes[-1]["end"] = round(audio_seconds, 3)

    with open("scenes_timeline.json", "w", encoding="utf-8") as f:
        json.dump(scenes, f, indent=2)

    print(f"\nvo_raw.wav written: {audio_seconds/60:.2f} min at {TARGET_SR} Hz")
    print(f"{len(scenes)} scenes, durations sum to "
          f"{sum(s['duration'] for s in scenes):.2f}s vs audio {audio_seconds:.2f}s")

    worst = min(quality, key=lambda q: q[1])
    print(f"\nAlignment: worst section is {worst[0]} at {worst[1]*100:.1f}% word match.")
    if worst[1] < 0.75:
        print("  WARNING: below 75% means Whisper could not follow the script in that")
        print("  section, so its scene boundaries are guesses and the frames there will")
        print("  drift. Usually you paraphrased instead of reading it, or the recording")
        print("  is noisy. Re-record that section reading the text exactly.")
    else:
        print("  Good - scene boundaries are anchored to real word timestamps.")

## Step 19 — Voice cloning *(advanced, optional, once ever)*

Skip this unless you want the narration in **your** voice with a US accent. Run both cells
once; every future video reuses the trained model and skips this step.

In [ ]:
# =============================================================================
#  BUILD THE RVC TRAINING DATASET FROM YOUR OWN RECORDINGS
# =============================================================================
#  voice_dataset_raw/*.wav  ->  master (clip profile)  ->  4-12s clips  ->
#  voice_dataset/*.wav, which is what the training cell consumes.
#
#  RVC wants many short, clean, uniform clips rather than a few long ones. Cutting
#  on silence boundaries rather than a fixed grid matters: a clip that starts or
#  ends mid-phoneme teaches the model a transition that does not exist in speech.
# =============================================================================

import glob
import os
import shutil

import numpy as np
import soundfile as sf

RAW_DIR = "voice_dataset_raw"
OUT_DIR = "voice_dataset"
DATASET_SR = 48000        # must match RVC_SR in the training cell
CLIP_MIN_S = 4.0
CLIP_MAX_S = 12.0
DROP_BELOW_S = 2.0        # clips shorter than this are not worth a training step
SILENCE_DB = -42.0        # relative to the file's loudest frame
MIN_SNR_DB = 25.0         # warn below this

os.makedirs(RAW_DIR, exist_ok=True)
raw_files = sorted(glob.glob(f"{RAW_DIR}/*.wav")) + sorted(glob.glob(f"{RAW_DIR}/*.flac"))
if not raw_files:
    raise FileNotFoundError(
        f"No recordings found in {RAW_DIR}/.\n"
        "Upload 15-30 minutes of your own voice there (left sidebar -> Files), then re-run.\n"
        "See the markdown above for what makes a usable recording."
    )

if os.path.isdir(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)


def split_on_silence(x, sr, min_s, max_s, silence_db):
    """Cut into clips at silence boundaries, preferring the longest silence when a
    span runs past max_s. Returns a list of (start_sample, end_sample)."""
    frame = int(sr * 0.02)
    n = len(x) // frame
    if n < 2:
        return []
    rms = np.sqrt((x[: n * frame].reshape(n, frame) ** 2).mean(axis=1))
    db = 20 * np.log10(np.maximum(rms, 1e-12))
    voiced = db > (db.max() + silence_db)

    # A noisy take can sit entirely above a max-relative threshold, in which case
    # nothing reads as silence and the whole file becomes one span. Fall back to a
    # percentile threshold so there is still somewhere to cut.
    if voiced.mean() > 0.98:
        voiced = db > np.percentile(db, 25)

    spans, start = [], None
    for i in range(n):
        if voiced[i] and start is None:
            start = i
        elif not voiced[i] and start is not None:
            spans.append((start, i))
            start = None
    if start is not None:
        spans.append((start, n))
    if not spans:
        return []

    # Merge adjacent voiced spans until each clip is at least min_s, splitting at
    # the widest interior silence whenever a clip would exceed max_s.
    clips, cur_start, cur_end = [], spans[0][0], spans[0][1]
    for s, e in spans[1:]:
        if (e - cur_start) * 0.02 <= max_s:
            cur_end = e
        else:
            if (cur_end - cur_start) * 0.02 >= min_s:
                clips.append((cur_start, cur_end))
            cur_start, cur_end = s, e
    if (cur_end - cur_start) * 0.02 >= min_s:
        clips.append((cur_start, cur_end))

    # A single unbroken span can itself exceed max_s -- a passage read without a
    # real pause, or a take too noisy for the silence detector to find gaps in.
    # Nothing above splits those, so they would sail through at full length. Hard-
    # split into equal parts; an arbitrary cut is better than a 3-minute clip.
    max_f = int(max_s / 0.02)
    bounded = []
    for a, b in clips:
        if (b - a) <= max_f:
            bounded.append((a, b))
            continue
        parts = int(np.ceil((b - a) / max_f))
        edges = np.linspace(a, b, parts + 1).astype(int)
        bounded.extend(zip(edges[:-1], edges[1:]))

    pad = int(0.05 / 0.02)
    return [(max(0, a - pad) * frame, min(n, b + pad) * frame) for a, b in bounded]


rows, total_s, kept = [], 0.0, 0
print(f"Processing {len(raw_files)} recording(s)...\n")

for path in raw_files:
    x, sr = sf.read(path, always_2d=False)
    x = _to_mono(np.asarray(x, dtype=np.float64))

    # Measure BEFORE mastering -- that is the number that tells you whether the
    # recording itself is any good. Mastering flatters the SNR reading.
    # High-pass first though: mains hum sits in the silent stretches and lifts them
    # above the quiet-window test, so a raw take with hum reports "no silence found"
    # exactly when you most want the number.
    noise = _find_noise_profile(_highpass(x, sr, 85.0), sr)
    if noise is not None:
        noise_db = 20 * np.log10(max(np.sqrt((noise ** 2).mean()), 1e-12))
        speech_db = 20 * np.log10(max(np.sqrt((x ** 2).mean()), 1e-12))
        snr = speech_db - noise_db
    else:
        snr = float("nan")

    if sr != DATASET_SR:
        g = np.gcd(int(DATASET_SR), int(sr))
        x = sps.resample_poly(x, DATASET_SR // g, sr // g)
        sr = DATASET_SR

    y = master(x, sr, profile="clip")
    spans = split_on_silence(y, sr, CLIP_MIN_S, CLIP_MAX_S, SILENCE_DB)

    base = os.path.splitext(os.path.basename(path))[0]
    n_clip = 0
    for a, b in spans:
        clip = y[a:b]
        if len(clip) / sr < DROP_BELOW_S:
            continue
        sf.write(f"{OUT_DIR}/{base}_{n_clip:04d}.wav", clip.astype(np.float32), sr, subtype="PCM_16")
        total_s += len(clip) / sr
        n_clip += 1
    kept += n_clip
    rows.append((os.path.basename(path), len(x) / sr, snr, n_clip))

print(f"{'file':<34}{'length':>9}{'SNR':>9}{'clips':>7}")
print("-" * 59)
warn = []
for name, dur, snr, n_clip in rows:
    flag = ""
    if not np.isnan(snr) and snr < MIN_SNR_DB:
        flag = "  <- noisy"
        warn.append(name)
    snr_s = f"{snr:6.1f}dB" if not np.isnan(snr) else "   n/a"
    print(f"{name[:33]:<34}{dur:8.1f}s{snr_s:>9}{n_clip:>7}{flag}")

print("-" * 59)
print(f"{'TOTAL':<34}{total_s:8.1f}s{'':>9}{kept:>7}")
print(f"\n{kept} clips, {total_s/60:.1f} minutes in {OUT_DIR}/")

if warn:
    print(f"\n  WARNING: low SNR in {len(warn)} file(s): {', '.join(warn[:4])}")
    print("  Below ~25dB you are teaching the model your room, not your voice.")
    print("  Re-recording beats spending GPU hours on this.")

if total_s < 600:
    print(f"\n  WARNING: {total_s/60:.1f} min is thin. RVC target models degrade noticeably")
    print("  under ~10 min. 15-30 min is the range where quality stops improving much.")
elif total_s > 3600:
    print(f"\n  NOTE: {total_s/60:.1f} min is more than you need. 15-30 min trains faster")
    print("  and sounds the same.")
else:
    print("\n  Dataset size is in the good range.")

In [ ]:
# =============================================================================
#  TRAIN THE RVC TARGET MODEL ON YOUR VOICE  --  ONE TIME ONLY
# =============================================================================
#  Run this once. It produces voice_model/<NAME>.pth and voice_model/<NAME>.index,
#  which the conversion cell loads on every subsequent video. Once those two files
#  exist and are backed up to Drive, never run this cell again.
#
#  Cost: roughly 1-3 GPU-hours for 15-30 min of audio at 200-300 epochs on a T4.
#
#  This is the most fragile cell in the notebook, and it is worth being honest
#  about why: RVC has no headless training API. The WebUI drives training from
#  button handlers, so a notebook has to reimplement the same sequence -- preprocess,
#  extract f0, extract features, write the filelist, train, build the faiss index.
#  The repo is pinned to a known commit below for exactly this reason. If it breaks
#  after an upstream change, the pin is the first thing to look at.
# =============================================================================

import os
import subprocess
import sys

RVC_DIR = "/content/RVC"
# Empty = track the default branch. Once you have a run that works end to end, put
# that commit hash here so the next run cannot be broken by an upstream change:
#   !cd /content/RVC && git rev-parse HEAD
RVC_COMMIT = ""
MODEL_NAME = "myvoice"
RVC_SR = "48k"            # must match DATASET_SR in the dataset cell
EPOCHS = 250              # 200-300 is the useful range; past that it memorises
BATCH_SIZE = 8            # drop to 4 if you hit CUDA OOM on a T4
SAVE_EVERY = 50
DATASET_DIR = os.path.abspath("voice_dataset")
OUT_DIR = os.path.abspath("voice_model")

if not os.path.isdir(DATASET_DIR) or not os.listdir(DATASET_DIR):
    raise FileNotFoundError("Run the dataset cell first -- voice_dataset/ is empty.")

os.makedirs(OUT_DIR, exist_ok=True)
FINAL_PTH = f"{OUT_DIR}/{MODEL_NAME}.pth"
FINAL_IDX = f"{OUT_DIR}/{MODEL_NAME}.index"

if os.path.exists(FINAL_PTH) and os.path.exists(FINAL_IDX):
    print(f"Model already trained:\n  {FINAL_PTH}\n  {FINAL_IDX}")
    print("Delete those two files if you genuinely want to retrain.")
else:
    def run(cmd, cwd=RVC_DIR, label=""):
        """Run a training step, streaming output. RVC's scripts report failure by
        writing to stderr and exiting 0, so a non-zero check alone is not enough --
        surface everything and let the caller assert on the artefacts."""
        if label:
            print(f"\n--- {label} " + "-" * (56 - len(label)))
        p = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str),
                           capture_output=True, text=True)
        out = (p.stdout or "") + (p.stderr or "")
        print(out[-4000:] if len(out) > 4000 else out)
        if p.returncode != 0:
            raise RuntimeError(f"step failed ({label}) with code {p.returncode}")

    # -- 1. repo + weights --------------------------------------------------
    if not os.path.isdir(RVC_DIR):
        run(f"git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI {RVC_DIR}",
            cwd="/content", label="clone RVC")
        if RVC_COMMIT:
            run(f"git checkout {RVC_COMMIT}", label="pin commit")

    run("pip install -q faiss-cpu praat-parselmouth pyworld torchcrepe "
        "'fairseq @ git+https://github.com/One-sixth/fairseq.git' || "
        "pip install -q faiss-cpu praat-parselmouth pyworld torchcrepe fairseq",
        label="training deps")

    HF = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main"
    for rel, url in [
        ("assets/hubert/hubert_base.pt", f"{HF}/hubert_base.pt"),
        ("assets/rmvpe/rmvpe.pt", f"{HF}/rmvpe.pt"),
        (f"assets/pretrained_v2/f0G{RVC_SR}.pth", f"{HF}/pretrained_v2/f0G{RVC_SR}.pth"),
        (f"assets/pretrained_v2/f0D{RVC_SR}.pth", f"{HF}/pretrained_v2/f0D{RVC_SR}.pth"),
    ]:
        dst = os.path.join(RVC_DIR, rel)
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            run(f"wget -q -O '{dst}' '{url}'", label=f"fetch {os.path.basename(rel)}")

    exp = os.path.join(RVC_DIR, "logs", MODEL_NAME)
    os.makedirs(exp, exist_ok=True)
    sr_hz = {"32k": 32000, "40k": 40000, "48k": 48000}[RVC_SR]

    # -- 2. preprocess / f0 / features --------------------------------------
    run([sys.executable, "infer/modules/train/preprocess.py",
         DATASET_DIR, str(sr_hz), "4", exp, "False", "3.0"], label="preprocess")
    run([sys.executable, "infer/modules/train/extract/extract_f0_rmvpe.py",
         "1", "0", "0", exp, "True"], label="extract f0 (rmvpe)")
    run([sys.executable, "infer/modules/train/extract_feature_print.py",
         "cuda:0", "1", "0", "0", exp, "v2", "True"], label="extract features")

    # -- 3. filelist ---------------------------------------------------------
    # The WebUI builds this in a button handler, so it has to be rebuilt here. Each
    # line ties the four artefacts for one clip together; any clip missing one of
    # them is dropped rather than crashing training a few minutes in.
    gt, co, f0d, f0nsf = (f"{exp}/0_gt_wavs", f"{exp}/3_feature768",
                          f"{exp}/2a_f0", f"{exp}/2b-f0nsf")
    names = (set(os.path.splitext(n)[0] for n in os.listdir(gt))
             & set(os.path.splitext(n)[0] for n in os.listdir(co))
             & set(os.path.splitext(n)[0] for n in os.listdir(f0d))
             & set(os.path.splitext(n)[0] for n in os.listdir(f0nsf)))
    if not names:
        raise RuntimeError("no complete training items -- an extraction step failed above")

    lines = [f"{gt}/{n}.wav|{co}/{n}.npy|{f0d}/{n}.wav.npy|{f0nsf}/{n}.wav.npy|0"
             for n in sorted(names)]
    proto = os.path.join(RVC_DIR, "logs", "mute")
    if os.path.isdir(proto):   # the mute sample stabilises early training
        lines.append(f"{proto}/0_gt_wavs/mute{RVC_SR}.wav|{proto}/3_feature768/mute.npy|"
                     f"{proto}/2a_f0/mute.wav.npy|{proto}/2b-f0nsf/mute.wav.npy|0")
    with open(f"{exp}/filelist.txt", "w") as f:
        f.write("\n".join(lines))
    print(f"\nfilelist: {len(lines)} items")

    import shutil
    for cfg in [f"configs/v2/{RVC_SR}.json", f"configs/{RVC_SR}.json"]:
        src = os.path.join(RVC_DIR, cfg)
        if os.path.exists(src):
            shutil.copy(src, f"{exp}/config.json")
            break

    # -- 4. train ------------------------------------------------------------
    print(f"\nTraining {EPOCHS} epochs on {len(names)} clips. This is the long part.")
    run([sys.executable, "infer/modules/train/train.py",
         "-e", MODEL_NAME, "-sr", RVC_SR, "-f0", "1", "-bs", str(BATCH_SIZE),
         "-g", "0", "-te", str(EPOCHS), "-se", str(SAVE_EVERY),
         "-pg", f"assets/pretrained_v2/f0G{RVC_SR}.pth",
         "-pd", f"assets/pretrained_v2/f0D{RVC_SR}.pth",
         "-l", "0", "-c", "0", "-sw", "1", "-v", "v2"], label="train")

    # -- 5. faiss index ------------------------------------------------------
    # The index is what `index_rate` blends against at inference. Without it the
    # conversion loses a noticeable amount of timbre accuracy.
    import faiss
    import numpy as np
    feats = np.concatenate([np.load(f"{co}/{n}") for n in sorted(os.listdir(co))], axis=0)
    if feats.shape[0] > 2e5:  # subsample; a huge index is slower for no gain
        km = __import__("sklearn.cluster", fromlist=["MiniBatchKMeans"]).MiniBatchKMeans(
            n_clusters=10000, batch_size=256, init="random", n_init=1).fit(feats)
        feats = km.cluster_centers_
    nlist = max(1, int(16 * np.sqrt(feats.shape[0])))
    index = faiss.index_factory(feats.shape[1], f"IVF{nlist},Flat")
    index.train(feats)
    index.nprobe = 1
    for i in range(0, feats.shape[0], 8192):
        index.add(feats[i:i + 8192])
    faiss.write_index(index, FINAL_IDX)

    # -- 6. collect the final weights ---------------------------------------
    weights = os.path.join(RVC_DIR, "assets", "weights", f"{MODEL_NAME}.pth")
    if not os.path.exists(weights):
        cands = [f"{exp}/{f}" for f in os.listdir(exp) if f.startswith("G_") and f.endswith(".pth")]
        if not cands:
            raise RuntimeError("training produced no .pth -- check the training log above")
        weights = max(cands, key=os.path.getmtime)
    shutil.copy(weights, FINAL_PTH)

    print(f"\nDone:\n  {FINAL_PTH}\n  {FINAL_IDX}")
    print("\n  BACK THESE TWO FILES UP to Drive now. Colab will delete them when the")
    print("  runtime recycles, and retraining costs GPU-hours you have already spent.")

## Step 20 — Polish the voice

Cleans up the audio and locks its length so the pictures stay in sync. Writes `full_vo.wav`,
which is the only audio file anything after this reads.

In [ ]:
# =============================================================================
#  STAGE B - RVC: CONVERT THE NARRATION TO YOUR VOICE, THEN MASTER
# =============================================================================
#  vo_raw.wav (either your recording, or Chatterbox output)
#     -> RVC, target = your trained model  -> vo_rvc.wav  (your timbre, US accent)
#     -> Stage A master, timeline profile  -> full_vo.wav (what the mux cell reads)
#
#  WHY THIS ORDER. RVC converts *timbre*. It takes phonetic content and pitch from
#  the source and re-synthesises them in the target speaker's voice -- so prosody,
#  rhythm and articulation, which is what an accent actually is, come from the
#  source and survive conversion. Running your own accented recording through a US
#  voice model gives a US-sounding voice with a Pakistani accent. Running native-US
#  TTS through a model of YOUR voice gives your voice with a US accent. Accent comes
#  from whatever generates the phonemes; identity comes from the RVC target.
#
#  THE LENGTH LOCK. The mux cell fails if frame durations drift >0.5s from the audio,
#  and scenes_timeline.json was computed from Chatterbox's sample counts. RVC is
#  frame-synchronous so duration should survive, but "should" is not a guarantee
#  worth betting a nine-minute render on -- the sample count is asserted below and
#  corrected if it drifts by a hair.
# =============================================================================

import gc
import os

import numpy as np
import soundfile as sf

RVC_MODEL = "voice_model/myvoice.pth"
RVC_INDEX = "voice_model/myvoice.index"

# --- the knobs that matter -------------------------------------------------
INDEX_RATE = 0.60     # timbre accuracy vs artefacts. 0.5-0.75 is the useful band.
PROTECT = 0.33        # protects consonants/breath from being voiced-over. Lower
                      # = stronger timbre transfer, but sibilants start to smear.
F0_METHOD = "rmvpe"   # best quality of the available pitch extractors
F0_UP_KEY = 0         # semitones. Non-zero only if your model's natural pitch sits
                      # far from Chatterbox's -- listen before changing it.
RMS_MIX_RATE = 0.25   # 0 = follow the source's dynamics, 1 = the model's
FILTER_RADIUS = 3     # median-filters the pitch curve, reduces warble

SRC = "vo_raw.wav"
DST_RVC = "vo_rvc.wav"
DST_FINAL = "full_vo.wav"

# On the human path RVC is off by default, and that is a deliberate default rather
# than a limitation. Converting your own recording through a model of your own voice
# changes almost nothing except adding vocoder artefacts, and it will NOT change your
# accent -- RVC preserves the source's prosody and articulation. Set this True only
# to hear that for yourself.
RVC_ON_HUMAN = False

if not os.path.exists(SRC):
    raise FileNotFoundError(
        f"{SRC} not found - run the narration cell for NARRATION={NARRATION!r} first."
    )

have_model = os.path.exists(RVC_MODEL)
if NARRATION == "human":
    use_rvc = have_model and RVC_ON_HUMAN
    if not use_rvc:
        print("Human narration: mastering your own voice, no conversion.")
        print("This is your real voice - no AI disclosure needed for the audio.\n")
else:
    use_rvc = have_model
    if not have_model:
        print(f"No RVC model at {RVC_MODEL}.")
        print("Skipping identity conversion - mastering the TTS output as-is.")
        print("Train the model in Cell 10.7 to enable this stage.\n")

src_x, src_sr = sf.read(SRC, always_2d=False)
src_seconds = len(src_x) / src_sr

# --------------------------------------------------------------- conversion
if use_rvc:
    # Installed lazily rather than in the env cell: it pulls fairseq and a torch
    # pin, and there is no reason to risk that against chatterbox's deps on runs
    # where no model exists yet.
    try:
        from rvc_python.infer import RVCInference
    except ImportError:
        import subprocess as _sp
        import sys as _sys
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "rvc-python"], check=True)
        from rvc_python.infer import RVCInference
    import torch

    print(f"Converting {src_seconds/60:.2f} min through {os.path.basename(RVC_MODEL)}...")
    rvc = RVCInference(device="cuda:0" if torch.cuda.is_available() else "cpu")
    rvc.load_model(RVC_MODEL, index_path=RVC_INDEX if os.path.exists(RVC_INDEX) else None)
    rvc.set_params(f0method=F0_METHOD, f0up_key=F0_UP_KEY, index_rate=INDEX_RATE,
                   protect=PROTECT, rms_mix_rate=RMS_MIX_RATE, filter_radius=FILTER_RADIUS)
    rvc.infer_file(SRC, DST_RVC)

    del rvc
    gc.collect()
    torch.cuda.empty_cache()

    x, sr = sf.read(DST_RVC, always_2d=False)
    x = _to_mono(np.asarray(x, dtype=np.float64))
    print(f"  RVC output: {len(x)/sr:.2f}s @ {sr} Hz  (source {src_seconds:.2f}s)")

    # RVC commonly returns a different sample rate than it was given. Resample back
    # so the timeline is expressed in the units the rest of the pipeline expects.
    if sr != src_sr:
        g = np.gcd(int(src_sr), int(sr))
        x = sps.resample_poly(x, int(src_sr) // g, int(sr) // g)
        sr = src_sr
        print(f"  resampled to {sr} Hz")

    drift = len(x) / sr - src_seconds
    if abs(drift) > 0.25:
        raise RuntimeError(
            f"RVC changed the duration by {drift:+.2f}s. scenes_timeline.json no longer\n"
            f"describes this audio and the mux would desync. Investigate before continuing."
        )
    want = int(round(src_seconds * sr))
    if len(x) != want:      # sub-sample rounding; pad or trim silently
        x = np.pad(x, (0, want - len(x))) if len(x) < want else x[:want]
    print(f"  duration locked to {len(x)/sr:.3f}s (drift was {drift*1000:+.0f}ms)")
else:
    x, sr = _to_mono(np.asarray(src_x, dtype=np.float64)), src_sr

# ------------------------------------------------------------ final mastering
print("\nMastering (timeline profile - length locked)...")
n_before = len(x)
y = master(x, sr, profile="timeline", target_lufs=-14.0)   # -14 = YouTube's normalisation
assert len(y) == n_before, "mastering changed the length"
sf.write(DST_FINAL, y.astype(np.float32), sr, subtype="PCM_16")

print()
report(DST_FINAL)

# ------------------------------------------------------------------ QA gate
final_seconds = len(y) / sr
timeline_seconds = sum(s["duration"] for s in scenes)
print(f"\n  timeline sums to {timeline_seconds:.2f}s, audio is {final_seconds:.2f}s, "
      f"drift {abs(timeline_seconds - final_seconds):.3f}s")
if abs(timeline_seconds - final_seconds) > 0.5:
    raise RuntimeError("drift exceeds the mux cell's tolerance - do not proceed")
print("  within the mux cell's 0.5s tolerance.\n")

if NARRATION == "human" and not use_rvc:
    AI_VOICE_DISCLOSURE = False
    print("  Your own voice, mastered only. Nothing synthetic in this audio.")
else:
    AI_VOICE_DISCLOSURE = True
    what = "AI-generated (TTS)" if not use_rvc else "AI-generated (TTS) and AI-converted (RVC)"
    if NARRATION == "human":
        what = "AI-converted (RVC)"
    print(f"  DISCLOSURE: this voiceover is {what}.")
    print("  YouTube requires altered/synthetic-content disclosure at upload. The")
    print("  upload cell reads AI_VOICE_DISCLOSURE - leave it True.")

## Step 21 — Check the voice *(optional)*

Transcribes what was said and compares it to what was written.

In [ ]:
# =============================================================================
#  OPTIONAL: TTS QUALITY CHECK
# =============================================================================
# Whisper is no longer load-bearing - Cell 11 already produced exact timings. It is kept
# here as a verification pass, which is a better use for it: transcribe the audio you
# actually generated and compare it against the script you meant to generate. Chatterbox
# does occasionally drop or slur a clause, and that failure is silent otherwise - you
# only find it by watching the finished video.
#
# Skip this cell if you're short on time. Nothing downstream depends on it.
# =============================================================================

import difflib
import gc
import re
import torch
from faster_whisper import WhisperModel

RUN_QA = True   # set False to skip

if RUN_QA:
    print("Transcribing generated audio for QA...")
    wm = WhisperModel("base", device="cuda", compute_type="float16")
    segs, _ = wm.transcribe("full_vo.wav", word_timestamps=False)
    heard = " ".join(s.text.strip() for s in segs)

    def norm(t):
        return re.sub(r"[^a-z0-9 ]", "", t.lower()).split()

    intended_w, heard_w = norm(master_script), norm(heard)
    ratio = difflib.SequenceMatcher(None, intended_w, heard_w).ratio()

    print(f"\n  intended: {len(intended_w)} words")
    print(f"  heard:    {len(heard_w)} words")
    print(f"  match:    {ratio*100:.1f}%")

    if ratio > 0.93:
        print("  -> TTS is clean.")
    else:
        print("  -> Below 93%. Chatterbox likely dropped or garbled something.")
        print("     Biggest divergences:")
        sm = difflib.SequenceMatcher(None, intended_w, heard_w)
        shown = 0
        for tag, i1, i2, j1, j2 in sm.get_opcodes():
            if tag != "equal" and (i2 - i1) > 3 and shown < 6:
                print(f"       script: ...{' '.join(intended_w[i1:i2])[:90]}...")
                print(f"       audio : ...{' '.join(heard_w[j1:j2])[:90]}...\n")
                shown += 1
        print("     Re-run Cell 11 (Chatterbox is stochastic) or shorten MAX_CHUNK_CHARS.")

    with open("tts_qa_transcript.txt", "w", encoding="utf-8") as f:
        f.write(heard)

    del wm
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
else:
    print("QA skipped.")


## Step 22 — Lock the look

Read the comment at the top of this cell. It explains what actually keeps 150 frames looking
like one channel &mdash; and which part of that FLUX is *not* good at.

In [ ]:
import os
import textwrap
import numpy as np
import requests
from PIL import Image, ImageDraw, ImageFont

# =============================================================================
#  STYLE LOCK  —  the part of visual consistency that is NOT left to FLUX
# =============================================================================
#  BE HONEST ABOUT WHAT FLUX CAN AND CANNOT DO.
#
#  A fixed seed does NOT keep your character consistent. A seed only decides the
#  starting noise; change the prompt and the image goes somewhere completely
#  different. Two frames generated with seed 42 and different action clauses are
#  as unrelated as two frames with different seeds. So character drift across a
#  150-frame video is real and prompting alone will not fix it.
#
#  What DOES hold the video together, in order of how much work it does:
#
#    1. PALETTE LOCK      every frame snapped to the same 6 colours. Deterministic,
#                         runs on CPU in ~0.2s per frame, and it is the single
#                         biggest reason frames start to look like one channel.
#    2. WHITE SNAP        every near-white pixel forced to pure white, so the
#                         background is identical in every frame.
#    3. CAPTION OVERLAY   the on-screen text drawn by PIL in one fixed font at one
#                         fixed position. FLUX cannot spell; this never misspells.
#    4. CHARACTER SHEET   you generate a few characters, pick the one you like, and
#                         it becomes the visual North Star you judge frames against
#                         in the review step.
#    5. PROMPT LOCK       CHARACTER_BIBLE + STYLE_PREFIX repeated verbatim. Helps,
#                         but it is the weakest layer, not the strongest.
#
#  If you later want true frame-to-frame character identity, the only real answer
#  is training a style LoRA on 20-40 images of this exact look. That needs more
#  VRAM than a free T4 gives you. Layers 1-4 get you most of the way there without it.
# =============================================================================


# ---------------------------------------------------------------- palette lock
def _palette_image(palette):
    p = Image.new("P", (1, 1))
    flat = [c for rgb in palette for c in rgb]
    flat += [0] * (768 - len(flat))
    p.putpalette(flat)
    return p


_PAL_IMG = _palette_image(PALETTE)


def lock_palette(img: Image.Image) -> Image.Image:
    """Force a frame onto the channel palette. This is what makes frame 3 and
    frame 140 look like the same person drew them."""
    img = img.convert("RGB")
    a = np.asarray(img).copy()
    a[(a > WHITE_SNAP).all(axis=2)] = 255            # background -> pure white
    img = Image.fromarray(a)
    q = img.quantize(palette=_PAL_IMG, dither=Image.Dither.NONE)
    return q.convert("RGB")


# ---------------------------------------------------------------- caption font
def ensure_font(path: str, url: str) -> str:
    if not os.path.exists(path):
        print(f"  fetching font {path} ...")
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    return path


FONT_URLS = {
    "PermanentMarker-Regular.ttf":
        "https://raw.githubusercontent.com/google/fonts/main/apache/permanentmarker/PermanentMarker-Regular.ttf",
    "Anton-Regular.ttf":
        "https://raw.githubusercontent.com/google/fonts/main/ofl/anton/Anton-Regular.ttf",
}
for _f, _u in FONT_URLS.items():
    ensure_font(_f, _u)


def draw_caption(img: Image.Image, text: str) -> Image.Image:
    """Hand-letter the script line into the empty space, top-left. Always the same
    font, always the same corner — that repetition is a big part of the style."""
    if not text:
        return img
    img = img.copy()
    d = ImageDraw.Draw(img)
    text = text.upper().strip()

    box_w = int(FRAME_WIDTH * CAPTION_MAX_W)
    box_h = int(FRAME_HEIGHT * CAPTION_MAX_H)

    size, font, wrapped, w, h = 92, None, text, 0, 0
    while size > 22:
        font = ImageFont.truetype(CAPTION_FONT, size)
        wrapped = textwrap.fill(text, width=max(6, int(box_w / (size * 0.52))))
        bb = d.multiline_textbbox((0, 0), wrapped, font=font, spacing=10)
        w, h = bb[2] - bb[0], bb[3] - bb[1]
        if w <= box_w and h <= box_h:
            break
        size -= 4

    d.multiline_text(
        (int(FRAME_WIDTH * CAPTION_X), int(FRAME_HEIGHT * CAPTION_Y)),
        wrapped, font=font, fill=CAPTION_COLOR, spacing=10,
    )
    return img


def finish_frame(img: Image.Image, caption: str = "") -> Image.Image:
    """Every frame in the video goes through exactly this, in exactly this order."""
    if PALETTE_LOCK:
        img = lock_palette(img)
    if CAPTIONS_ON and caption:
        img = draw_caption(img, caption)
    return img


print("Style lock ready.")
print(f"  palette lock : {'ON' if PALETTE_LOCK else 'off'}  ({len(PALETTE)} colours)")
print(f"  captions     : {'ON' if CAPTIONS_ON else 'off'}  ({CAPTION_FONT})")


  fetching font PermanentMarker-Regular.ttf ...
  fetching font Anton-Regular.ttf ...
Style lock ready.
  palette lock : ON  (6 colours)
  captions     : ON  (PermanentMarker-Regular.ttf)


## Step 23 — Pick your character *(once per channel)*

In [ ]:
!pip install --upgrade pip
!pip install -U accelerate bitsandbytes diffusers transformers
import gc
import os
import torch
import ipywidgets as widgets
from IPython.display import display
# =============================================================================
#  Generate a few versions of your character, look at them, pick your favourite.
#  The one you pick is saved as character_ref.png and shown beside every frame in
#  the review step — so "does this frame still look like my character?" becomes a
#  question you can actually answer instead of a vibe.
#  Takes ~3 minutes. You only need to do this once per channel, not per video.
# =============================================================================

N_CANDIDATES = 4
os.makedirs("character", exist_ok=True)

POSES = [
    "standing still facing forward, arms relaxed at the sides",
    "sitting on a plain flat-coloured chair, hands on knees",
    "one arm raised, pointing at something off to the right",
    "walking to the left, mid-stride",
]

# BUG FIX: this used device_map="auto" + enable_model_cpu_offload(), which pinned
# all 24GB of FLUX weights onto a 15GB T4 and OOM'd before generating anything.
# load_flux() (Step 1 loader cell) does the 4-bit load that actually fits.
free_gpu()          # make sure nothing from an earlier cell is still holding VRAM
gpu_report("before load")
pipe = load_flux()

paths = []
for i, pose in enumerate(POSES[:N_CANDIDATES]):
    prompt = f"{STYLE_PREFIX}{CHARACTER_BIBLE}, {pose}"
    g = torch.Generator("cpu").manual_seed(FLUX_SEED + i * 1000)
    with torch.inference_mode():
        im = pipe(prompt, width=FRAME_WIDTH, height=FRAME_HEIGHT,
                  num_inference_steps=FLUX_STEPS, output_type="pil", generator=g).images[0]
    im = finish_frame(im)                      # same style lock the real frames get
    p = f"character/candidate_{i}.png"
    im.save(p)
    paths.append(p)
    print(f"  [{i+1}/{N_CANDIDATES}] {p}")

free_gpu(pipe)
del pipe
gpu_report("after unload")

# ---- pick one ----------------------------------------------------------------
thumbs, status = [], widgets.HTML()
for i, p in enumerate(paths):
    with open(p, "rb") as f:
        thumbs.append(widgets.VBox([
            widgets.Image(value=f.read(), format="png", width=300),
            widgets.Button(description=f"Use #{i}", button_style="primary"),
        ]))


def _make_pick(i):
    def _pick(_):
        Image.open(paths[i]).save("character_ref.png")
        status.value = (f'<b style="color:green">Character #{i} saved as '
                        f'character_ref.png — this is your channel character now.</b>')
    return _pick


for i, box in enumerate(thumbs):
    box.children[1].on_click(_make_pick(i))

display(widgets.VBox([
    widgets.HTML("<b>Pick the character you want. Click the button under it.</b>"),
    widgets.HBox(thumbs[:2]), widgets.HBox(thumbs[2:]), status,
]))

ModuleNotFoundError: No module named 'torch'

In [ ]:
!nvidia-smi

Fri Aug 21 12:45:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             27W /   70W |   14837MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 24 — Plan the pictures

In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor

with open("scenes_timeline.json", "r", encoding="utf-8") as f:
    scenes = json.load(f)

# =============================================================================
#  For each spoken line, decide TWO things:
#    action  — what the character is physically doing (goes to FLUX)
#    caption — the words hand-lettered onto the frame (drawn by PIL, not FLUX)
#
#  The caption is the reference channel's signature move: a short instruction or
#  punchline sitting in the white space next to the drawing. It is not on every
#  frame — mostly on directives, questions, and hard short statements — so the
#  model returns "" when a frame should be picture-only.
# =============================================================================

SECTION_DIRECTION = {
    1: "tight and physical, close on the body part. The character is doing the thing the viewer was just told to do.",
    2: "wider and more contextual. A comparison, a timeline, or an object being examined.",
    3: "the evidence made visible - a measurement, a comparison, an experiment in progress.",
    4: "process and machinery. Show the mechanism as a simple diagram or cutaway the character interacts with.",
    5: "sparse and wide. Landscape, weather, distance, one small figure. Room to breathe.",
    6: "return to the opening composition, quieter. The character alone, still.",
}

VISUAL_PROMPT_SYSTEM = """You are the visual director for a minimalist stick-figure
cartoon channel. The drawing style is fixed and handled elsewhere - never mention it.

For ONE line of voiceover, return ONLY this JSON, no fences, no commentary:

{"action": "...", "caption": "..."}

"action": 5-12 words describing ONLY what the character is physically doing and what
  concrete object is in the scene. It must be drawable by someone who can draw a stick
  figure and one simple prop. No abstract nouns. No style words. No camera words.
  At most ONE prop in the scene.

"caption": the words to hand-letter onto the frame, or "" for no text.
  - Use a caption when the line is a direct instruction to the viewer, a question, or
    a blunt short statement that lands harder in writing.
  - Use "" when the line is explanatory, descriptive, or long. Most frames get "".
  - Never more than 5 words. Take the words from the voiceover line itself, shortened.
  - Never invent a fact, number or name that is not in the line."""

MAX_RETRIES = 3
MAX_WORKERS = 8


def _valid(d):
    if not isinstance(d, dict):
        return False
    a = (d.get("action") or "").strip()
    c = (d.get("caption") or "").strip()
    if not a or len(a.split()) > 16:
        return False
    if len(c.split()) > 5:
        return False
    return not a.lower().startswith(("here is", "here's", "i cannot", "sure,", "note:"))


def plan_scene(scene):
    direction = SECTION_DIRECTION.get(scene.get("section", 1), "")
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            raw = call_model(
                MODEL_CHECKER,
                f"Section direction: {direction}\n\nVoiceover line: {scene['text']}",
                system=VISUAL_PROMPT_SYSTEM, max_tokens=150,
                temperature=0.3, retries=1,
            )
            d = parse_json_response(raw)
            if _valid(d):
                return {"action": d["action"].strip().strip('"'),
                        "caption": (d.get("caption") or "").strip().strip('"')}
            last = f"invalid: {raw[:80]!r}"
        except Exception as e:
            last = str(e)
        time.sleep(1.5 * (attempt + 1))
    print(f"  [warn] fallback for: {scene['text'][:55]!r} ({last})")
    return {"action": "", "caption": ""}


print(f"Planning visuals for {len(scenes)} scenes ({MAX_WORKERS} at a time)...")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    plans = list(pool.map(plan_scene, scenes))

for scene, plan in zip(scenes, plans):
    scene["action_clause"] = plan["action"]
    scene["caption"] = plan["caption"]
    scene["visual_prompt"] = (
        f"{STYLE_PREFIX}{CHARACTER_BIBLE}, {plan['action']}" if plan["action"]
        else f"{STYLE_PREFIX}{CHARACTER_BIBLE}, standing still facing forward"
    )

scenes.sort(key=lambda s: s["scene_index"])
with open("scenes_planned.json", "w", encoding="utf-8") as f:
    json.dump(scenes, f, indent=2)

n_fb = sum(1 for s in scenes if not s["action_clause"])
n_cap = sum(1 for s in scenes if s["caption"])
print(f"\nDone. {len(scenes) - n_fb}/{len(scenes)} scenes planned"
      f"{f', {n_fb} fell back to a default pose' if n_fb else ''}.")
print(f"{n_cap} frames will carry on-screen text.\n")
for s in scenes[:6]:
    print(f'  #{s["scene_index"]}  action: {s["action_clause"]}')
    print(f'       caption: {s["caption"] or "(none)"}')


## Step 25 — Draw the pictures

The slow one. Safe to interrupt — re-running picks up where it stopped.

In [ ]:
import gc
import os
import json
import time
import torch
# =============================================================================
#  Draw every frame, then run each one through the style lock (palette + caption).
#  Safe to interrupt: frames already on disk are skipped, so a disconnect costs you
#  only the frames that weren't finished. Re-run this cell and it picks up.
#
#  Roughly 25-45 seconds per frame on a free T4. A 9-minute script is usually
#  120-180 frames, so budget 1.5-2.5 hours and keep the browser tab in front.
# =============================================================================

with open("scenes_planned.json", "r", encoding="utf-8") as f:
    scenes = json.load(f)

os.makedirs("frames", exist_ok=True)


def frame_path(idx):
    return f"frames/frame_{idx:04d}.png"


todo = [s for s in scenes if not os.path.exists(frame_path(s["scene_index"]))]
print(f"{len(scenes)} scenes total, {len(todo)} still to draw "
      f"({len(scenes) - len(todo)} already done)")

if todo:
    free_gpu()
    pipe = load_flux()

    t0 = time.time()
    for n, scene in enumerate(todo):
        idx = scene["scene_index"]
        g = torch.Generator("cpu").manual_seed(FLUX_SEED)
        with torch.inference_mode():
            image = pipe(
                scene["visual_prompt"],
                width=FRAME_WIDTH, height=FRAME_HEIGHT,
                num_inference_steps=FLUX_STEPS,
                output_type="pil", generator=g,
            ).images[0]

        finish_frame(image, scene.get("caption", "")).save(frame_path(idx))

        del image
        gc.collect()
        torch.cuda.empty_cache()

        if (n + 1) % 10 == 0 or n == len(todo) - 1:
            el = time.time() - t0
            rate = el / (n + 1)
            print(f"  {n+1}/{len(todo)}  ({rate:.1f}s each, ~{rate*(len(todo)-n-1)/60:.0f} min left)")

    free_gpu(pipe)
    del pipe
    gpu_report("after unload")

missing = [s["scene_index"] for s in scenes if not os.path.exists(frame_path(s["scene_index"]))]
if missing:
    print(f"\nWARNING: {len(missing)} frame(s) still missing: {missing[:10]}")
    print("Just re-run this cell — it resumes.")
else:
    print(f"\nAll {len(scenes)} frames drawn at {FRAME_WIDTH}x{FRAME_HEIGHT}.")
    print("Now run Step 24 and flick through them.")


## Step 26 — Check the pictures

In [ ]:
import gc
import os
import json
import torch
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass

# =============================================================================
#  Flick through every frame next to your chosen character. If a frame doesn't
#  look like the same channel, fix its words and hit Redraw. Nothing else in the
#  pipeline needs to be re-run — just this frame.
# =============================================================================

with open("scenes_planned.json", "r", encoding="utf-8") as f:
    scenes = json.load(f)

by_index = {s["scene_index"]: s for s in scenes}
order = sorted(by_index)
state = {"pos": 0, "pipe": None}


def _fp(i):
    return f"frames/frame_{i:04d}.png"


def _pipe():
    if state["pipe"] is None:
        status.value = "Loading FLUX (first redraw only, ~2 min)..."
        free_gpu()
        state["pipe"] = load_flux()
    return state["pipe"]


position   = widgets.HTML()
img        = widgets.Image(format="png", width=560)
ref_img    = widgets.Image(format="png", width=210)
line_lbl   = widgets.HTML()
action_box = widgets.Text(description="Action:", layout=widgets.Layout(width="97%"),
                          style={"description_width": "70px"})
cap_box    = widgets.Text(description="Caption:", layout=widgets.Layout(width="97%"),
                          style={"description_width": "70px"})
seed_box   = widgets.IntText(value=FLUX_SEED, description="Seed:", layout=widgets.Layout(width="170px"))
jump_box   = widgets.IntText(value=0, description="Go to:", layout=widgets.Layout(width="170px"))
status     = widgets.HTML()

prev_b  = widgets.Button(description="< Prev")
next_b  = widgets.Button(description="Next >")
jump_b  = widgets.Button(description="Jump")
redraw_b= widgets.Button(description="Redraw this frame", button_style="warning")
cap_b   = widgets.Button(description="Update caption only", button_style="info")
free_b  = widgets.Button(description="Free GPU memory", button_style="danger")

if os.path.exists("character_ref.png"):
    with open("character_ref.png", "rb") as f:
        ref_img.value = f.read()


def refresh():
    idx = order[state["pos"]]
    sc = by_index[idx]
    p = _fp(idx)
    if os.path.exists(p):
        with open(p, "rb") as f:
            img.value = f.read()
    else:
        img.value = b""
        status.value = f"(no frame yet at {p})"
    line_lbl.value = (f'<div style="font-family:system-ui">'
                      f'<b>Section {sc.get("section","?")} &middot; {sc["duration"]}s</b>'
                      f'<br><i>"{sc["text"]}"</i></div>')
    action_box.value = sc.get("action_clause", "")
    cap_box.value = sc.get("caption", "")
    position.value = (f'<h4 style="font-family:system-ui;margin:4px 0">'
                      f'Frame {state["pos"]+1} of {len(order)}</h4>')


def _save():
    with open("scenes_planned.json", "w", encoding="utf-8") as f:
        json.dump(scenes, f, indent=2)


def on_redraw(_):
    idx = order[state["pos"]]
    sc = by_index[idx]
    action = action_box.value.strip()
    if not action:
        status.value = "Write what the character is doing first."
        return
    sc["action_clause"] = action
    sc["caption"] = cap_box.value.strip()
    sc["visual_prompt"] = f"{STYLE_PREFIX}{CHARACTER_BIBLE}, {action}"

    status.value = "Drawing..."
    g = torch.Generator("cpu").manual_seed(seed_box.value)
    with torch.inference_mode():
        im = _pipe()(sc["visual_prompt"], width=FRAME_WIDTH, height=FRAME_HEIGHT,
                     num_inference_steps=FLUX_STEPS, output_type="pil", generator=g).images[0]
    finish_frame(im, sc["caption"]).save(_fp(idx))
    del im
    gc.collect()
    torch.cuda.empty_cache()
    _save()
    refresh()
    status.value = "Redrawn and saved."


def on_caption_only(_):
    """Change the words without spending 40s redrawing. Only works if the frame
    hasn't had a caption burned in yet — otherwise redraw."""
    idx = order[state["pos"]]
    sc = by_index[idx]
    sc["caption"] = cap_box.value.strip()
    _save()
    status.value = ("Caption saved to the plan. The frame on disk still has the old "
                    "text burned in — hit Redraw to see it.")


prev_b.on_click(lambda _: (state.update(pos=max(0, state["pos"] - 1)), refresh()))
next_b.on_click(lambda _: (state.update(pos=min(len(order) - 1, state["pos"] + 1)), refresh()))
jump_b.on_click(lambda _: (state.update(pos=order.index(jump_box.value)), refresh())
                if jump_box.value in by_index else None)
redraw_b.on_click(on_redraw)
cap_b.on_click(on_caption_only)
def _on_free(_):
    p = state.get("pipe")
    state["pipe"] = None
    free_gpu(p)
    status.value = "GPU memory freed."
    gpu_report("now")

free_b.on_click(_on_free)

refresh()

display(widgets.VBox([
    position,
    widgets.HBox([img, widgets.VBox([
        widgets.HTML('<div style="font-family:system-ui;font-size:12px;color:#666">'
                     'Your character:</div>'), ref_img])]),
    line_lbl,
    widgets.HTML("<hr>"),
    action_box, cap_box,
    widgets.HBox([seed_box, jump_box, jump_b]),
    widgets.HBox([prev_b, next_b, redraw_b, cap_b]),
    status, widgets.HTML("<hr>"), free_b,
]))


## Step 27 — Build the video

In [ ]:
import os
import json
import subprocess
import wave
import contextlib

with open("scenes_planned.json", "r", encoding="utf-8") as f:
    scenes = json.load(f)

# =============================================================================
#  PREFLIGHT - v4 went straight to ffmpeg and failed mid-encode on a missing frame
# =============================================================================
print("Preflight:")
errors = []

missing = [s["scene_index"] for s in scenes if not os.path.exists(f"frames/frame_{s['scene_index']:04d}.png")]
if missing:
    errors.append(f"{len(missing)} frame(s) missing: {missing[:10]}")
print(f"  frames on disk           {'OK' if not missing else 'MISSING ' + str(len(missing))}")

if not os.path.exists("full_vo.wav"):
    errors.append("full_vo.wav not found - run the TTS cell")
    audio_len = 0.0
else:
    with contextlib.closing(wave.open("full_vo.wav", "r")) as w:
        audio_len = w.getnframes() / float(w.getframerate())
print(f"  audio length             {audio_len:.2f}s")

video_len = sum(s["duration"] for s in scenes)
print(f"  frame durations sum      {video_len:.2f}s")
drift = abs(video_len - audio_len)
if drift > 0.5:
    errors.append(f"video/audio length mismatch of {drift:.2f}s - the last frame will hold or the audio will be cut")
print(f"  drift                    {drift:.3f}s {'OK' if drift <= 0.5 else 'TOO HIGH'}")

if errors:
    print("\nPREFLIGHT FAILED:")
    for e in errors:
        print(f"  - {e}")
    raise RuntimeError("Fix the above before muxing.")
print("  all checks passed\n")

# =============================================================================
#  CONCAT LIST
# =============================================================================
# The concat demuxer ignores the duration of the final entry, so the last file is
# repeated with no duration to close the timeline out. That part v4 got right.
concat_path = "concat_list.txt"
with open(concat_path, "w", encoding="utf-8") as f:
    for s in scenes:
        f.write(f"file 'frames/frame_{s['scene_index']:04d}.png'\n")
        f.write(f"duration {s['duration']}\n")
    f.write(f"file 'frames/frame_{scenes[-1]['scene_index']:04d}.png'\n")

# =============================================================================
#  ENCODE
# =============================================================================
# Added over v4: an explicit output framerate and a scale/pad filter. Concatenated
# stills with variable durations produce a variable-framerate stream, which some
# players and YouTube's transcoder handle badly; "-r 30" with "-vsync cfr" resolves
# it to a clean constant-rate 30fps. The scale/pad guarantees exact 16:9 output even
# if a frame was regenerated at a different size in the review widget.
print(f"Muxing to {FRAME_WIDTH}x{FRAME_HEIGHT} @ {OUTPUT_FPS}fps...")

vf = (f"scale={FRAME_WIDTH}:{FRAME_HEIGHT}:force_original_aspect_ratio=decrease,"
      f"pad={FRAME_WIDTH}:{FRAME_HEIGHT}:(ow-iw)/2:(oh-ih)/2:white,"
      f"format=yuv420p")

cmd = [
    "ffmpeg", "-y",
    "-f", "concat", "-safe", "0", "-i", concat_path,
    "-i", "full_vo.wav",
    "-vf", vf,
    "-r", str(OUTPUT_FPS),
    "-vsync", "cfr",
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac", "-b:a", "192k", "-ar", "48000",
    "-movflags", "+faststart",
    "-shortest",
    "final_video.mp4",
]

proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise RuntimeError("ffmpeg failed - see the log above.")

size_mb = os.path.getsize("final_video.mp4") / 1e6
probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", "final_video.mp4"],
    capture_output=True, text=True,
)
final_len = float(probe.stdout.strip()) if probe.stdout.strip() else 0.0

print("\n" + "=" * 70)
print("RENDER COMPLETE: final_video.mp4")
print("=" * 70)
print(f"  {final_len/60:.2f} min | {size_mb:.1f} MB | {FRAME_WIDTH}x{FRAME_HEIGHT} @ {OUTPUT_FPS}fps")
print(f"  audio was {audio_len/60:.2f} min - a gap here means the VO got clipped")
print("\nAlso ready to upload:")
for fn in ["youtube_description.txt", "youtube_tags.txt", "pinned_comment.txt", "packaging.json"]:
    if os.path.exists(fn):
        print(f"  - {fn}")


## Step 28 — Thumbnail

In [ ]:
import gc
import os
import json
import textwrap

import torch
from PIL import Image, ImageDraw, ImageFont

with open("packaging.json", "r", encoding="utf-8") as f:
    pack = json.load(f)

thumb_candidates = pack.get("thumbnail_text", [])[:4]
if not thumb_candidates:
    raise RuntimeError("No thumbnail_text in packaging.json — re-run Cell 10 first.")

os.makedirs("thumbnails", exist_ok=True)

# =============================================================================
#  STYLE — deliberately different from CHARACTER_BIBLE/STYLE_PREFIX (Cell 2), which
#  is the flat black-on-white in-video whiteboard look. Thumbnails on the reference
#  channel are colored, lit scenes with a soft vignette — a completely different
#  register that only has to agree on one thing: the character reads as the same
#  "person" as the body of the video (round head, dot eyes, simple limbs).
# =============================================================================
THUMBNAIL_CHARACTER = (
    "a single character with a plain round head, two small dot eyes, no pupils detailed, "
    "eyebrows raised in worry or confusion, small simple mouth, thin uniform limbs, "
    "gender-neutral, no clothing patterns"
)

THUMBNAIL_STYLE_PREFIX = (
    "Flat digital illustration, bold clean shapes, muted warm earth-tone color palette, "
    "soft directional lighting, a single relevant background scene rendered with gentle "
    "depth, subtle dark vignette toward the frame edges, no text, no logos, no watermark, "
    "16:9 composition, "
)

SEED = 42  # same seed family as Cell 14, for whatever continuity FLUX's seed gives across styles

print("Loading FLUX [schnell] for thumbnail backgrounds...")
free_gpu()
pipe = load_flux()

backgrounds = []
for i, cand in enumerate(thumb_candidates):
    prompt = f"{THUMBNAIL_STYLE_PREFIX}{THUMBNAIL_CHARACTER}, {cand.get('visual', '')}, theme: {TOPIC}"
    generator = torch.Generator("cpu").manual_seed(SEED + i)
    with torch.inference_mode():
        image = pipe(
            prompt,
            width=FRAME_WIDTH,
            height=FRAME_HEIGHT,
            num_inference_steps=4,
            output_type="pil",
            generator=generator,
        ).images[0]
    path = f"thumbnails/bg_{i}.png"
    image.save(path)
    backgrounds.append(path)
    print(f"  [{i+1}/{len(thumb_candidates)}] background rendered: {path}")

free_gpu(pipe)
del pipe
gpu_report("after unload")

# =============================================================================
#  FONT — a real bold display font, not a system fallback. Impact/Anton-style is what
#  actually reads as "YouTube thumbnail" at small sizes; DejaVu-Bold does not.
#  Downloaded once, then reused — comment out the download if you already have one on
#  disk you prefer.
# =============================================================================
FONT_PATH = "Anton-Regular.ttf"
if not os.path.exists(FONT_PATH):
    import requests
    print("Fetching a bold display font (Anton, OFL-licensed, Google Fonts)...")
    r = requests.get(
        "https://raw.githubusercontent.com/google/fonts/main/ofl/anton/Anton-Regular.ttf",
        timeout=20,
    )
    r.raise_for_status()
    with open(FONT_PATH, "wb") as f:
        f.write(r.content)


def draw_thumbnail_text(bg_path: str, text: str, out_path: str):
    img = Image.open(bg_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    text = text.upper()
    max_width = int(FRAME_WIDTH * 0.9)
    max_height = int(FRAME_HEIGHT * 0.34)  # text lives in the top third, like the reference set

    # Shrink font size until the (possibly wrapped) text fits the box — no fixed guess.
    size = 160
    while size > 30:
        font = ImageFont.truetype(FONT_PATH, size)
        wrapped = textwrap.fill(text, width=max(4, int(max_width / (size * 0.6))))
        bbox = draw.multiline_textbbox((0, 0), wrapped, font=font, spacing=8, align="center")
        w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
        if w <= max_width and h <= max_height:
            break
        size -= 4

    x = (FRAME_WIDTH - w) / 2
    y = int(FRAME_HEIGHT * 0.06)
    draw.multiline_text(
        (x, y), wrapped, font=font, fill=(255, 214, 0),
        stroke_width=max(4, size // 18), stroke_fill=(0, 0, 0),
        align="center", spacing=8,
    )
    img.save(out_path)


final_thumbs = []
for i, cand in enumerate(thumb_candidates):
    out = f"thumbnails/thumb_{i}.png"
    draw_thumbnail_text(backgrounds[i], cand["text"], out)
    final_thumbs.append(out)
    print(f"  wrote {out}   text: \"{cand['text']}\"")

print(f"\n{len(final_thumbs)} thumbnail candidates in ./thumbnails/ — pick one before Cell 22 (upload).")
print("Set THUMBNAIL_CHOICE below to the index you're going with:")
THUMBNAIL_CHOICE = 0
print(f"  currently: thumbnails/thumb_{THUMBNAIL_CHOICE}.png")


## Step 29 — Save the run

Copies everything into `runs/<topic>_<date>/` so making the next video can't overwrite this one.

In [ ]:
import shutil
import datetime

# =============================================================================
#  ARCHIVE THE RUN
# =============================================================================
# Every cell above writes to flat filenames in the working directory, so producing a
# second topic silently overwrites the first. On a channel that needs a video a week
# that is the thing that eventually costs you a finished script. This copies the whole
# run into runs/<topic>_<timestamp>/ and writes a manifest, which is also what makes
# batching across topics practical: change TOPIC_KEY, re-run, nothing is lost.
# =============================================================================

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
run_dir = f"runs/{TOPIC_KEY}_{stamp}"
os.makedirs(run_dir, exist_ok=True)

ARTIFACTS = [
    "vo_raw.txt", "script_tagged.json", "source_pack.json", "evidence_profile.json",
    "packaging.json", "youtube_description.txt", "youtube_tags.txt", "pinned_comment.txt",
    "hook_tournament.json", "scenes_timeline.json", "scenes_planned.json",
    "sources_for_description.txt", "tts_qa_transcript.txt",
    "full_vo.wav", "final_video.mp4", "concat_list.txt",
]
ARTIFACTS += [f for f in os.listdir(".") if re.match(r"(validation|retention|lint)_round_\d+\.json", f)]

copied, skipped = [], []
for fn in ARTIFACTS:
    if os.path.exists(fn):
        shutil.copy2(fn, os.path.join(run_dir, fn))
        copied.append(fn)
    else:
        skipped.append(fn)

manifest = {
    "topic_key": TOPIC_KEY,
    "topic": TOPIC,
    "timestamp": stamp,
    "evidence_type": profile.get("evidence_type"),
    "section_3_template": BEAT_SHEET[3]["name"],
    "sources": [{"title": s["title"], "url": s["url"], "chars": len(s["excerpt"])} for s in source_pack],
    "script": {
        "words": len(strip_tags_to_voiceover(tagged_sentences).split()),
        "sentences": len(tagged_sentences),
        "sections": {
            str(n): sum(len(i["sentence"].split()) for i in tagged_sentences if i["section"] == n)
            for n in sorted(BEAT_SHEET)
        },
    },
    "open_issues": {
        "fact": len(fact_issues),
        "smuggled": len(smuggled_issues),
        "lint": len(lint_issues),
        "retention": len(retention_issues),
    },
    "api_usage": dict(USAGE),
    "files": copied,
}

with open(os.path.join(run_dir, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("=" * 70)
print(f"RUN ARCHIVED: {run_dir}/")
print("=" * 70)
print(f"  {len(copied)} artifact(s) copied")
if skipped:
    print(f"  not present (cells not run yet): {', '.join(skipped[:6])}"
          f"{' ...' if len(skipped) > 6 else ''}")
print(f"\n  topic          {TOPIC}")
print(f"  evidence type  {manifest['evidence_type']} -> \"{manifest['section_3_template']}\"")
print(f"  script         {manifest['script']['words']} words / {manifest['script']['sentences']} sentences")
print(f"  open issues    {sum(manifest['open_issues'].values())}")
print_usage()
print(f"\nTo produce the next video: change TOPIC_KEY in Cell 2 and re-run from Cell 3.")
print("Nothing in this run will be overwritten.")


## Step 30 — Upload *(optional)*

Uploads as **private** by default. Flip `PRIVACY_STATUS` when you're ready.

In [ ]:
import json
import time

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

VIDEO_FILE = "final_video.mp4"
THUMBNAIL_FILE = f"thumbnails/thumb_{THUMBNAIL_CHOICE}.png"   # set THUMBNAIL_CHOICE in Cell 18
PRIVACY_STATUS = "private"     # "private" | "unlisted" | "public" — flip to public only when ready
PUBLISH_AT = None              # e.g. "2026-08-25T15:00:00Z" for scheduled release (requires "private" + a future time)

with open("packaging.json", "r", encoding="utf-8") as f:
    pack = json.load(f)
with open("youtube_description.txt", "r", encoding="utf-8") as f:
    description = f.read()
with open("youtube_tags.txt", "r", encoding="utf-8") as f:
    tags = [t.strip() for t in f.read().split(",") if t.strip()]

CHOSEN_TITLE = pack["titles"][0]["text"]   # <-- pick the title you're going with, index into pack["titles"]

creds = get_google_creds()  # defined in Cell 19 — same token, union of scopes
youtube = build("youtube", "v3", credentials=creds)


def upload_video() -> str:
    body = {
        "snippet": {
            "title": CHOSEN_TITLE,
            "description": description,
            "tags": tags,
            "categoryId": "27",  # Education
        },
        "status": {
            "privacyStatus": PRIVACY_STATUS,
            "selfDeclaredMadeForKids": False,
        },
    }
    if PUBLISH_AT:
        body["status"]["publishAt"] = PUBLISH_AT

    media = MediaFileUpload(VIDEO_FILE, chunksize=-1, resumable=True, mimetype="video/mp4")
    request = youtube.videos().insert(part="snippet,status", body=body, media_body=media)

    response = None
    print("Uploading...")
    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f"  {int(status.progress() * 100)}%")
    video_id = response["id"]
    print(f"Uploaded: https://youtu.be/{video_id}")
    return video_id


def set_thumbnail(video_id: str):
    youtube.thumbnails().set(videoId=video_id, media_body=MediaFileUpload(THUMBNAIL_FILE)).execute()
    print(f"Thumbnail set from {THUMBNAIL_FILE}")


def post_comment(video_id: str, text: str):
    """Posts a normal top-level comment. NOT pinned — the Data API has no pin endpoint.
    Pin it yourself in Studio (Comments > ⋮ > Pin) — one click, thirty seconds."""
    youtube.commentThreads().insert(
        part="snippet",
        body={"snippet": {"videoId": video_id,
                           "topLevelComment": {"snippet": {"textOriginal": text}}}},
    ).execute()
    print("Comment posted (pin it manually in Studio — the API can't).")


def save_community_post_draft(text: str):
    """The Data API has no Community-post endpoint at all — this just saves the copy
    Cell 10 already wrote so you can paste it into Studio yourself."""
    with open("community_post_draft.txt", "w", encoding="utf-8") as f:
        f.write(text)
    print("Community post text saved to community_post_draft.txt — paste into YouTube Studio manually.")


# --- Run the upload ---
video_id = upload_video()
set_thumbnail(video_id)
post_comment(video_id, pack.get("pinned_comment", ""))
save_community_post_draft(pack.get("community_post", ""))

print(f"\nDone. Once this video has a few days of data, run Cell 19 with video_id={video_id!r} "
      f"to pull its retention curve into the next strategist run.")


## Step 31 — Learn from it *(optional, after a few days live)*

Pulls the real retention curve, looks at what's working on similar channels, and writes
notes into `learnings.json` that the script judge reads on the next video.

In [ ]:
import os
import json
import pickle

from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# =============================================================================
#  AUTH — shared with the upload cell (Cell 22). One token file, union of scopes,
#  so you only go through the consent screen once for the whole notebook.
# =============================================================================
SCOPES = [
    "https://www.googleapis.com/auth/yt-analytics.readonly",
    "https://www.googleapis.com/auth/youtube.upload",
    "https://www.googleapis.com/auth/youtube.force-ssl",
]
CLIENT_SECRETS_FILE = "client_secrets.json"  # download from Google Cloud Console > Credentials
TOKEN_FILE = "token.pickle"


def get_google_creds():
    creds = None
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE, "rb") as f:
            creds = pickle.load(f)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists(CLIENT_SECRETS_FILE):
                raise RuntimeError(
                    f"{CLIENT_SECRETS_FILE} not found. In Google Cloud Console: create an OAuth "
                    "client (Desktop app type), enable the YouTube Data API v3 and YouTube "
                    "Analytics API, download the JSON, upload it here as client_secrets.json."
                )
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRETS_FILE, SCOPES)
            # run_console(): Colab has no localhost redirect, so this prints a URL, you
            # authorize in a browser tab, and paste the code back here — one-time only.
            creds = flow.run_console()
        with open(TOKEN_FILE, "wb") as f:
            pickle.dump(creds, f)
    return creds


CHANNEL_ID = "UCxxxxxxxxxxxxxxxxxxxxxxxx"  # <-- your channel ID


def get_section_boundaries(scenes_path="scenes_planned.json"):
    """Cumulative section durations -> {section: (start_s, end_s)}, then normalized
    to (start_ratio, end_ratio) once total video length is known."""
    with open(scenes_path, "r", encoding="utf-8") as f:
        scenes = json.load(f)
    scenes = sorted(scenes, key=lambda s: s["scene_index"])
    total = sum(s["duration"] for s in scenes)
    bounds, t = {}, 0.0
    for s in scenes:
        sec = s["section"]
        if sec not in bounds:
            bounds[sec] = [t, t]
        bounds[sec][1] = t + s["duration"]
        t += s["duration"]
    return {sec: (a / total, b / total) for sec, (a, b) in bounds.items()}, total


def fetch_retention_curve(video_id: str, creds):
    """Returns [(elapsedVideoTimeRatio, audienceWatchRatio), ...] — the raw retention
    curve as YouTube Analytics reports it."""
    yta = build("youtubeAnalytics", "v2", credentials=creds)
    resp = yta.reports().query(
        ids=f"channel=={CHANNEL_ID}",
        startDate="2020-01-01",
        endDate="2030-01-01",
        metrics="audienceWatchRatio,relativeRetentionPerformance",
        dimensions="elapsedVideoTimeRatio",
        filters=f"video=={video_id}",
    ).execute()
    return [(row[0], row[1], row[2]) for row in resp.get("rows", [])]


def per_section_retention(video_id: str, scenes_path="scenes_planned.json"):
    """Maps the raw curve onto your beat-sheet sections and returns average watch
    ratio + relative performance (vs. YouTube's similar-video benchmark) per section
    — the number that actually says 'this section is where people leave'."""
    creds = get_google_creds()
    bounds, _ = get_section_boundaries(scenes_path)
    curve = fetch_retention_curve(video_id, creds)

    per_section = {}
    for sec, (lo, hi) in bounds.items():
        points = [(watch, rel) for ratio, watch, rel in curve if lo <= ratio < hi]
        if points:
            per_section[sec] = {
                "avg_watch_ratio": sum(p[0] for p in points) / len(points),
                "avg_relative_performance": sum(p[1] for p in points) / len(points),
                "name": BEAT_SHEET.get(sec, {}).get("name", f"section {sec}"),
            }
    return per_section


# --- Example usage — fill in a published video ID once you have one ---
# PUBLISHED_VIDEO_ID = "dQw4w9WgXcQ"
# result = per_section_retention(PUBLISHED_VIDEO_ID)
# print(json.dumps(result, indent=2))
print("Analytics helpers ready. Call per_section_retention('<video_id>') once a video is live.")


In [ ]:
import os
import json
import datetime as dt

from googleapiclient.discovery import build

YOUTUBE_API_KEY = os.environ.get("YOUTUBE_API_KEY", "")  # <-- paste or set as env var
if not YOUTUBE_API_KEY:
    raise RuntimeError("Set YOUTUBE_API_KEY — this is a separate, simple API key from Cloud "
                        "Console (API key, not OAuth client), scoped to YouTube Data API v3.")

COMPETITOR_CHANNEL_IDS = [
    # "UCxxxxxxxxxxxxxxxxxxxxxxxx",   # <-- fill in competitor channel IDs
]

yt = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)


def _uploads_playlist_id(channel_id: str) -> str:
    resp = yt.channels().list(part="contentDetails", id=channel_id).execute()
    items = resp.get("items", [])
    if not items:
        raise RuntimeError(f"Channel not found: {channel_id}")
    return items[0]["contentDetails"]["relatedPlaylists"]["uploads"]


def fetch_channel_recent_videos(channel_id: str, max_videos: int = 25) -> list[dict]:
    playlist_id = _uploads_playlist_id(channel_id)
    video_ids, page_token = [], None
    while len(video_ids) < max_videos:
        resp = yt.playlistItems().list(
            part="contentDetails", playlistId=playlist_id,
            maxResults=min(50, max_videos - len(video_ids)), pageToken=page_token,
        ).execute()
        video_ids.extend(i["contentDetails"]["videoId"] for i in resp.get("items", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    videos = []
    for i in range(0, len(video_ids), 50):
        chunk = video_ids[i:i + 50]
        resp = yt.videos().list(part="snippet,statistics,contentDetails", id=",".join(chunk)).execute()
        for v in resp.get("items", []):
            published = dt.datetime.fromisoformat(v["snippet"]["publishedAt"].replace("Z", "+00:00"))
            age_days = max(1, (dt.datetime.now(dt.timezone.utc) - published).days)
            views = int(v["statistics"].get("viewCount", 0))
            videos.append({
                "video_id": v["id"],
                "title": v["snippet"]["title"],
                "published": v["snippet"]["publishedAt"],
                "views": views,
                "view_velocity": round(views / age_days, 1),
                "duration": v["contentDetails"]["duration"],  # ISO 8601, e.g. PT8M31S
            })
    return videos


def fetch_competitor_intel(channel_ids: list[str]) -> dict:
    intel = {}
    for cid in channel_ids:
        print(f"  fetching {cid}...")
        vids = fetch_channel_recent_videos(cid)
        vids.sort(key=lambda v: v["view_velocity"], reverse=True)
        intel[cid] = vids
    with open("competitor_intel.json", "w", encoding="utf-8") as f:
        json.dump(intel, f, indent=2)
    return intel


if COMPETITOR_CHANNEL_IDS:
    print("Fetching competitor intel...")
    competitor_intel = fetch_competitor_intel(COMPETITOR_CHANNEL_IDS)
    for cid, vids in competitor_intel.items():
        print(f"\n{cid} — top 5 by view velocity:")
        for v in vids[:5]:
            print(f"  {v['view_velocity']:>8}/day  {v['views']:>10} views   {v['title']}")
else:
    print("COMPETITOR_CHANNEL_IDS is empty — fill it in with channel IDs to fetch, "
          "then re-run this cell before Cell 21.")


In [ ]:
import json
import datetime as dt

STRATEGIST_SYSTEM = """You are the retention strategist for a YouTube science channel.
You are given: (1) this channel's own section-level audience retention on videos already
published, (2) what competitor channels are currently uploading and how fast those videos
are gaining views, and (3) the exact writing rules and title formula the channel is
currently using.

Your job is not to praise or summarize. Find the smallest number of concrete, falsifiable
notes that would change what gets written next time. A note must be specific enough to
quote back to a writer or a retention judge — not "improve pacing" but "Section 4 loses
viewers within the first two sentences on both published videos; the Re-Hook beat is
landing too late relative to the beat sheet's spec."

Do not invent a pattern from a single data point. Do not recommend anything that
contradicts the channel's non-negotiables (no invented facts, no adjective-stacking, no
call-to-action in the outro) — flag a tension with those instead of resolving it yourself.
If the data doesn't support a confident note in some category, leave that category empty."""


def build_strategist_prompt(own_retention: dict, competitor_intel: dict) -> str:
    own_block = "\n".join(
        f"  Video {vid}: " + ", ".join(
            f"S{sec}({d['name']}) watch={d['avg_watch_ratio']:.2f} "
            f"rel_vs_similar={d['avg_relative_performance']:+.2f}"
            for sec, d in per_sec.items()
        )
        for vid, per_sec in own_retention.items()
    ) or "  (no published videos with analytics yet)"

    comp_block = "\n".join(
        f"  {cid}: " + "; ".join(f'"{v["title"]}" ({v["view_velocity"]}/day)' for v in vids[:8])
        for cid, vids in competitor_intel.items()
    ) or "  (no competitor data fetched yet)"

    return f"""THIS CHANNEL'S OWN RETENTION, BY SECTION, ACROSS PUBLISHED VIDEOS:
{own_block}

COMPETITOR UPLOADS AND VIEW VELOCITY:
{comp_block}

CURRENT TITLE FORMULA:
{json.dumps(TITLE_FORMULA, indent=2)}

CURRENT BEAT SHEET SECTION NAMES: {[v['name'] for v in BEAT_SHEET.values()]}

Respond with ONLY valid JSON, no fences:
{{
  "retention_notes": ["specific, quotable note for the Cell 8 judge", "..."],
  "title_notes": ["specific note for the title-generation prompt", "..."],
  "thumbnail_notes": ["specific note for thumbnail generation", "..."],
  "banned_phrase_additions": ["phrase or construction to add to the deterministic linter"],
  "tensions_flagged": ["anything the data suggests that conflicts with a channel non-negotiable — do not resolve, just flag"]
}}
Cap each list at 5 items. Empty list is a valid, honest answer for any category."""


def run_strategist(own_retention: dict, competitor_intel: dict) -> dict:
    prompt = build_strategist_prompt(own_retention, competitor_intel)
    result = parse_json_response(
        call_model(MODEL_JUDGE, prompt, system=STRATEGIST_SYSTEM, max_tokens=2000, temperature=0.2)
    )
    print("=" * 70)
    print("STRATEGIST PROPOSAL (not yet applied — review, then run the merge cell)")
    print("=" * 70)
    for k in ["retention_notes", "title_notes", "thumbnail_notes", "banned_phrase_additions"]:
        print(f"\n{k}:")
        for note in result.get(k, []):
            print(f"  - {note}")
    if result.get("tensions_flagged"):
        print("\nTENSIONS FLAGGED (not applied, needs your judgment):")
        for t in result["tensions_flagged"]:
            print(f"  ! {t}")
    return result


def merge_strategist_proposal(proposal: dict, video_ids: list[str], source: str = "analytics+competitor"):
    """Explicit, separate step — nothing above writes to learnings.json on its own."""
    for k in ["retention_notes", "title_notes", "thumbnail_notes", "banned_phrase_additions"]:
        LEARNINGS.setdefault(k, []).extend(proposal.get(k, []))
    LEARNINGS.setdefault("history", []).append({
        "date": dt.datetime.now().isoformat(timespec="seconds"),
        "video_ids": video_ids,
        "source": source,
    })
    with open(LEARNINGS_PATH, "w", encoding="utf-8") as f:
        json.dump(LEARNINGS, f, indent=2)
    print(f"Merged into {LEARNINGS_PATH}. Next run of Cell 8/Cell 10 will pick it up automatically.")


# --- Example usage, once Cells 19 & 20 have real data ---
# own_retention = {"<video_id>": per_section_retention("<video_id>")}
# proposal = run_strategist(own_retention, competitor_intel)
# merge_strategist_proposal(proposal, video_ids=list(own_retention.keys()))
print("Strategist ready. Call run_strategist(own_retention, competitor_intel) once both exist, "
      "review the printed proposal, then call merge_strategist_proposal(...) to apply it.")
